In [0]:
import os
import uuid
import hashlib
import requests

from datetime import datetime, timezone
from urllib.parse import urlsplit
from pyspark.sql import functions as F
from pyspark.sql.window import Window

INVENTORY_TABLE = "workspace.oil_gas_ops.source_inventory_runs"
AUDIT_TABLE = "workspace.oil_gas_ops.ingestion_files"
PLAN_TABLE = "workspace.oil_gas_ops.ingestion_batch_plan"
RAW_ROOT = "/Volumes/workspace/oil_gas_bronze/raw_files"

EXPANSION_RUN_ID = str(uuid.uuid4())
MAX_FILE_BYTES = 64 * 1024 * 1024

latest_discovery = (
    spark.table(INVENTORY_TABLE)
    .groupBy("discovery_run_id")
    .agg(F.max("descoberto_em").alias("ultima_descoberta"))
    .orderBy(
        F.desc("ultima_descoberta"),
        F.desc("discovery_run_id"),
    )
    .first()
)

if latest_discovery is None:
    raise RuntimeError("Nenhum inventário disponível.")

discovery_id = latest_discovery["discovery_run_id"]

inventory = (
    spark.table(INVENTORY_TABLE)
    .filter(F.col("discovery_run_id") == discovery_id)
)

if (
    inventory.groupBy("source_url")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("O inventário contém endereços repetidos.")

urls = [
    row["source_url"]
    for row in inventory.select("source_url").collect()
]

if not urls:
    raise RuntimeError("Inventário vazio.")

history_window = Window.partitionBy("source_url").orderBy(
    F.desc("collected_at"),
    F.desc("run_id"),
)

previous_versions = (
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("source_url").isin(*urls)
        & F.col("status").isin("STORED", "REUSED")
    )
    .withColumn("_position", F.row_number().over(history_window))
    .filter(F.col("_position") == 1)
    .select(
        "source_url",
        F.col("file_sha256").alias("previous_file_sha256"),
    )
)

batch_plan = (
    inventory
    .select(
        "source_file_id",
        "dataset",
        "source_url",
        "nome_arquivo",
        "anos_mencionados_no_endereco",
        "trimestre_indicado",
    )
    .join(previous_versions, "source_url", "left")
    .withColumn("run_id", F.lit(EXPANSION_RUN_ID))
    .withColumn("discovery_run_id", F.lit(discovery_id))
    .withColumn("planejado_em", F.current_timestamp())
)

(
    batch_plan.write
    .format("delta")
    .mode("append")
    .saveAsTable(PLAN_TABLE)
)

plan_rows = (
    batch_plan.orderBy(
        "dataset",
        "anos_mencionados_no_endereco",
        "trimestre_indicado",
        "nome_arquivo",
    )
    .collect()
)

audit_schema = """
    run_id STRING,
    dataset STRING,
    source_url STRING,
    collected_at TIMESTAMP,
    file_path STRING,
    file_sha256 STRING,
    size_bytes LONG,
    status STRING,
    error_message STRING
"""

def stored_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def collect_original(item):
    file_path = None
    file_hash = None
    size_bytes = None

    try:
        if urlsplit(item["source_url"]).hostname != "www.gov.br":
            raise RuntimeError("Endereço fora do domínio esperado.")

        payload = bytearray()

        with requests.get(
            item["source_url"],
            stream=True,
            timeout=(10, 60),
        ) as response:
            response.raise_for_status()

            if urlsplit(response.url).hostname != "www.gov.br":
                raise RuntimeError("Redirecionamento inesperado.")

            content_type = response.headers.get("Content-Type", "")
            if "text/html" in content_type.lower():
                raise RuntimeError("Recebido HTML em vez de CSV.")

            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue

                if len(payload) + len(chunk) > MAX_FILE_BYTES:
                    raise RuntimeError("Arquivo excedeu o limite de 64 MiB.")

                payload.extend(chunk)

        size_bytes = len(payload)

        if size_bytes == 0:
            raise RuntimeError("Arquivo vazio.")

        
        probe = bytes(payload[:512]).decode(
            "utf-8-sig", errors="ignore"
        ).lstrip().lower()

        if probe.startswith(("<!doctype", "<html")):
            raise RuntimeError("Conteúdo aparenta ser HTML.")

        file_hash = hashlib.sha256(payload).hexdigest()

        destination_dir = (
            f"{RAW_ROOT}/datasets/{item['dataset']}/sha256={file_hash}"
        )
        os.makedirs(destination_dir, exist_ok=True)

        file_path = f"{destination_dir}/{item['nome_arquivo']}"

        if os.path.exists(file_path):
            if stored_sha256(file_path) != file_hash:
                raise RuntimeError("Arquivo existente com hash incompatível.")
            status = "REUSED"

        else:
            with open(file_path, "xb") as destination:
                destination.write(payload)

            if stored_sha256(file_path) != file_hash:
                raise RuntimeError("Falha na verificação do arquivo gravado.")
            status = "STORED"

        error_message = None

    except Exception as error:
        status = "FAILED"
        error_message = str(error)[:2000]

    return (
        EXPANSION_RUN_ID,
        item["dataset"],
        item["source_url"],
        datetime.now(timezone.utc),
        file_path,
        file_hash,
        size_bytes,
        status,
        error_message,
    )

print(f"Inventário selecionado: {discovery_id}", flush=True)
print(f"Execução de coleta: {EXPANSION_RUN_ID}", flush=True)
print(f"Arquivos planejados: {len(plan_rows)}", flush=True)

for position, item in enumerate(plan_rows, start=1):
    result = collect_original(item)

    spark.createDataFrame(
        [result], schema=audit_schema
    ).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

    print(
        f"[{position}/{len(plan_rows)}] "
        f"{item['dataset']} | {item['nome_arquivo']} | "
        f"{result[7]} | {result[6]} bytes",
        flush=True,
    )

current_audit = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == EXPANSION_RUN_ID)
)

results = (
    current_audit
    .join(
        spark.table(PLAN_TABLE)
        .filter(F.col("run_id") == EXPANSION_RUN_ID)
        .select(
            "run_id", "dataset", "source_url",
            "nome_arquivo", "previous_file_sha256",
        ),
        ["run_id", "dataset", "source_url"],
        "inner",
    )
    .withColumn(
        "comparacao_conteudo",
        F.when(F.col("status") == "FAILED", "COLETA_FALHOU")
        .when(
            F.col("previous_file_sha256").isNull(),
            "PRIMEIRA_COLETA_DO_ENDERECO",
        )
        .when(
            F.col("previous_file_sha256") == F.col("file_sha256"),
            "MESMO_CONTEUDO",
        )
        .otherwise("CONTEUDO_ALTERADO"),
    )
)

display(
    results.select(
        "dataset",
        "nome_arquivo",
        "status",
        "comparacao_conteudo",
        "size_bytes",
        "error_message",
    )
    .orderBy("dataset", "nome_arquivo")
)

display(
    results.groupBy("dataset", "status", "comparacao_conteudo")
    .count()
    .orderBy("dataset", "status", "comparacao_conteudo")
)

failed = current_audit.filter(F.col("status") == "FAILED").count()

if failed:
    raise RuntimeError(
        f"{failed} arquivo(s) falharam. "
        "Confira os erros; os demais resultados foram preservados."
    )

print(
    "\nColeta concluída. Os arquivos ainda aguardam inspeção "
    "de layout e validação estrutural."
)

Inventário selecionado: b9383b3a-e9cb-43b8-a489-f781bd7f1c2f
Execução de coleta: 1eddf192-e8eb-42eb-b26c-b6227916c739
Arquivos planejados: 15
[1/15] intervencoes | intervencao_em_pocos_2023-2024.csv | STORED | 1347561 bytes
[2/15] intervencoes | intervencao_em_pocos_2025-2026.csv | REUSED | 1108877 bytes
[3/15] producao_mar | producao_por_poco_2024.csv | STORED | 1818223 bytes
[4/15] producao_mar | producao-mar-2025.csv | REUSED | 1996681 bytes
[5/15] producao_mar | producao-mar-2026.csv | STORED | 1700011 bytes
[6/15] producao_terra | producao-por-poco-terra-trim-1.csv | STORED | 3820565 bytes
[7/15] producao_terra | producao-por-poco-terra-trim-2.csv | STORED | 3855319 bytes
[8/15] producao_terra | producao-por-poco-terra-trim-3.csv | STORED | 3901948 bytes
[9/15] producao_terra | producao_por_poco_terra_trim_4.csv | STORED | 3981012 bytes
[10/15] producao_terra | producao_por_poco_terra_2025_1_trim.csv | REUSED | 4025387 bytes
[11/15] producao_terra | producao_por_poco_terra_2025_2_

dataset,nome_arquivo,status,comparacao_conteudo,size_bytes,error_message
intervencoes,intervencao_em_pocos_2023-2024.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,1347561,null
intervencoes,intervencao_em_pocos_2025-2026.csv,REUSED,MESMO_CONTEUDO,1108877,null
producao_mar,producao-mar-2025.csv,REUSED,MESMO_CONTEUDO,1996681,null
producao_mar,producao-mar-2026.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,1700011,null
producao_mar,producao_por_poco_2024.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,1818223,null
producao_terra,producao-por-poco-terra-trim-1.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,3820565,null
producao_terra,producao-por-poco-terra-trim-2.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,3855319,null
producao_terra,producao-por-poco-terra-trim-3.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,3901948,null
producao_terra,producao_por_poco_terra_2025_1_trim.csv,REUSED,MESMO_CONTEUDO,4025387,null
producao_terra,producao_por_poco_terra_2025_2_trim.csv,STORED,PRIMEIRA_COLETA_DO_ENDERECO,4083229,null


dataset,status,comparacao_conteudo,count
intervencoes,REUSED,MESMO_CONTEUDO,1
intervencoes,STORED,PRIMEIRA_COLETA_DO_ENDERECO,1
producao_mar,REUSED,MESMO_CONTEUDO,1
producao_mar,STORED,PRIMEIRA_COLETA_DO_ENDERECO,2
producao_terra,REUSED,MESMO_CONTEUDO,1
producao_terra,STORED,PRIMEIRA_COLETA_DO_ENDERECO,9



Coleta concluída. Os arquivos ainda aguardam inspeção de layout e validação estrutural.


In [0]:
import csv
import hashlib
import io
import unicodedata
import re
import uuid

from datetime import datetime, timezone
from pyspark.sql import functions as F

PLAN_TABLE = "workspace.oil_gas_ops.ingestion_batch_plan"
AUDIT_TABLE = "workspace.oil_gas_ops.ingestion_files"
PROFILE_TABLE = "workspace.oil_gas_ops.source_file_structure_checks"

PILOT_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
CHECK_RUN_ID = str(uuid.uuid4())

latest_batch = (
    spark.table(PLAN_TABLE)
    .groupBy("run_id")
    .agg(F.max("planejado_em").alias("planejado_em"))
    .orderBy(F.desc("planejado_em"), F.desc("run_id"))
    .first()
)

if latest_batch is None:
    raise RuntimeError("Nenhum lote encontrado no plano de ingestão.")

batch_run_id = latest_batch["run_id"]

plan = (
    spark.table(PLAN_TABLE)
    .filter(F.col("run_id") == batch_run_id)
)

audit = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == batch_run_id)
)

# Cada endereço precisa ter exatamente um planejamento e um resultado.
for frame, label in [(plan, "plano"), (audit, "auditoria")]:
    duplicates = (
        frame.groupBy("dataset", "source_url")
        .count()
        .filter(F.col("count") != 1)
    )

    if duplicates.limit(1).count():
        raise RuntimeError(
            f"Há mais de um registro por dataset/endereço no {label}."
        )

files = (
    plan.select("run_id", "dataset", "source_url", "nome_arquivo")
    .join(
        audit.select(
            "run_id", "dataset", "source_url",
            "file_path", "file_sha256", "status"
        ),
        ["run_id", "dataset", "source_url"],
        "left"
    )
)

if files.filter(
    F.col("status").isNull()
    | ~F.col("status").isin("STORED", "REUSED")
).limit(1).count():
    raise RuntimeError("O lote contém arquivos sem coleta bem-sucedida.")


def normalize_column(name):
    ascii_name = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
    )
    return re.sub(r"[^a-z0-9]+", "_", ascii_name.lower()).strip("_")


def read_csv_content(path):
    with open(path, "rb") as source:
        payload = source.read()

    try:
        content = payload.decode("utf-8-sig")
        encoding = "utf-8-sig"
    except UnicodeDecodeError:
        content = payload.decode("cp1252")
        encoding = "cp1252"

    sample = content[:131072]

    dialect = csv.Sniffer().sniff(
        sample,
        delimiters=",;\t|"
    )

    reader = csv.reader(
        io.StringIO(content, newline=""),
        dialect=dialect,
        strict=True
    )

    header = next(reader, None)

    if not header or any(not column.strip() for column in header):
        raise ValueError("Cabeçalho ausente ou com nome de coluna vazio.")

    return payload, encoding, dialect.delimiter, header, reader


pilot_files = (
    spark.table(AUDIT_TABLE)
    .filter(
        (F.col("run_id") == PILOT_RUN_ID)
        & F.col("status").isin("STORED", "REUSED")
        & F.col("dataset").isin(
            "producao_mar", "producao_terra", "intervencoes"
        )
    )
    .select("dataset", "file_path")
    .distinct()
    .collect()
)

reference_headers = {}

for item in pilot_files:
    _, _, _, header, _ = read_csv_content(item["file_path"])

    if (
        item["dataset"] in reference_headers
        and reference_headers[item["dataset"]] != header
    ):
        raise RuntimeError("Mais de um layout de referência no piloto.")

    reference_headers[item["dataset"]] = header

required_datasets = {"producao_mar", "producao_terra", "intervencoes"}

if set(reference_headers) != required_datasets:
    raise RuntimeError("Não foi possível recuperar todos os layouts do piloto.")


schema = """
    check_run_id STRING,
    ingestion_run_id STRING,
    verificado_em TIMESTAMP,
    dataset STRING,
    nome_arquivo STRING,
    source_url STRING,
    source_file_sha256 STRING,
    codificacao STRING,
    separador STRING,
    quantidade_colunas INT,
    registros_nao_vazios BIGINT,
    registros_em_branco BIGINT,
    registros_com_estrutura_invalida BIGINT,
    primeiras_linhas_csv_invalidas ARRAY<BIGINT>,
    colunas_normalizadas ARRAY<STRING>,
    colunas_adicionadas ARRAY<STRING>,
    colunas_ausentes ARRAY<STRING>,
    ordem_colunas_igual BOOLEAN,
    nomes_normalizados_unicos BOOLEAN,
    hash_confere BOOLEAN,
    status STRING,
    error_message STRING
"""

results = []

for item in files.orderBy("dataset", "nome_arquivo").collect():
    encoding = delimiter = None
    header = []
    normalized = []
    added = []
    missing = []
    nonempty = blank = invalid = 0
    invalid_lines = []
    same_order = unique_names = hash_matches = False
    status = "ERROR"
    error = None

    try:
        payload, encoding, delimiter, header, reader = read_csv_content(
            item["file_path"]
        )

        hash_matches = (
            hashlib.sha256(payload).hexdigest() == item["file_sha256"]
        )

        normalized = [normalize_column(column) for column in header]
        reference = [
            normalize_column(column)
            for column in reference_headers[item["dataset"]]
        ]

        added = sorted(set(normalized) - set(reference))
        missing = sorted(set(reference) - set(normalized))
        same_order = normalized == reference
        unique_names = (
            len(normalized) == len(set(normalized))
            and all(normalized)
        )

        for row in reader:
            if not row or all(not value.strip() for value in row):
                blank += 1
                continue

            nonempty += 1

            if len(row) != len(header):
                invalid += 1

                if len(invalid_lines) < 5:
                    invalid_lines.append(reader.line_num)

        issues = []

        if not hash_matches:
            issues.append("Hash diferente do registrado na coleta")
        if not unique_names:
            issues.append("Colunas vazias ou colisão após normalização")
        if not same_order:
            issues.append("Layout diferente do piloto")
        if invalid:
            issues.append("Registros com quantidade incorreta de campos")
        if nonempty == 0:
            issues.append("Arquivo sem registros de dados")
        if encoding != "utf-8-sig":
            issues.append("Codificação alternativa requer revisão")

        status = "REVIEW" if issues else "PASS"
        error = "; ".join(issues) if issues else None

    except Exception as exc:
        status = "ERROR"
        error = str(exc)[:2000]

    results.append((
        CHECK_RUN_ID,
        batch_run_id,
        datetime.now(timezone.utc),
        item["dataset"],
        item["nome_arquivo"],
        item["source_url"],
        item["file_sha256"],
        encoding,
        delimiter,
        len(header),
        nonempty,
        blank,
        invalid,
        invalid_lines,
        normalized,
        added,
        missing,
        same_order,
        unique_names,
        hash_matches,
        status,
        error
    ))

checks = spark.createDataFrame(results, schema=schema)

checks.write.format("delta").mode("append").saveAsTable(PROFILE_TABLE)

persisted = (
    spark.table(PROFILE_TABLE)
    .filter(F.col("check_run_id") == CHECK_RUN_ID)
)

print(f"Lote de ingestão: {batch_run_id}")
print(f"Execução da validação: {CHECK_RUN_ID}")

display(
    persisted.select(
        "dataset",
        "nome_arquivo",
        "codificacao",
        "separador",
        "quantidade_colunas",
        "registros_nao_vazios",
        "registros_em_branco",
        "registros_com_estrutura_invalida",
        "status"
    ).orderBy("dataset", "nome_arquivo")
)

display(
    persisted.filter(F.col("status") != "PASS").select(
        "dataset",
        "nome_arquivo",
        "colunas_adicionadas",
        "colunas_ausentes",
        "ordem_colunas_igual",
        "primeiras_linhas_csv_invalidas",
        "error_message"
    ).orderBy("dataset", "nome_arquivo")
)

if persisted.filter(F.col("status") != "PASS").limit(1).count():
    print("Há arquivos que precisam de revisão antes da carga Bronze.")
else:
    print("Todos os arquivos passaram pela validação estrutural.")

Lote de ingestão: 1eddf192-e8eb-42eb-b26c-b6227916c739
Execução da validação: 6629c509-a338-4933-8e97-e0ad5c2bb98c


dataset,nome_arquivo,codificacao,separador,quantidade_colunas,registros_nao_vazios,registros_em_branco,registros_com_estrutura_invalida,status
intervencoes,intervencao_em_pocos_2023-2024.csv,utf-8-sig,",",12,9752,0,0,PASS
intervencoes,intervencao_em_pocos_2025-2026.csv,utf-8-sig,",",12,8065,0,0,PASS
producao_mar,producao-mar-2025.csv,utf-8-sig,",",21,16690,0,0,PASS
producao_mar,producao-mar-2026.csv,utf-8-sig,",",21,12000,0,0,PASS
producao_mar,producao_por_poco_2024.csv,utf-8-sig,",",21,15229,0,0,PASS
producao_terra,producao-por-poco-terra-trim-1.csv,utf-8-sig,",",21,28232,0,0,PASS
producao_terra,producao-por-poco-terra-trim-2.csv,utf-8-sig,",",21,28485,0,0,PASS
producao_terra,producao-por-poco-terra-trim-3.csv,utf-8-sig,",",21,28847,0,0,PASS
producao_terra,producao_por_poco_terra_2025_1_trim.csv,utf-8-sig,",",21,29802,0,0,PASS
producao_terra,producao_por_poco_terra_2025_2_trim.csv,utf-8-sig,",",21,30190,0,0,PASS


dataset,nome_arquivo,colunas_adicionadas,colunas_ausentes,ordem_colunas_igual,primeiras_linhas_csv_invalidas,error_message
producao_terra,producao_por_poco_terra_2025_4_trim.csv,List(),List(),false,List(),Layout diferente do piloto


Há arquivos que precisam de revisão antes da carga Bronze.


In [0]:
import csv
import io
import unicodedata
import re

from pyspark.sql import functions as F

PROFILE_TABLE = "workspace.oil_gas_ops.source_file_structure_checks"
AUDIT_TABLE = "workspace.oil_gas_ops.ingestion_files"

PILOT_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"

latest_check = (
    spark.table(PROFILE_TABLE)
    .groupBy("check_run_id")
    .agg(F.max("verificado_em").alias("verificado_em"))
    .orderBy(F.desc("verificado_em"), F.desc("check_run_id"))
    .first()
)

if latest_check is None:
    raise RuntimeError("Nenhuma validação estrutural encontrada.")

pending = (
    spark.table(PROFILE_TABLE)
    .filter(
        (F.col("check_run_id") == latest_check["check_run_id"])
        & (F.col("status") != "PASS")
    )
    .collect()
)


def normalize_column(name):
    ascii_name = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
    )
    return re.sub(r"[^a-z0-9]+", "_", ascii_name.lower()).strip("_")


def read_header(path):
    with open(path, "rb") as source:
        content = source.read().decode("utf-8-sig")

    dialect = csv.Sniffer().sniff(
        content[:131072],
        delimiters=",;\t|"
    )

    return next(
        csv.reader(
            io.StringIO(content, newline=""),
            dialect=dialect,
            strict=True
        )
    )


audit = spark.table(AUDIT_TABLE)
comparisons = []

for item in pending:
    reference_paths = (
        audit.filter(
            (F.col("run_id") == PILOT_RUN_ID)
            & (F.col("dataset") == item["dataset"])
            & F.col("status").isin("STORED", "REUSED")
        )
        .select("file_path")
        .distinct()
        .collect()
    )

    candidate_paths = (
        audit.filter(
            (F.col("run_id") == item["ingestion_run_id"])
            & (F.col("dataset") == item["dataset"])
            & (F.col("source_url") == item["source_url"])
            & (F.col("file_sha256") == item["source_file_sha256"])
            & F.col("status").isin("STORED", "REUSED")
        )
        .select("file_path")
        .distinct()
        .collect()
    )

    if len(reference_paths) != 1 or len(candidate_paths) != 1:
        raise RuntimeError(
            f"Não foi possível identificar um único arquivo de cada lado: "
            f"{item['nome_arquivo']}"
        )

    reference = read_header(reference_paths[0]["file_path"])
    candidate = read_header(candidate_paths[0]["file_path"])

    for index in range(max(len(reference), len(candidate))):
        expected = reference[index] if index < len(reference) else None
        observed = candidate[index] if index < len(candidate) else None

        expected_normalized = (
            normalize_column(expected) if expected is not None else None
        )
        observed_normalized = (
            normalize_column(observed) if observed is not None else None
        )

        comparisons.append((
            item["dataset"],
            item["nome_arquivo"],
            index + 1,
            expected,
            observed,
            expected_normalized == observed_normalized
        ))

schema = """
    dataset STRING,
    nome_arquivo STRING,
    posicao INT,
    coluna_no_piloto STRING,
    coluna_no_arquivo STRING,
    mesma_coluna_normalizada BOOLEAN
"""

if comparisons:
    comparison = spark.createDataFrame(comparisons, schema=schema)

    display(
        comparison
        .filter(~F.col("mesma_coluna_normalizada"))
        .orderBy("dataset", "nome_arquivo", "posicao")
    )
else:
    print("Nenhum arquivo pendente de revisão.")

dataset,nome_arquivo,posicao,coluna_no_piloto,coluna_no_arquivo,mesma_coluna_normalizada
producao_terra,producao_por_poco_terra_2025_4_trim.csv,3,[Estado],[Campo],false
producao_terra,producao_por_poco_terra_2025_4_trim.csv,5,[Campo],[Instalação],false
producao_terra,producao_por_poco_terra_2025_4_trim.csv,7,[Ambiente],[Estado],false
producao_terra,producao_por_poco_terra_2025_4_trim.csv,8,[Instalação],[Ambiente],false


In [0]:
import csv
import io
import re
import unicodedata
import uuid

from datetime import datetime, timezone
from pyspark.sql import functions as F

PROFILE_TABLE = "workspace.oil_gas_ops.source_file_structure_checks"
AUDIT_TABLE = "workspace.oil_gas_ops.ingestion_files"
CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"

PILOT_RUN_ID = "1f0f7d61-7bae-4d59-af8a-316ab87ab334"
CONTRACT_VERSION = "v1"
CONTRACT_RUN_ID = str(uuid.uuid4())


def normalize_column(name):
    value = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
    )
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


profiles = spark.table(PROFILE_TABLE)

latest_check = (
    profiles.groupBy("check_run_id")
    .agg(F.max("verificado_em").alias("verificado_em"))
    .orderBy(F.desc("verificado_em"), F.desc("check_run_id"))
    .first()
)

if latest_check is None:
    raise RuntimeError("Nenhuma validação estrutural encontrada.")

checks = (
    profiles
    .filter(F.col("check_run_id") == latest_check["check_run_id"])
    .collect()
)

if not checks:
    raise RuntimeError("A validação selecionada não contém arquivos.")

if len({row["ingestion_run_id"] for row in checks}) != 1:
    raise RuntimeError("A validação contém mais de um lote de ingestão.")

identities = [
    (row["dataset"], row["source_url"])
    for row in checks
]

if len(identities) != len(set(identities)):
    raise RuntimeError("Há arquivos repetidos na validação selecionada.")

references = {}

for dataset in sorted({row["dataset"] for row in checks}):
    paths = (
        spark.table(AUDIT_TABLE)
        .filter(
            (F.col("run_id") == PILOT_RUN_ID)
            & (F.col("dataset") == dataset)
            & F.col("status").isin("STORED", "REUSED")
        )
        .select("file_path")
        .distinct()
        .collect()
    )

    if len(paths) != 1:
        raise RuntimeError(
            f"Esperado um arquivo de referência para {dataset}."
        )

    with open(paths[0]["file_path"], "rb") as source:
        content = source.read().decode("utf-8-sig")

    header = next(
        csv.reader(
            io.StringIO(content, newline=""),
            delimiter=",",
            strict=True
        )
    )

    canonical_columns = [normalize_column(name) for name in header]

    if (
        not all(canonical_columns)
        or len(canonical_columns) != len(set(canonical_columns))
    ):
        raise RuntimeError(f"Cabeçalho de referência ambíguo: {dataset}.")

    references[dataset] = canonical_columns


contracts = []
blocked = []

for item in checks:
    actual = list(item["colunas_normalizadas"])
    canonical = references[item["dataset"]]

    compatible = (
        item["status"] in ("PASS", "REVIEW")
        and item["hash_confere"] is True
        and item["nomes_normalizados_unicos"] is True
        and item["codificacao"] == "utf-8-sig"
        and item["separador"] == ","
        and item["registros_nao_vazios"] > 0
        and item["registros_em_branco"] == 0
        and item["registros_com_estrutura_invalida"] == 0
        and len(actual) == len(canonical)
        and len(actual) == len(set(actual))
        and set(actual) == set(canonical)
    )

    if not compatible:
        blocked.append(item["nome_arquivo"])
        continue

    layout_status = (
        "MESMA_ORDEM"
        if actual == canonical
        else "ORDEM_DIFERENTE_COMPATIVEL"
    )

    contracts.append((
        CONTRACT_RUN_ID,
        CONTRACT_VERSION,
        item["check_run_id"],
        item["ingestion_run_id"],
        datetime.now(timezone.utc),
        item["dataset"],
        item["nome_arquivo"],
        item["source_url"],
        item["source_file_sha256"],
        item["codificacao"],
        item["separador"],
        actual,
        canonical,
        item["registros_nao_vazios"],
        layout_status,
        "ALINHAR_POR_NOME_NORMALIZADO"
    ))

# Nenhum contrato é publicado se houver arquivo incompatível no lote.
if blocked:
    raise RuntimeError(
        "Arquivos sem contrato aprovado: " + ", ".join(sorted(blocked))
    )

schema = """
    contract_run_id STRING,
    contract_version STRING,
    check_run_id STRING,
    ingestion_run_id STRING,
    registrado_em TIMESTAMP,
    dataset STRING,
    nome_arquivo STRING,
    source_url STRING,
    source_file_sha256 STRING,
    codificacao STRING,
    separador STRING,
    colunas_na_ordem_do_arquivo ARRAY<STRING>,
    colunas_na_ordem_canonica ARRAY<STRING>,
    registros_esperados BIGINT,
    status_layout STRING,
    regra_alinhamento STRING
"""

spark.createDataFrame(contracts, schema=schema).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(CONTRACT_TABLE)

published = (
    spark.table(CONTRACT_TABLE)
    .filter(F.col("contract_run_id") == CONTRACT_RUN_ID)
)

print(f"Execução dos contratos: {CONTRACT_RUN_ID}")

display(
    published.select(
        "dataset",
        "nome_arquivo",
        "registros_esperados",
        "status_layout",
        "regra_alinhamento"
    ).orderBy("dataset", "nome_arquivo")
)

display(
    published.groupBy("status_layout")
    .count()
    .orderBy("status_layout")
)

Execução dos contratos: aea99472-c0f7-4fb7-b8b6-e2336f272fc4


dataset,nome_arquivo,registros_esperados,status_layout,regra_alinhamento
intervencoes,intervencao_em_pocos_2023-2024.csv,9752,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
intervencoes,intervencao_em_pocos_2025-2026.csv,8065,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_mar,producao-mar-2025.csv,16690,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_mar,producao-mar-2026.csv,12000,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_mar,producao_por_poco_2024.csv,15229,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_terra,producao-por-poco-terra-trim-1.csv,28232,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_terra,producao-por-poco-terra-trim-2.csv,28485,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_terra,producao-por-poco-terra-trim-3.csv,28847,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_terra,producao_por_poco_terra_2025_1_trim.csv,29802,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO
producao_terra,producao_por_poco_terra_2025_2_trim.csv,30190,MESMA_ORDEM,ALINHAR_POR_NOME_NORMALIZADO


status_layout,count
MESMA_ORDEM,14
ORDEM_DIFERENTE_COMPATIVEL,1


In [0]:
import hashlib
import re
import unicodedata
import uuid

from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
INGESTION_TABLE = "workspace.oil_gas_ops.ingestion_files"
LOAD_AUDIT_TABLE = "workspace.oil_gas_ops.incremental_bronze_file_runs"

BRONZE_LOAD_RUN_ID = str(uuid.uuid4())


def normalize_column(name):
    value = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
    )
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


contracts_table = spark.table(CONTRACT_TABLE)

latest_contract = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest_contract is None:
    raise RuntimeError("Nenhum contrato de leitura encontrado.")

contracts = (
    contracts_table
    .filter(
        F.col("contract_run_id")
        == latest_contract["contract_run_id"]
    )
)

if contracts.select("ingestion_run_id").distinct().count() != 1:
    raise RuntimeError("Os contratos precisam pertencer a um único lote.")

if (
    contracts.groupBy("dataset", "source_file_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Há contratos repetidos para o mesmo conteúdo.")

ingestion = (
    spark.table(INGESTION_TABLE)
    .filter(F.col("status").isin("STORED", "REUSED"))
    .select(
        F.col("run_id").alias("ingestion_run_id"),
        "dataset",
        "source_url",
        F.col("file_sha256").alias("source_file_sha256"),
        "file_path",
        "collected_at"
    )
)

join_keys = [
    "ingestion_run_id",
    "dataset",
    "source_url",
    "source_file_sha256"
]

files = contracts.join(ingestion, join_keys, "left")

if files.count() != contracts.count():
    raise RuntimeError("A associação com a auditoria multiplicou arquivos.")

if files.filter(
    F.col("file_path").isNull() | F.col("collected_at").isNull()
).limit(1).count():
    raise RuntimeError("Contrato sem arquivo ou data de coleta correspondente.")


audit_schema = """
    bronze_load_run_id STRING,
    contract_run_id STRING,
    ingestion_run_id STRING,
    registrado_em TIMESTAMP,
    dataset STRING,
    nome_arquivo STRING,
    source_file_sha256 STRING,
    tabela STRING,
    registros_esperados BIGINT,
    registros_na_bronze_do_arquivo BIGINT,
    status STRING,
    error_message STRING
"""

allowed_datasets = {
    "producao_mar",
    "producao_terra",
    "intervencoes"
}

print(f"Execução Bronze: {BRONZE_LOAD_RUN_ID}")

for item in files.orderBy("dataset", "nome_arquivo").collect():
    target_table = None
    observed_count = None
    status = "FAILED"
    error = None

    try:
        dataset = item["dataset"]

        if dataset not in allowed_datasets:
            raise RuntimeError(f"Dataset não autorizado nesta carga: {dataset}")

        if (
            item["contract_version"] != "v1"
            or item["regra_alinhamento"] != "ALINHAR_POR_NOME_NORMALIZADO"
            or item["codificacao"] != "utf-8-sig"
            or item["separador"] != ","
        ):
            raise RuntimeError("Contrato de leitura não suportado.")

        target_table = f"workspace.oil_gas_bronze.{dataset}"
        target = spark.table(target_table)

        canonical = list(item["colunas_na_ordem_canonica"])
        business_fields = [
            field for field in target.schema.fields
            if not field.name.startswith("_")
        ]

        if {field.name for field in business_fields} != set(canonical):
            raise RuntimeError("Schema da tabela Bronze diferente do contrato.")

        if any(
            not isinstance(field.dataType, StringType)
            for field in business_fields
        ):
            raise RuntimeError("As colunas de negócio da Bronze devem ser STRING.")

        if file_sha256(item["file_path"]) != item["source_file_sha256"]:
            raise RuntimeError("O conteúdo do arquivo mudou após a validação.")

        expected_count = item["registros_esperados"]

        observed_count = (
            target
            .filter(
                F.col("_source_file_sha256")
                == item["source_file_sha256"]
            )
            .count()
        )

        if observed_count == expected_count:
            status = "REUSED"

        elif observed_count != 0:
            raise RuntimeError(
                f"Conteúdo já presente com contagem divergente: "
                f"{observed_count} encontrados; {expected_count} esperados."
            )

        else:
            
            raw = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "false")
                .option("encoding", "UTF-8")
                .option("sep", item["separador"])
                .option("quote", '"')
                .option("escape", '"')
                .option("multiLine", "true")
                .option("mode", "FAILFAST")
                .option("ignoreLeadingWhiteSpace", "false")
                .option("ignoreTrailingWhiteSpace", "false")
                .csv(item["file_path"])
            )

            normalized = [normalize_column(name) for name in raw.columns]

            if normalized != list(item["colunas_na_ordem_do_arquivo"]):
                raise RuntimeError("Cabeçalho lido diferente do contrato.")

            if len(normalized) != len(set(normalized)):
                raise RuntimeError("Colisão entre nomes normalizados.")

            # Primeiro renomeia na ordem real; depois seleciona por nome.
            aligned = raw.toDF(*normalized).select(*canonical)

            read_count = aligned.count()

            if read_count != expected_count:
                raise RuntimeError(
                    f"Leitura Spark: {read_count} registros; "
                    f"validação CSV: {expected_count}."
                )

            bronze = (
                aligned
                .withColumn("_source_url", F.lit(item["source_url"]))
                .withColumn("_source_file_path", F.lit(item["file_path"]))
                .withColumn(
                    "_source_file_sha256",
                    F.lit(item["source_file_sha256"])
                )
                .withColumn(
                    "_source_collected_at",
                    F.lit(item["collected_at"]).cast("timestamp")
                )
                .withColumn(
                    "_ingestion_run_id",
                    F.lit(item["ingestion_run_id"])
                )
                .withColumn(
                    "_bronze_load_run_id",
                    F.lit(BRONZE_LOAD_RUN_ID)
                )
                .withColumn("_bronze_loaded_at", F.current_timestamp())
            )

            missing_columns = set(target.columns) - set(bronze.columns)

            if missing_columns:
                raise RuntimeError(
                    f"Colunas exigidas pela tabela estão ausentes: "
                    f"{sorted(missing_columns)}"
                )

            bronze.select(*target.columns).write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(target_table)

            observed_count = (
                spark.table(target_table)
                .filter(
                    F.col("_source_file_sha256")
                    == item["source_file_sha256"]
                )
                .count()
            )

            if observed_count != expected_count:
                raise RuntimeError("Contagem após a gravação divergente.")

            status = "LOADED"

    except Exception as exc:
        error = str(exc)[:2000]

    audit_row = (
        BRONZE_LOAD_RUN_ID,
        item["contract_run_id"],
        item["ingestion_run_id"],
        datetime.now(timezone.utc),
        item["dataset"],
        item["nome_arquivo"],
        item["source_file_sha256"],
        target_table,
        item["registros_esperados"],
        observed_count,
        status,
        error
    )

    spark.createDataFrame([audit_row], schema=audit_schema).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(LOAD_AUDIT_TABLE)

    print(
        f"{item['dataset']} | {item['nome_arquivo']} | "
        f"{status} | {observed_count}",
        flush=True
    )

    if status == "FAILED":
        raise RuntimeError(
            f"Carga interrompida em {item['nome_arquivo']}: {error}"
        )


results = (
    spark.table(LOAD_AUDIT_TABLE)
    .filter(F.col("bronze_load_run_id") == BRONZE_LOAD_RUN_ID)
)

display(
    results.select(
        "dataset",
        "nome_arquivo",
        "registros_esperados",
        "registros_na_bronze_do_arquivo",
        "status"
    ).orderBy("dataset", "nome_arquivo")
)

display(
    results.groupBy("dataset", "status")
    .agg(
        F.count("*").alias("arquivos"),
        F.sum("registros_na_bronze_do_arquivo").alias("registros")
    )
    .orderBy("dataset", "status")
)

Execução Bronze: 0d32cb46-b946-4560-9c47-1bc7c7998adb
intervencoes | intervencao_em_pocos_2023-2024.csv | LOADED | 9752
intervencoes | intervencao_em_pocos_2025-2026.csv | REUSED | 8065
producao_mar | producao-mar-2025.csv | REUSED | 16690
producao_mar | producao-mar-2026.csv | LOADED | 12000
producao_mar | producao_por_poco_2024.csv | LOADED | 15229
producao_terra | producao-por-poco-terra-trim-1.csv | LOADED | 28232
producao_terra | producao-por-poco-terra-trim-2.csv | LOADED | 28485
producao_terra | producao-por-poco-terra-trim-3.csv | LOADED | 28847
producao_terra | producao_por_poco_terra_2025_1_trim.csv | REUSED | 29802
producao_terra | producao_por_poco_terra_2025_2_trim.csv | LOADED | 30190
producao_terra | producao_por_poco_terra_2025_3_trim.csv | LOADED | 30461
producao_terra | producao_por_poco_terra_2025_4_trim.csv | LOADED | 30123
producao_terra | producao_por_poco_terra_2026_1_trim.csv | LOADED | 20389
producao_terra | producao_por_poco_terra_2026_2_trim.csv | LOADED | 15

dataset,nome_arquivo,registros_esperados,registros_na_bronze_do_arquivo,status
intervencoes,intervencao_em_pocos_2023-2024.csv,9752,9752,LOADED
intervencoes,intervencao_em_pocos_2025-2026.csv,8065,8065,REUSED
producao_mar,producao-mar-2025.csv,16690,16690,REUSED
producao_mar,producao-mar-2026.csv,12000,12000,LOADED
producao_mar,producao_por_poco_2024.csv,15229,15229,LOADED
producao_terra,producao-por-poco-terra-trim-1.csv,28232,28232,LOADED
producao_terra,producao-por-poco-terra-trim-2.csv,28485,28485,LOADED
producao_terra,producao-por-poco-terra-trim-3.csv,28847,28847,LOADED
producao_terra,producao_por_poco_terra_2025_1_trim.csv,29802,29802,REUSED
producao_terra,producao_por_poco_terra_2025_2_trim.csv,30190,30190,LOADED


dataset,status,arquivos,registros
intervencoes,LOADED,1,9752
intervencoes,REUSED,1,8065
producao_mar,LOADED,2,27229
producao_mar,REUSED,1,16690
producao_terra,LOADED,9,241635
producao_terra,REUSED,1,29802


In [0]:
import uuid

from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
COVERAGE_TABLE = "workspace.oil_gas_ops.production_file_coverage_checks"

CHECK_RUN_ID = str(uuid.uuid4())

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        "contract_run_id",
        "dataset",
        "source_file_sha256",
        "nome_arquivo",
        "registros_esperados"
    )
)

if (
    contracts.groupBy("dataset", "source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")


def clean_text(column):
    value = F.upper(F.trim(F.col(column)))
    return F.when(F.length(value) > 0, value)


frames = []

for dataset in ["producao_mar", "producao_terra"]:
    selected_files = (
        contracts
        .filter(F.col("dataset") == dataset)
        .drop("dataset")
    )

    frame = (
        spark.table(f"workspace.oil_gas_bronze.{dataset}")
        .join(
            selected_files,
            F.col("_source_file_sha256")
            == F.col("source_file_sha256"),
            "inner"
        )
        .select(
            "contract_run_id",
            "nome_arquivo",
            "source_file_sha256",
            "registros_esperados",
            "ano",
            "mes_ano",
            "poco",
            "campo",
            "instalacao"
        )
        .withColumn("dataset", F.lit(dataset))
        .withColumn(
            "competencia",
            F.when(
                F.trim(F.col("mes_ano")).rlike(
                    r"^(0[1-9]|1[0-2])/[0-9]{4}$"
                ),
                F.to_date(
                    F.try_to_timestamp(
                        F.concat(F.lit("01/"), F.trim(F.col("mes_ano"))),
                        F.lit("dd/MM/yyyy")
                    )
                )
            )
        )
        .withColumn("ano_informado", F.expr("try_cast(trim(ano) AS INT)"))
        .withColumn("poco_chave", clean_text("poco"))
        .withColumn("campo_chave", clean_text("campo"))
        .withColumn("instalacao_chave", clean_text("instalacao"))
    )

    frames.append(frame)

production = frames[0].unionByName(frames[1])

incomplete_key = (
    F.col("poco_chave").isNull()
    | F.col("competencia").isNull()
    | F.col("campo_chave").isNull()
    | F.col("instalacao_chave").isNull()
)

invalid_year = (
    F.col("ano_informado").isNull()
    | (F.col("ano_informado") != F.year("competencia"))
)

coverage = (
    production.groupBy(
        "contract_run_id",
        "dataset",
        "nome_arquivo",
        "source_file_sha256",
        "registros_esperados"
    )
    .agg(
        F.count("*").alias("registros_encontrados"),
        F.sum(
            F.when(F.col("competencia").isNull(), 1).otherwise(0)
        ).alias("competencias_invalidas"),
        F.sum(
            F.when(
                F.col("competencia").isNotNull() & invalid_year,
                1
            ).otherwise(0)
        ).alias("ano_invalido_ou_divergente"),
        F.sum(
            F.when(incomplete_key, 1).otherwise(0)
        ).alias("registros_com_chave_incompleta"),
        F.countDistinct("competencia").alias("meses"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
        F.sort_array(
            F.collect_set(F.date_format("competencia", "yyyy-MM"))
        ).alias("competencias_presentes")
    )
    .withColumn(
        "status",
        F.when(
            (F.col("registros_encontrados") == F.col("registros_esperados"))
            & (F.col("competencias_invalidas") == 0)
            & (F.col("ano_invalido_ou_divergente") == 0)
            & (F.col("registros_com_chave_incompleta") == 0),
            "PASS"
        ).otherwise("REVIEW")
    )
    .withColumn("check_run_id", F.lit(CHECK_RUN_ID))
    .withColumn("verificado_em", F.current_timestamp())
)

coverage.write.format("delta").mode("append").saveAsTable(COVERAGE_TABLE)

display(
    spark.table(COVERAGE_TABLE)
    .filter(F.col("check_run_id") == CHECK_RUN_ID)
    .select(
        "dataset",
        "nome_arquivo",
        "registros_encontrados",
        "competencias_invalidas",
        "ano_invalido_ou_divergente",
        "registros_com_chave_incompleta",
        "meses",
        "primeira_competencia",
        "ultima_competencia",
        "competencias_presentes",
        "status"
    )
    .orderBy("dataset", "primeira_competencia")
)


repeated_keys = (
    production
    .filter(~incomplete_key)
    .groupBy(
        "dataset",
        "poco_chave",
        "competencia",
        "campo_chave",
        "instalacao_chave"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("source_file_sha256").alias("arquivos_distintos"),
        F.sort_array(
            F.collect_set("nome_arquivo")
        ).alias("arquivos")
    )
    .filter(F.col("registros") > 1)
    .withColumn(
        "tipo_repeticao",
        F.when(
            F.col("arquivos_distintos") > 1,
            "CHAVE_PRESENTE_EM_MAIS_DE_UM_ARQUIVO"
        ).otherwise("CHAVE_REPETIDA_NO_MESMO_ARQUIVO")
    )
)

display(
    repeated_keys.groupBy("dataset", "tipo_repeticao")
    .agg(
        F.count("*").alias("grupos_com_chave_repetida"),
        F.sum("registros").alias("registros_nos_grupos"),
        F.sum(F.col("registros") - 1).alias("registros_excedentes")
    )
    .orderBy("dataset", "tipo_repeticao")
)

display(
    repeated_keys
    .orderBy(F.desc("registros"), "dataset", "poco_chave", "competencia")
    .limit(20)
)

dataset,nome_arquivo,registros_encontrados,competencias_invalidas,ano_invalido_ou_divergente,registros_com_chave_incompleta,meses,primeira_competencia,ultima_competencia,competencias_presentes,status
producao_mar,producao_por_poco_2024.csv,15229,0,0,0,12,2024-01-01,2024-12-01,"List(2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12)",PASS
producao_mar,producao-mar-2025.csv,16690,0,0,0,12,2025-01-01,2025-12-01,"List(2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12)",PASS
producao_mar,producao-mar-2026.csv,12000,0,0,0,8,2026-01-01,2026-08-01,"List(2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08)",PASS
producao_terra,producao_por_poco_terra_2025_4_trim.csv,30123,30123,0,30123,0,null,null,List(),REVIEW
producao_terra,producao-por-poco-terra-trim-1.csv,28232,0,0,0,3,2024-01-01,2024-03-01,"List(2024-01, 2024-02, 2024-03)",PASS
producao_terra,producao-por-poco-terra-trim-2.csv,28485,0,0,0,3,2024-04-01,2024-06-01,"List(2024-04, 2024-05, 2024-06)",PASS
producao_terra,producao-por-poco-terra-trim-3.csv,28847,0,0,0,3,2024-07-01,2024-09-01,"List(2024-07, 2024-08, 2024-09)",PASS
producao_terra,producao_por_poco_terra_trim_4.csv,29422,0,0,0,3,2024-10-01,2024-12-01,"List(2024-10, 2024-11, 2024-12)",PASS
producao_terra,producao_por_poco_terra_2025_1_trim.csv,29802,0,0,0,3,2025-01-01,2025-03-01,"List(2025-01, 2025-02, 2025-03)",PASS
producao_terra,producao_por_poco_terra_2025_2_trim.csv,30190,0,0,0,3,2025-04-01,2025-06-01,"List(2025-04, 2025-05, 2025-06)",PASS


dataset,tipo_repeticao,grupos_com_chave_repetida,registros_nos_grupos,registros_excedentes


dataset,poco_chave,competencia,campo_chave,instalacao_chave,registros,arquivos_distintos,arquivos,tipo_repeticao


In [0]:
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

target_files = [
    "producao_por_poco_terra_2025_4_trim.csv",
    "producao_por_poco_terra_2026_1_trim.csv",
    "producao_por_poco_terra_2026_2_trim.csv"
]

selected_files = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & (F.col("dataset") == "producao_terra")
        & F.col("nome_arquivo").isin(target_files)
    )
    .select(
        "nome_arquivo",
        F.col("source_file_sha256").alias("_source_file_sha256")
    )
)

if selected_files.count() != 3:
    raise RuntimeError("Esperados os três arquivos para diagnóstico.")

if (
    selected_files.groupBy("_source_file_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Há arquivos repetidos na seleção.")

data = (
    spark.table("workspace.oil_gas_bronze.producao_terra")
    .join(selected_files, "_source_file_sha256", "inner")
)

print("1. Competências recebidas no quarto trimestre de 2025")

display(
    data
    .filter(
        F.col("nome_arquivo")
        == "producao_por_poco_terra_2025_4_trim.csv"
    )
    .groupBy("ano", "mes_ano")
    .count()
    .orderBy("ano", "mes_ano")
)

print("2. Amostra das colunas do quarto trimestre de 2025")

display(
    data
    .filter(
        F.col("nome_arquivo")
        == "producao_por_poco_terra_2025_4_trim.csv"
    )
    .select(
        "ano",
        "mes_ano",
        "estado",
        "bacia",
        "campo",
        "poco",
        "ambiente",
        "instalacao"
    )
    .orderBy("mes_ano", "poco", "campo")
    .limit(10)
)

def missing(column):
    return (
        F.col(column).isNull()
        | (F.length(F.trim(F.col(column))) == 0)
    )

data_2026 = (
    data
    .filter(F.col("nome_arquivo").isin(target_files[1:]))
    .withColumn("poco_ausente", missing("poco"))
    .withColumn("campo_ausente", missing("campo"))
    .withColumn("instalacao_ausente", missing("instalacao"))
)

incomplete = (
    F.col("poco_ausente")
    | F.col("campo_ausente")
    | F.col("instalacao_ausente")
)

print("3. Combinações de campos ausentes em 2026")

display(
    data_2026
    .filter(incomplete)
    .groupBy(
        "nome_arquivo",
        "mes_ano",
        "poco_ausente",
        "campo_ausente",
        "instalacao_ausente"
    )
    .count()
    .orderBy("nome_arquivo", "mes_ano")
)

print("4. Amostra dos registros com chave incompleta")

display(
    data_2026
    .filter(incomplete)
    .select(
        "nome_arquivo",
        "mes_ano",
        "poco",
        "campo",
        "instalacao",
        "estado",
        "bacia",
        "ambiente",
        "producao_de_oleo_m3",
        "injecao_de_agua_para_recuperacao_secundaria_m3"
    )
    .orderBy("nome_arquivo", "mes_ano", "poco")
    .limit(20)
)

1. Competências recebidas no quarto trimestre de 2025


ano,mes_ano,count
2025,dez/2025,9859
2025,nov/2025,10137
2025,out/2025,10127


2. Amostra das colunas do quarto trimestre de 2025


ano,mes_ano,estado,bacia,campo,poco,ambiente,instalacao
2025,dez/2025,Bahia,Recôncavo,ÁGUA GRANDE,1-AG-1-BA,TERRA,Estação Rio Ventura
2025,dez/2025,Rio Grande do Norte,Potiguar,ALTO ALEGRE,1-ALG-1-RN,TERRA,Ponto de Coleta do Poço 1-ALG-1-RN
2025,dez/2025,Bahia,Recôncavo,CABURÉ LESTE,1-ALV-11-BA,TERRA,Ponto de Entrega Prosperidade I
2025,dez/2025,Bahia,Recôncavo,TIÊ,1-ALV-2-BA,TERRA,ESTAÇÃO COLETORA DE TIÊ
2025,dez/2025,Bahia,Recôncavo,MURUCUTUTU,1-ALV-5-BA,TERRA,Estação Coletora de Murucututu
2025,dez/2025,Bahia,Recôncavo,MURUCUTUTU,1-ALV-6D-BA,TERRA,Estação Coletora de Murucututu
2025,dez/2025,Bahia,Recôncavo,CABURÉ,1-ALV-7D-BA,TERRA,Ponto de Entrega Prosperidade I
2025,dez/2025,Bahia,Recôncavo,MÃE-DA-LUA,1-ALV-8D-BA,TERRA,Estação de Tratamento Mãe da Lua
2025,dez/2025,Rio Grande do Norte,Potiguar,CANTO DO AMARO,1-AP-1-RN,TERRA,Estação Coletora de Canto do Amaro N
2025,dez/2025,Rio Grande do Norte,Potiguar,CANTO DO AMARO,1-AP-2-RN,TERRA,Estação Coletora Alto da Pedra B


3. Combinações de campos ausentes em 2026


nome_arquivo,mes_ano,poco_ausente,campo_ausente,instalacao_ausente,count
producao_por_poco_terra_2026_1_trim.csv,01/2026,false,false,true,217
producao_por_poco_terra_2026_1_trim.csv,02/2026,false,false,true,214
producao_por_poco_terra_2026_1_trim.csv,03/2026,false,false,true,226
producao_por_poco_terra_2026_2_trim.csv,05/2026,false,false,true,228
producao_por_poco_terra_2026_2_trim.csv,06/2026,false,false,true,231


4. Amostra dos registros com chave incompleta


nome_arquivo,mes_ano,poco,campo,instalacao,estado,bacia,ambiente,producao_de_oleo_m3,injecao_de_agua_para_recuperacao_secundaria_m3
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-CA-2-BA,Camaçari,null,Bahia,Recôncavo,TERRA,"0,00","0,00"
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-FAV-2-BA,FAZENDA ALVORADA,null,Bahia,Recôncavo,TERRA,null,"2.866,80"
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-FGT-1-AL,FAZENDA GUINDASTE,null,Alagoas,Alagoas,TERRA,"0,00",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-MPE-1-BA,UIRAPURU,null,Bahia,Recôncavo,TERRA,"16,11",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-MPE-5DP-BA,CANÁRIO,null,Bahia,Recôncavo,TERRA,"455,28",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-PIA-5-AL,PIAÇABUÇU,null,Alagoas,Alagoas,TERRA,"135,24",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-PSY-18-BA,TROVOADA,null,Bahia,Recôncavo,TERRA,"1,59",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-PSY-4-RN,ARAÇARI,null,Rio Grande do Norte,Potiguar,TERRA,"71,92",null
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-PTX-11-RN,CARDEAL,null,Rio Grande do Norte,Potiguar,TERRA,null,"5.358,10"
producao_por_poco_terra_2026_1_trim.csv,01/2026,1-PV-1-BA,PARAMIRIM DO VENCIMENTO,null,Bahia,Recôncavo,TERRA,"30,00","0,00"


In [0]:
import uuid

from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
NUMERIC_AUDIT_TABLE = "workspace.oil_gas_ops.expanded_numeric_checks"

CHECK_RUN_ID = str(uuid.uuid4())

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        "dataset",
        "nome_arquivo",
        F.col("source_file_sha256").alias("_source_file_sha256")
    )
)

if (
    contracts.groupBy("dataset", "_source_file_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

frames = []

for dataset in ["producao_mar", "producao_terra"]:
    frame = (
        spark.table(f"workspace.oil_gas_bronze.{dataset}")
        .join(
            contracts.filter(F.col("dataset") == dataset),
            "_source_file_sha256",
            "inner"
        )
    )
    frames.append(frame)

production = frames[0].unionByName(frames[1])

month_pairs = [
    ("jan", "01"), ("fev", "02"), ("mar", "03"),
    ("abr", "04"), ("mai", "05"), ("jun", "06"),
    ("jul", "07"), ("ago", "08"), ("set", "09"),
    ("out", "10"), ("nov", "11"), ("dez", "12")
]

month_map = F.create_map(*[
    F.lit(value)
    for pair in month_pairs
    for value in pair
])

production = production.withColumn(
    "_competencia_texto",
    F.lower(F.trim(F.col("mes_ano")))
)

text = F.col("_competencia_texto")

month_number = F.when(
    text.rlike(r"^(0[1-9]|1[0-2])/[0-9]{4}$"),
    F.substring(text, 1, 2)
).when(
    text.rlike(
        r"^(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)/[0-9]{4}$"
    ),
    F.element_at(month_map, F.substring(text, 1, 3))
)

production = (
    production
    .withColumn(
        "_competencia_validada",
        F.to_date(
            F.try_to_timestamp(
                F.concat(
                    F.lit("01/"),
                    month_number,
                    F.lit("/"),
                    F.regexp_extract(text, r"/([0-9]{4})$", 1)
                ),
                F.lit("dd/MM/yyyy")
            )
        )
    )
    .withColumn("_ano_validado", F.expr("try_cast(trim(ano) AS INT)"))
)

print("1. Cobertura após reconhecer os meses em português")

display(
    production.groupBy("dataset", "nome_arquivo")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(
                F.col("_competencia_validada").isNull(), 1
            ).otherwise(0)
        ).alias("competencias_invalidas"),
        F.sum(
            F.when(
                F.col("_competencia_validada").isNotNull()
                & (
                    F.col("_ano_validado").isNull()
                    | (
                        F.col("_ano_validado")
                        != F.year("_competencia_validada")
                    )
                ),
                1
            ).otherwise(0)
        ).alias("ano_invalido_ou_divergente"),
        F.min("_competencia_validada").alias("primeira_competencia"),
        F.max("_competencia_validada").alias("ultima_competencia"),
        F.sort_array(
            F.collect_set(
                F.date_format("_competencia_validada", "yyyy-MM")
            )
        ).alias("competencias_presentes")
    )
    .orderBy("dataset", "primeira_competencia")
)

measures = sorted([
    column for column in production.columns
    if column.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError(f"Esperadas 13 medidas; encontradas {len(measures)}.")

long_values = (
    production.select(
        "dataset",
        "nome_arquivo",
        "_source_file_sha256",
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(column).alias("medida"),
                    F.col(column).alias("valor_original")
                )
                for column in measures
            ])
        ).alias("item")
    )
    .select(
        "dataset",
        "nome_arquivo",
        "_source_file_sha256",
        "item.*"
    )
    .withColumn("valor_limpo", F.trim(F.col("valor_original")))
)


numeric_pattern = r"^[+-]?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+)(?:,[0-9]{1,9})?$"

long_values = (
    long_values
    .withColumn(
        "ausente",
        F.col("valor_limpo").isNull()
        | (F.length("valor_limpo") == 0)
    )
    .withColumn(
        "formato_valido",
        F.coalesce(
            F.col("valor_limpo").rlike(numeric_pattern),
            F.lit(False)
        )
    )
    .withColumn(
        "valor_decimal",
        F.when(
            ~F.col("ausente") & F.col("formato_valido"),
            F.expr("""
                try_cast(
                    replace(replace(valor_limpo, '.', ''), ',', '.')
                    AS DECIMAL(28,9)
                )
            """)
        )
    )
    .withColumn(
        "invalido",
        ~F.col("ausente")
        & (
            ~F.col("formato_valido")
            | F.col("valor_decimal").isNull()
        )
    )
)

numeric_checks = (
    long_values.groupBy(
        "dataset", "nome_arquivo", "_source_file_sha256", "medida"
    )
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("ausente"), 1).otherwise(0)
        ).alias("valores_ausentes"),
        F.sum(
            F.when(F.col("invalido"), 1).otherwise(0)
        ).alias("valores_invalidos"),
        F.sum(
            F.when(F.col("valor_decimal") < 0, 1).otherwise(0)
        ).alias("valores_negativos"),
        F.sum(
            F.when(
                ~F.col("ausente")
                & ~F.col("invalido")
                & F.col("valor_limpo").contains("."),
                1
            ).otherwise(0)
        ).alias("valores_com_separador_milhar"),
        F.min("valor_decimal").alias("menor_valor"),
        F.max("valor_decimal").alias("maior_valor")
    )
    .withColumn("check_run_id", F.lit(CHECK_RUN_ID))
    .withColumn("contract_run_id", F.lit(latest["contract_run_id"]))
    .withColumn("verificado_em", F.current_timestamp())
)

numeric_checks.write.format("delta").mode("append") \
    .saveAsTable(NUMERIC_AUDIT_TABLE)

persisted = (
    spark.table(NUMERIC_AUDIT_TABLE)
    .filter(F.col("check_run_id") == CHECK_RUN_ID)
)

print("2. Resumo numérico por dataset e medida")

display(
    persisted.groupBy("dataset", "medida")
    .agg(
        F.sum("valores_ausentes").alias("valores_ausentes"),
        F.sum("valores_invalidos").alias("valores_invalidos"),
        F.sum("valores_negativos").alias("valores_negativos"),
        F.sum("valores_com_separador_milhar").alias(
            "valores_com_separador_milhar"
        ),
        F.min("menor_valor").alias("menor_valor"),
        F.max("maior_valor").alias("maior_valor")
    )
    .orderBy("dataset", "medida")
)

print("3. Exemplos de valores inválidos, se houver")

display(
    long_values
    .filter(F.col("invalido"))
    .groupBy("dataset", "nome_arquivo", "medida", "valor_original")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

1. Cobertura após reconhecer os meses em português


dataset,nome_arquivo,registros,competencias_invalidas,ano_invalido_ou_divergente,primeira_competencia,ultima_competencia,competencias_presentes
producao_mar,producao_por_poco_2024.csv,15229,0,0,2024-01-01,2024-12-01,"List(2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12)"
producao_mar,producao-mar-2025.csv,16690,0,0,2025-01-01,2025-12-01,"List(2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12)"
producao_mar,producao-mar-2026.csv,12000,0,0,2026-01-01,2026-08-01,"List(2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08)"
producao_terra,producao-por-poco-terra-trim-1.csv,28232,0,0,2024-01-01,2024-03-01,"List(2024-01, 2024-02, 2024-03)"
producao_terra,producao-por-poco-terra-trim-2.csv,28485,0,0,2024-04-01,2024-06-01,"List(2024-04, 2024-05, 2024-06)"
producao_terra,producao-por-poco-terra-trim-3.csv,28847,0,0,2024-07-01,2024-09-01,"List(2024-07, 2024-08, 2024-09)"
producao_terra,producao_por_poco_terra_trim_4.csv,29422,0,0,2024-10-01,2024-12-01,"List(2024-10, 2024-11, 2024-12)"
producao_terra,producao_por_poco_terra_2025_1_trim.csv,29802,0,0,2025-01-01,2025-03-01,"List(2025-01, 2025-02, 2025-03)"
producao_terra,producao_por_poco_terra_2025_2_trim.csv,30190,0,0,2025-04-01,2025-06-01,"List(2025-04, 2025-05, 2025-06)"
producao_terra,producao_por_poco_terra_2025_3_trim.csv,30461,0,0,2025-07-01,2025-09-01,"List(2025-07, 2025-08, 2025-09)"


2. Resumo numérico por dataset e medida


dataset,medida,valores_ausentes,valores_invalidos,valores_negativos,valores_com_separador_milhar,menor_valor,maior_valor
producao_mar,injecao_de_agua_para_descarte_m3,28370,0,0,35,0E-9,179740.630000000
producao_mar,injecao_de_agua_para_recuperacao_secundaria_m3,28370,0,0,2809,0E-9,446422.500000000
producao_mar,injecao_de_gas_carbonico_mm3,28370,0,0,684,0E-9,67162.419000000
producao_mar,injecao_de_gas_mm3,28370,0,0,819,0E-9,133741.416000000
producao_mar,injecao_de_nitrogenio_mm3,28370,0,0,0,0E-9,5.210000000
producao_mar,injecao_de_outros_fluidos_m3,28370,0,0,0,0E-9,0E-9
producao_mar,injecao_de_polimeros_m3,28370,0,0,0,0E-9,101.344000000
producao_mar,injecao_de_vapor_de_agua_t,28370,0,0,0,0E-9,0E-9
producao_mar,producao_de_agua_m3,15538,0,0,3584,0E-9,240420.063000000
producao_mar,producao_de_condensado_m3,15538,0,0,24,0E-9,3882.411000000


3. Exemplos de valores inválidos, se houver


dataset,nome_arquivo,medida,valor_original,count


In [0]:
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        "dataset",
        "nome_arquivo",
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if (
    contracts.groupBy("dataset", "_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")


def clean_text(column):
    value = F.upper(F.trim(F.col(column)))
    return F.when(F.length(value) > 0, value)


month_pairs = [
    ("jan", "01"), ("fev", "02"), ("mar", "03"),
    ("abr", "04"), ("mai", "05"), ("jun", "06"),
    ("jul", "07"), ("ago", "08"), ("set", "09"),
    ("out", "10"), ("nov", "11"), ("dez", "12")
]

month_map = F.create_map(*[
    F.lit(value)
    for pair in month_pairs
    for value in pair
])

frames = []

for dataset in ["producao_mar", "producao_terra"]:
    frame = (
        spark.table(f"workspace.oil_gas_bronze.{dataset}")
        .join(
            contracts.filter(F.col("dataset") == dataset),
            "_source_file_sha256",
            "inner"
        )
        .withColumn("_mes_texto", F.lower(F.trim(F.col("mes_ano"))))
    )

    text = F.col("_mes_texto")

    month_number = F.when(
        text.rlike(r"^(0[1-9]|1[0-2])/[0-9]{4}$"),
        F.substring(text, 1, 2)
    ).when(
        text.rlike(
            r"^(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)/[0-9]{4}$"
        ),
        F.element_at(month_map, F.substring(text, 1, 3))
    )

    frame = (
        frame
        .withColumn(
            "competencia",
            F.to_date(
                F.try_to_timestamp(
                    F.concat(
                        F.lit("01/"),
                        month_number,
                        F.lit("/"),
                        F.regexp_extract(text, r"/([0-9]{4})$", 1)
                    ),
                    F.lit("dd/MM/yyyy")
                )
            )
        )
        .select(
            "dataset",
            "nome_arquivo",
            "_source_file_sha256",
            "competencia",
            clean_text("poco").alias("poco"),
            clean_text("campo").alias("campo"),
            clean_text("instalacao").alias("instalacao")
        )
    )

    frames.append(frame)

production = frames[0].unionByName(frames[1])

expected_count = contracts.agg(
    F.sum("registros_esperados").alias("total")
).first()["total"]

actual_count = production.count()

if actual_count != expected_count:
    raise RuntimeError(
        f"Contagem divergente: {actual_count} encontrados; "
        f"{expected_count} esperados."
    )

print("1. Integridade dos componentes da chave")

display(
    production.groupBy("dataset")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("competencia").isNull(), 1).otherwise(0)
        ).alias("competencias_invalidas"),
        F.sum(
            F.when(F.col("poco").isNull(), 1).otherwise(0)
        ).alias("pocos_ausentes"),
        F.sum(
            F.when(F.col("campo").isNull(), 1).otherwise(0)
        ).alias("campos_ausentes"),
        F.sum(
            F.when(F.col("instalacao").isNull(), 1).otherwise(0)
        ).alias("instalacoes_ausentes")
    )
    .orderBy("dataset")
)


repeated_keys = (
    production.groupBy(
        "dataset",
        "poco",
        "competencia",
        "campo",
        "instalacao"
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("_source_file_sha256").alias("arquivos_distintos"),
        F.sort_array(F.collect_set("nome_arquivo")).alias("arquivos")
    )
    .filter(F.col("registros") > 1)
    .withColumn(
        "tipo_repeticao",
        F.when(
            F.col("arquivos_distintos") > 1,
            "ENTRE_ARQUIVOS"
        ).otherwise("NO_MESMO_ARQUIVO")
    )
    .withColumn(
        "status_instalacao",
        F.when(
            F.col("instalacao").isNull(),
            "AUSENTE"
        ).otherwise("INFORMADA")
    )
)

print("2. Resumo das chaves candidatas repetidas")

display(
    repeated_keys.groupBy(
        "dataset", "tipo_repeticao", "status_instalacao"
    )
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum("registros").alias("registros_nos_grupos"),
        F.sum(F.col("registros") - 1).alias("registros_excedentes")
    )
    .orderBy("dataset", "tipo_repeticao", "status_instalacao")
)

print("3. Exemplos de chaves candidatas repetidas")

display(
    repeated_keys
    .orderBy(F.desc("registros"), "dataset", "poco", "competencia")
    .limit(20)
)

1. Integridade dos componentes da chave


dataset,registros,competencias_invalidas,pocos_ausentes,campos_ausentes,instalacoes_ausentes
producao_mar,43919,0,0,0,0
producao_terra,271437,0,0,0,1116


2. Resumo das chaves candidatas repetidas


dataset,tipo_repeticao,status_instalacao,grupos_repetidos,registros_nos_grupos,registros_excedentes


3. Exemplos de chaves candidatas repetidas


dataset,poco,competencia,campo,instalacao,registros,arquivos_distintos,arquivos,tipo_repeticao,status_instalacao


In [0]:
import operator
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
TARGET = "workspace.oil_gas_silver.producao_injecao_mensal_v2"
TRANSFORM_VERSION = "v2"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        "_contract_hash",
        "registros_esperados"
    )
) if False else (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if (
    contracts.groupBy("_source_dataset", "_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

frames = []

for dataset in ["producao_mar", "producao_terra"]:
    hashes = (
        contracts
        .filter(F.col("_source_dataset") == dataset)
        .select("_source_file_sha256")
    )

    frames.append(
        spark.table(f"workspace.oil_gas_bronze.{dataset}")
        .join(hashes, "_source_file_sha256", "inner")
        .withColumn("_source_dataset", F.lit(dataset))
    )

source = frames[0].unionByName(frames[1])

counts = (
    source.groupBy("_source_dataset", "_source_file_sha256")
    .count()
)

count_check = contracts.join(
    counts,
    ["_source_dataset", "_source_file_sha256"],
    "left"
)

if count_check.filter(
    F.col("count").isNull()
    | (F.col("count") != F.col("registros_esperados"))
).limit(1).count():
    raise RuntimeError("Contagem Bronze diferente dos contratos.")

measures = sorted([
    name for name in source.columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")

business_columns = sorted([
    name for name in source.columns
    if not name.startswith("_")
])

source = source.withColumn(
    "_source_record_sha256",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(name) for name in business_columns]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )
)

month_pairs = [
    ("jan", "01"), ("fev", "02"), ("mar", "03"),
    ("abr", "04"), ("mai", "05"), ("jun", "06"),
    ("jul", "07"), ("ago", "08"), ("set", "09"),
    ("out", "10"), ("nov", "11"), ("dez", "12")
]

month_map = F.create_map(*[
    F.lit(value) for pair in month_pairs for value in pair
])

text = F.lower(F.trim(F.col("mes_ano")))

month_number = F.when(
    text.rlike(r"^(0[1-9]|1[0-2])/[0-9]{4}$"),
    F.substring(text, 1, 2)
).when(
    text.rlike(
        r"^(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)/[0-9]{4}$"
    ),
    F.element_at(month_map, F.substring(text, 1, 3))
)

competencia = F.to_date(
    F.try_to_timestamp(
        F.concat(
            F.lit("01/"),
            month_number,
            F.lit("/"),
            F.regexp_extract(text, r"/([0-9]{4})$", 1)
        ),
        F.lit("dd/MM/yyyy")
    )
)

numeric_pattern = (
    r"^[+-]?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+)"
    r"(?:,[0-9]{1,9})?$"
)

converted = {}
invalid_conditions = []

for name in measures:
    raw = F.trim(F.col(name))
    present = raw.isNotNull() & (F.length(raw) > 0)

    value = F.when(
        raw.rlike(numeric_pattern),
        F.expr(
            f"try_cast(replace(replace(`{name}`, '.', ''), ',', '.') "
            "AS DECIMAL(28,9))"
        )
    )

    converted[name] = value

    invalid_conditions.append(
        present & (
            ~raw.rlike(numeric_pattern)
            | value.isNull()
        )
    )

has_invalid_number = reduce(operator.or_, invalid_conditions)

if source.filter(has_invalid_number).limit(1).count():
    raise RuntimeError("Há valores numéricos que não podem ser convertidos.")


def clean_text(name):
    value = F.upper(F.trim(F.col(name)))
    return F.when(F.length(value) > 0, value)


silver = source.select(
    *[F.col(name) for name in source.columns if name.startswith("_")],
    F.expr("try_cast(trim(ano) AS INT)").alias("ano"),
    F.col("mes_ano").alias("mes_ano_origem"),
    competencia.alias("competencia"),
    *[
        clean_text(name).alias(name)
        for name in [
            "estado", "bacia", "campo", "poco", "ambiente", "instalacao"
        ]
    ],
    *[converted[name].alias(name) for name in measures]
)

invalid_required = (
    F.col("competencia").isNull()
    | F.col("ano").isNull()
    | (F.col("ano") != F.year("competencia"))
    | F.col("poco").isNull()
    | F.col("campo").isNull()
    | F.col("ambiente").isNull()
)

if silver.filter(invalid_required).limit(1).count():
    raise RuntimeError("Data, ano ou identificadores essenciais inválidos.")

if silver.filter(
    reduce(operator.or_, [F.col(name) < 0 for name in measures])
).limit(1).count():
    raise RuntimeError("Há medidas negativas.")

has_production = reduce(
    operator.or_,
    [F.col(name).isNotNull() for name in measures
     if name.startswith("producao_de_")]
)

has_injection = reduce(
    operator.or_,
    [F.col(name).isNotNull() for name in measures
     if name.startswith("injecao_de_")]
)

silver = (
    silver
    .withColumn(
        "perfil_medidas",
        F.when(has_production & has_injection, "PRODUCAO_E_INJECAO")
        .when(has_production, "SOMENTE_PRODUCAO")
        .when(has_injection, "SOMENTE_INJECAO")
        .otherwise("SEM_MEDIDAS_INFORMADAS")
    )
    .withColumn(
        "status_instalacao",
        F.when(F.col("instalacao").isNull(), "AUSENTE")
        .otherwise("INFORMADA")
    )
    .withColumn(
        "registro_id",
        F.sha2(
            F.to_json(
                F.struct(
                    "ambiente", "poco", "competencia", "campo", "instalacao"
                ),
                options={"ignoreNullFields": "false"}
            ),
            256
        )
    )
    .withColumn("_transform_version", F.lit(TRANSFORM_VERSION))
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

if (
    silver.groupBy(
        "_source_dataset", "ambiente", "poco",
        "competencia", "campo", "instalacao"
    )
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Chave candidata repetida após a transformação.")

if not spark.catalog.tableExists(TARGET):
    (
        silver.limit(0)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET)
    )

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        silver.alias("source"),
        """
        target._source_dataset = source._source_dataset
        AND target._source_file_sha256 = source._source_file_sha256
        AND target.registro_id = source.registro_id
        AND target._transform_version = source._transform_version
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

published = (
    spark.table(TARGET)
    .filter(F.col("_transform_version") == TRANSFORM_VERSION)
    .join(
        contracts.select("_source_dataset", "_source_file_sha256"),
        ["_source_dataset", "_source_file_sha256"],
        "inner"
    )
)

print(f"Tabela: {TARGET}")

display(
    published.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("registro_id").alias("registros_distintos"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia"),
        F.sum(
            F.when(F.col("instalacao").isNull(), 1).otherwise(0)
        ).alias("instalacoes_ausentes")
    )
    .orderBy("_source_dataset")
)

display(
    published.groupBy("_source_dataset", "perfil_medidas")
    .count()
    .orderBy("_source_dataset", "perfil_medidas")
)

Tabela: workspace.oil_gas_silver.producao_injecao_mensal_v2


_source_dataset,registros,registros_distintos,primeira_competencia,ultima_competencia,instalacoes_ausentes
producao_mar,43919,43919,2024-01-01,2026-08-01,0
producao_terra,271437,271437,2024-01-01,2026-06-01,1116


_source_dataset,perfil_medidas,count
producao_mar,PRODUCAO_E_INJECAO,11
producao_mar,SOMENTE_INJECAO,15538
producao_mar,SOMENTE_PRODUCAO,28370
producao_terra,PRODUCAO_E_INJECAO,35726
producao_terra,SOMENTE_INJECAO,59474
producao_terra,SOMENTE_PRODUCAO,176237


In [0]:
import csv
import re
import unicodedata
import uuid

from decimal import Decimal, localcontext
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
INGESTION_TABLE = "workspace.oil_gas_ops.ingestion_files"
SILVER_TABLE = "workspace.oil_gas_silver.producao_injecao_mensal_v2"
RECON_TABLE = "workspace.oil_gas_ops.silver_v2_measure_reconciliation"

RECON_RUN_ID = str(uuid.uuid4())

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        "ingestion_run_id",
        "dataset",
        "source_url",
        "source_file_sha256",
        "nome_arquivo",
        "registros_esperados"
    )
)

ingestion = (
    spark.table(INGESTION_TABLE)
    .filter(F.col("status").isin("STORED", "REUSED"))
    .select(
        F.col("run_id").alias("ingestion_run_id"),
        "dataset",
        "source_url",
        F.col("file_sha256").alias("source_file_sha256"),
        "file_path"
    )
)

keys = [
    "ingestion_run_id", "dataset", "source_url", "source_file_sha256"
]

files = contracts.join(ingestion, keys, "left")

if (
    files.count() != contracts.count()
    or files.filter(F.col("file_path").isNull()).limit(1).count()
):
    raise RuntimeError("Associação dos contratos com os arquivos inconsistente.")

if (
    contracts.groupBy("dataset", "source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Conteúdos repetidos nos contratos.")

silver = (
    spark.table(SILVER_TABLE)
    .filter(F.col("_transform_version") == "v2")
    .join(
        contracts.select(
            F.col("dataset").alias("_source_dataset"),
            F.col("source_file_sha256").alias("_source_file_sha256")
        ),
        ["_source_dataset", "_source_file_sha256"],
        "inner"
    )
)

measures = sorted([
    name for name in silver.columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")


def normalize_column(name):
    value = (
        unicodedata.normalize("NFKD", name)
        .encode("ascii", "ignore")
        .decode("ascii")
    )
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


pattern = re.compile(
    r"^[+-]?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+)(?:,[0-9]{1,9})?$"
)

reference_rows = []

with localcontext() as context:
    context.prec = 50

    for item in files.orderBy("dataset", "nome_arquivo").collect():
        stats = {
            name: {"total": None, "filled": 0, "zeros": 0}
            for name in measures
        }

        row_count = 0

        with open(
            item["file_path"],
            "r",
            encoding="utf-8-sig",
            newline=""
        ) as source:
            reader = csv.reader(source, delimiter=",", strict=True)
            header = next(reader)
            normalized = [normalize_column(name) for name in header]

            if len(normalized) != len(set(normalized)):
                raise RuntimeError("Cabeçalho com nomes normalizados repetidos.")

            positions = {
                name: normalized.index(name)
                for name in measures
            }

            for row in reader:
                if not row or all(not value.strip() for value in row):
                    continue

                if len(row) != len(header):
                    raise RuntimeError(
                        f"Estrutura inválida em {item['nome_arquivo']}."
                    )

                row_count += 1

                for name, position in positions.items():
                    raw = row[position].strip()

                    if not raw:
                        continue

                    if not pattern.fullmatch(raw):
                        raise RuntimeError(
                            f"Número inválido: {item['nome_arquivo']} | "
                            f"{name} | {raw}"
                        )

                    value = Decimal(raw.replace(".", "").replace(",", "."))

                    current = stats[name]
                    current["total"] = (
                        value if current["total"] is None
                        else current["total"] + value
                    )
                    current["filled"] += 1
                    current["zeros"] += int(value == 0)

        if row_count != item["registros_esperados"]:
            raise RuntimeError(
                f"Contagem do CSV divergente: {item['nome_arquivo']}."
            )

        for name, values in stats.items():
            reference_rows.append((
                item["dataset"],
                item["source_file_sha256"],
                item["nome_arquivo"],
                name,
                row_count,
                values["total"],
                values["filled"],
                values["zeros"]
            ))

reference_schema = """
    dataset STRING,
    source_file_sha256 STRING,
    nome_arquivo STRING,
    medida STRING,
    registros_csv BIGINT,
    total_csv DECIMAL(38,9),
    preenchidos_csv BIGINT,
    zeros_csv BIGINT
"""

reference = spark.createDataFrame(
    reference_rows,
    schema=reference_schema
)

silver_long = (
    silver.select(
        F.col("_source_dataset").alias("dataset"),
        F.col("_source_file_sha256").alias("source_file_sha256"),
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(name).alias("medida"),
                    F.col(name).alias("valor")
                )
                for name in measures
            ])
        ).alias("item")
    )
    .select("dataset", "source_file_sha256", "item.*")
)

silver_stats = (
    silver_long.groupBy("dataset", "source_file_sha256", "medida")
    .agg(
        F.count("*").alias("registros_silver"),
        F.sum("valor").alias("total_silver"),
        F.count("valor").alias("preenchidos_silver"),
        F.sum(
            F.when(F.col("valor") == 0, 1).otherwise(0)
        ).alias("zeros_silver")
    )
)

reconciliation = (
    reference.join(
        silver_stats,
        ["dataset", "source_file_sha256", "medida"],
        "full"
    )
    .withColumn(
        "diferenca",
        F.col("total_silver") - F.col("total_csv")
    )
    .withColumn(
        "status",
        F.when(
            F.col("registros_csv").eqNullSafe(F.col("registros_silver"))
            & F.col("total_csv").eqNullSafe(F.col("total_silver"))
            & F.col("preenchidos_csv").eqNullSafe(
                F.col("preenchidos_silver")
            )
            & F.col("zeros_csv").eqNullSafe(F.col("zeros_silver")),
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn("recon_run_id", F.lit(RECON_RUN_ID))
    .withColumn("contract_run_id", F.lit(latest["contract_run_id"]))
    .withColumn("verificado_em", F.current_timestamp())
)

reconciliation.write.format("delta").mode("append").saveAsTable(RECON_TABLE)

published = (
    spark.table(RECON_TABLE)
    .filter(F.col("recon_run_id") == RECON_RUN_ID)
)

print(f"Reconciliação: {RECON_RUN_ID}")

display(
    published.groupBy("dataset", "status")
    .agg(
        F.count("*").alias("verificacoes_arquivo_medida"),
        F.countDistinct("source_file_sha256").alias("arquivos")
    )
    .orderBy("dataset", "status")
)

display(
    published.filter(F.col("status") != "PASS")
    .select(
        "dataset",
        "nome_arquivo",
        "medida",
        "registros_csv",
        "registros_silver",
        "total_csv",
        "total_silver",
        "diferenca",
        "preenchidos_csv",
        "preenchidos_silver",
        "zeros_csv",
        "zeros_silver",
        "status"
    )
    .orderBy("dataset", "nome_arquivo", "medida")
)

if published.filter(F.col("status") != "PASS").limit(1).count():
    raise RuntimeError("A reconciliação encontrou divergências.")

print("Totais, contagens, preenchimentos e zeros conciliados por arquivo.")

Reconciliação: 5186fc62-65c2-473d-af27-d294b08cbb78


dataset,status,verificacoes_arquivo_medida,arquivos
producao_mar,PASS,39,3
producao_terra,PASS,130,10


dataset,nome_arquivo,medida,registros_csv,registros_silver,total_csv,total_silver,diferenca,preenchidos_csv,preenchidos_silver,zeros_csv,zeros_silver,status


Totais, contagens, preenchimentos e zeros conciliados por arquivo.


In [0]:
import uuid

from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
QUALITY_TABLE = "workspace.oil_gas_ops.expanded_intervention_checks"

CHECK_RUN_ID = str(uuid.uuid4())

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & (F.col("dataset") == "intervencoes")
    )
    .select(
        "nome_arquivo",
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if (
    contracts.groupBy("_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

source = (
    spark.table("workspace.oil_gas_bronze.intervencoes")
    .join(contracts, "_source_file_sha256", "inner")
)

business_columns = sorted([
    name
    for name in spark.table(
        "workspace.oil_gas_bronze.intervencoes"
    ).columns
    if not name.startswith("_")
])


def clean_text(name):
    value = F.upper(F.trim(F.col(name)))
    return F.when(F.length(value) > 0, value)


def present(name):
    return (
        F.col(name).isNotNull()
        & (F.length(F.trim(F.col(name))) > 0)
    )


def parse_date(name):
    value = F.trim(F.col(name))
    return F.when(
        value.rlike(r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$"),
        F.to_date(
            F.try_to_timestamp(value, F.lit("dd/MM/yyyy"))
        )
    )


data = (
    source
    .withColumn(
        "_linha_original_sha256",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(name) for name in business_columns]),
                options={"ignoreNullFields": "false"}
            ),
            256
        )
    )
    .withColumn("_inicio", parse_date("data_inicio"))
    .withColumn("_termino", parse_date("data_termino"))
    .withColumn(
        "_dias",
        F.expr("try_cast(trim(dias_em_intervencao) AS INT)")
    )
    .withColumn("_poco", clean_text("nome_poco_anp"))
    .withColumn("_sonda", clean_text("sonda"))
    .withColumn("_objetivo", clean_text("objetivo"))
    .withColumn(
        "_dias_calendario",
        F.datediff("_termino", "_inicio") + 1
    )
)


def count_when(condition):
    return F.sum(F.when(condition, 1).otherwise(0))


quality = (
    data.groupBy(
        "nome_arquivo", "_source_file_sha256", "registros_esperados"
    )
    .agg(
        F.count("*").alias("registros"),
        count_when(F.col("_poco").isNull()).alias("sem_id_anp"),
        count_when(~present("data_inicio")).alias("inicio_ausente"),
        count_when(
            present("data_inicio") & F.col("_inicio").isNull()
        ).alias("inicio_invalido"),
        count_when(~present("data_termino")).alias("termino_ausente"),
        count_when(
            present("data_termino") & F.col("_termino").isNull()
        ).alias("termino_invalido"),
        count_when(~present("dias_em_intervencao")).alias("dias_ausentes"),
        count_when(
            present("dias_em_intervencao") & F.col("_dias").isNull()
        ).alias("dias_invalidos"),
        count_when(F.col("_dias") < 0).alias("dias_negativos"),
        count_when(
            F.col("_termino") < F.col("_inicio")
        ).alias("termino_antes_do_inicio"),
        count_when(
            F.col("_dias").isNotNull()
            & F.col("_dias_calendario").isNotNull()
        ).alias("duracoes_comparaveis"),
        count_when(
            F.col("_dias") != F.col("_dias_calendario")
        ).alias("duracoes_diferentes"),
        F.min("_inicio").alias("menor_data_inicio"),
        F.max("_inicio").alias("maior_data_inicio"),
        F.max("_termino").alias("maior_data_termino")
    )
    .withColumn("check_run_id", F.lit(CHECK_RUN_ID))
    .withColumn("contract_run_id", F.lit(latest["contract_run_id"]))
    .withColumn("verificado_em", F.current_timestamp())
)

quality.write.format("delta").mode("append").saveAsTable(QUALITY_TABLE)

print("1. Datas, identificação e duração por arquivo")

display(
    spark.table(QUALITY_TABLE)
    .filter(F.col("check_run_id") == CHECK_RUN_ID)
    .drop("check_run_id", "contract_run_id", "verificado_em")
    .orderBy("nome_arquivo")
)

print("2. Linhas de negócio exatamente repetidas")

exact_duplicates = (
    data.groupBy("_linha_original_sha256")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("_source_file_sha256").alias("arquivos_distintos"),
        F.sort_array(F.collect_set("nome_arquivo")).alias("arquivos")
    )
    .filter(F.col("registros") > 1)
)

display(exact_duplicates.orderBy(F.desc("registros")).limit(20))

print("3. Chaves candidatas de eventos repetidas")

event_duplicates = (
    data
    .filter(F.col("_poco").isNotNull())
    .groupBy("_poco", "_inicio", "_termino", "_sonda", "_objetivo")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("_source_file_sha256").alias("arquivos_distintos"),
        F.countDistinct("_linha_original_sha256").alias(
            "conteudos_distintos"
        ),
        F.sort_array(F.collect_set("nome_arquivo")).alias("arquivos")
    )
    .filter(F.col("registros") > 1)
)

display(
    event_duplicates
    .orderBy(F.desc("registros"), "_poco", "_inicio")
    .limit(20)
)

1. Datas, identificação e duração por arquivo


nome_arquivo,_source_file_sha256,registros_esperados,registros,sem_id_anp,inicio_ausente,inicio_invalido,termino_ausente,termino_invalido,dias_ausentes,dias_invalidos,dias_negativos,termino_antes_do_inicio,duracoes_comparaveis,duracoes_diferentes,menor_data_inicio,maior_data_inicio,maior_data_termino
intervencao_em_pocos_2023-2024.csv,7fff3ad158b2a48f1004a3023272a609e8b10b71fcff9446a54cb84e8b7fff9f,9752,9752,0,0,0,0,0,0,0,0,0,9752,0,2023-01-01,2024-12-31,2025-03-31
intervencao_em_pocos_2025-2026.csv,0994f2ec9a1dea96b7f861bbb0c8a93cb25dfe6b431164d73d51f26ce9133361,8065,8065,1,0,0,0,0,0,0,0,0,8065,0,2025-01-01,2026-08-31,2026-09-20


2. Linhas de negócio exatamente repetidas


_linha_original_sha256,registros,arquivos_distintos,arquivos


3. Chaves candidatas de eventos repetidas


_poco,_inicio,_termino,_sonda,_objetivo,registros,arquivos_distintos,conteudos_distintos,arquivos


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
BRONZE_TABLE = "workspace.oil_gas_bronze.intervencoes"
TARGET = "workspace.oil_gas_silver.intervencoes_v2"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & (F.col("dataset") == "intervencoes")
    )
    .select(
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "nome_arquivo",
        "registros_esperados"
    )
)

if contracts.count() == 0:
    raise RuntimeError("Nenhum arquivo de intervenções selecionado.")

if (
    contracts.groupBy("_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

source = (
    spark.table(BRONZE_TABLE)
    .join(
        contracts.select("_source_file_sha256"),
        "_source_file_sha256",
        "inner"
    )
)

source_counts = (
    source.groupBy("_source_file_sha256")
    .count()
)

if (
    contracts.join(source_counts, "_source_file_sha256", "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contagem Bronze diferente dos contratos.")

business_columns = sorted([
    name for name in source.columns
    if not name.startswith("_")
])

source = source.withColumn(
    "_source_record_sha256",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(name) for name in business_columns]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )
)


def clean_text(name):
    value = F.upper(F.trim(F.col(name)))
    return F.when(F.length(value) > 0, value)


def parse_date(name):
    value = F.trim(F.col(name))
    return F.when(
        value.rlike(r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$"),
        F.to_date(
            F.try_to_timestamp(value, F.lit("dd/MM/yyyy"))
        )
    )


text_columns = [
    "nome_poco_anp",
    "nome_poco_operador",
    "campo",
    "bacia",
    "operador_atual",
    "operador_epoca",
    "ambiente",
    "sonda",
    "objetivo"
]

silver = (
    source.select(
        *[F.col(name) for name in source.columns if name.startswith("_")],
        *[clean_text(name).alias(name) for name in text_columns],
        parse_date("data_inicio").alias("data_inicio"),
        parse_date("data_termino").alias("data_termino"),
        F.expr(
            "try_cast(trim(dias_em_intervencao) AS INT)"
        ).alias("dias_em_intervencao")
    )
    .withColumn(
        "dias_calendario_inclusivos",
        F.datediff("data_termino", "data_inicio") + 1
    )
    .withColumn(
        "identificador_anp_presente",
        F.col("nome_poco_anp").isNotNull()
    )
    .withColumn(
        "status_identificacao",
        F.when(
            F.col("nome_poco_anp").isNotNull(),
            "ID_ANP_INFORMADO"
        ).otherwise("SEM_ID_ANP")
    )
    .withColumn("_source_dataset", F.lit("intervencoes"))
    .withColumn("_transform_version", F.lit("v2"))
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

invalid = (
    F.col("data_inicio").isNull()
    | F.col("data_termino").isNull()
    | F.col("dias_em_intervencao").isNull()
    | (F.col("dias_em_intervencao") < 1)
    | (F.col("data_termino") < F.col("data_inicio"))
    | (
        F.col("dias_em_intervencao")
        != F.col("dias_calendario_inclusivos")
    )
)

if silver.filter(invalid).limit(1).count():
    raise RuntimeError("Datas ou duração inconsistentes na transformação.")

if (
    silver.groupBy("_source_file_sha256", "_source_record_sha256")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Linhas originais repetidas no mesmo arquivo.")

if not spark.catalog.tableExists(TARGET):
    (
        silver.limit(0)
        .write.format("delta")
        .mode("append")
        .saveAsTable(TARGET)
    )

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        silver.alias("source"),
        """
        target._source_file_sha256 = source._source_file_sha256
        AND target._source_record_sha256 = source._source_record_sha256
        AND target._transform_version = source._transform_version
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

published = (
    spark.table(TARGET)
    .filter(F.col("_transform_version") == "v2")
    .join(
        contracts.select("_source_file_sha256"),
        "_source_file_sha256",
        "inner"
    )
)


comparison_columns = [
    name for name in silver.columns
    if name != "_silver_loaded_at"
]

expected = silver.select(*comparison_columns)
actual = published.select(*comparison_columns)

if (
    expected.exceptAll(actual).limit(1).count()
    or actual.exceptAll(expected).limit(1).count()
):
    raise RuntimeError("Conteúdo publicado diferente da transformação.")

print(f"Tabela: {TARGET}")
print("Conferência do conteúdo publicado: PASS")

display(
    published.join(contracts, "_source_file_sha256", "inner")
    .groupBy("nome_arquivo", "registros_esperados")
    .agg(
        F.count("*").alias("registros_silver"),
        F.min("data_inicio").alias("primeiro_inicio"),
        F.max("data_termino").alias("ultimo_termino")
    )
    .orderBy("nome_arquivo")
)

display(
    published.groupBy("status_identificacao")
    .count()
    .orderBy("status_identificacao")
)

Tabela: workspace.oil_gas_silver.intervencoes_v2
Conferência do conteúdo publicado: PASS


nome_arquivo,registros_esperados,registros_silver,primeiro_inicio,ultimo_termino
intervencao_em_pocos_2023-2024.csv,9752,9752,2023-01-01,2025-03-31
intervencao_em_pocos_2025-2026.csv,8065,8065,2025-01-01,2026-09-20


status_identificacao,count
ID_ANP_INFORMADO,17816
SEM_ID_ANP,1


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

TARGET = "workspace.oil_gas_gold.dim_data"

calendar = spark.sql("""
    SELECT explode(
        sequence(
            DATE '2023-01-01',
            DATE '2026-12-31',
            INTERVAL 1 DAY
        )
    ) AS data
""")

month_names = F.array(*[
    F.lit(name)
    for name in [
        "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL",
        "MAIO", "JUNHO", "JULHO", "AGOSTO",
        "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
    ]
])

expanded = calendar.select(
    F.date_format("data", "yyyyMMdd").cast("int").alias("data_id"),
    "data",
    F.year("data").alias("ano"),
    F.quarter("data").alias("trimestre"),
    F.month("data").alias("mes"),
    F.element_at(month_names, F.month("data")).alias("nome_mes"),
    F.date_format("data", "yyyyMM").cast("int").alias("mes_id"),
    F.dayofmonth("data").alias("dia"),
    F.trunc("data", "month").alias("primeiro_dia_mes"),
    F.last_day("data").alias("ultimo_dia_mes"),
    (F.dayofmonth("data") == 1).alias("eh_primeiro_dia_mes"),
    F.dayofweek("data").isin(1, 7).alias("eh_fim_de_semana")
)

existing = spark.table(TARGET)

for key in ["data_id", "data"]:
    if (
        existing.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Dimensão existente com duplicidade em {key}.")

if existing.filter(
    F.col("data_id").isNull() | F.col("data").isNull()
).limit(1).count():
    raise RuntimeError("Dimensão existente com chave ou data ausente.")

overlap = existing.join(
    expanded.select("data_id"),
    "data_id",
    "inner"
).select(*expanded.columns)

expected_overlap = expanded.join(
    existing.select("data_id"),
    "data_id",
    "inner"
).select(*expanded.columns)

if (
    overlap.exceptAll(expected_overlap).limit(1).count()
    or expected_overlap.exceptAll(overlap).limit(1).count()
):
    raise RuntimeError(
        "Há diferenças nos atributos dos dias existentes. "
        "Revisar antes de ampliar a dimensão."
    )

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        expanded.alias("source"),
        "target.data_id = source.data_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

published = (
    spark.table(TARGET)
    .filter(
        F.col("data").between(
            F.lit("2023-01-01").cast("date"),
            F.lit("2026-12-31").cast("date")
        )
    )
    .select(*expanded.columns)
)

if (
    expanded.exceptAll(published).limit(1).count()
    or published.exceptAll(expanded).limit(1).count()
):
    raise RuntimeError("Calendário publicado diferente do esperado.")

print(f"Tabela: {TARGET}")
print(f"Dias no intervalo: {published.count():,}")
print("Conferência do calendário: PASS")

display(
    published.groupBy("ano")
    .agg(
        F.count("*").alias("dias"),
        F.min("data").alias("inicio"),
        F.max("data").alias("fim")
    )
    .orderBy("ano")
)

display(
    published
    .filter(F.col("data") == F.lit("2024-02-29").cast("date"))
    .select("data_id", "data", "ano", "mes", "dia", "ultimo_dia_mes")
)

Tabela: workspace.oil_gas_gold.dim_data
Dias no intervalo: 1,461
Conferência do calendário: PASS


ano,dias,inicio,fim
2023,365,2023-01-01,2023-12-31
2024,366,2024-01-01,2024-12-31
2025,365,2025-01-01,2025-12-31
2026,365,2026-01-01,2026-12-31


data_id,data,ano,mes,dia,ultimo_dia_mes
20240229,2024-02-29,2024,2,29,2024-02-29


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
TARGET = "workspace.oil_gas_gold.dim_poco"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin(
            "producao_mar", "producao_terra", "intervencoes"
        )
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if (
    contracts.groupBy("_source_dataset", "_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

production = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal_v2")
    .filter(F.col("_transform_version") == "v2")
    .select(
        "_source_dataset",
        "_source_file_sha256",
        F.col("poco").alias("nome_poco_anp")
    )
)

interventions = (
    spark.table("workspace.oil_gas_silver.intervencoes_v2")
    .filter(F.col("_transform_version") == "v2")
    .select(
        "_source_dataset",
        "_source_file_sha256",
        "nome_poco_anp"
    )
)

sources = (
    production.unionByName(interventions)
    .join(
        contracts.select("_source_dataset", "_source_file_sha256"),
        ["_source_dataset", "_source_file_sha256"],
        "inner"
    )
)

counts = sources.groupBy(
    "_source_dataset", "_source_file_sha256"
).count()

if (
    contracts.join(
        counts,
        ["_source_dataset", "_source_file_sha256"],
        "left"
    )
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("A Silver não contém todos os registros contratados.")

known = (
    sources.select("nome_poco_anp")
    .filter(F.col("nome_poco_anp").isNotNull())
    .distinct()
    .select(
        F.sha2("nome_poco_anp", 256).alias("poco_id"),
        "nome_poco_anp",
        F.lit("ID_ANP_INFORMADO").alias("tipo_membro")
    )
)

unknown = spark.createDataFrame(
    [("SEM_ID_ANP", None, "SEM_ID_ANP")],
    "poco_id STRING, nome_poco_anp STRING, tipo_membro STRING"
)

candidates = known.unionByName(unknown)

existing = spark.table(TARGET)

for frame, label in [
    (existing, "dimensão existente"),
    (candidates, "membros candidatos")
]:
    if (
        frame.groupBy("poco_id")
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Chave repetida em {label}.")

    if frame.filter(F.col("poco_id").isNull()).limit(1).count():
        raise RuntimeError(f"Chave ausente em {label}.")

columns = ["poco_id", "nome_poco_anp", "tipo_membro"]

current_overlap = (
    existing.join(candidates.select("poco_id"), "poco_id", "inner")
    .select(*columns)
)

expected_overlap = (
    candidates.join(existing.select("poco_id"), "poco_id", "inner")
    .select(*columns)
)

if (
    current_overlap.exceptAll(expected_overlap).limit(1).count()
    or expected_overlap.exceptAll(current_overlap).limit(1).count()
):
    raise RuntimeError("Membro existente diferente da regra de identificação.")

new_members_count = (
    candidates.join(existing.select("poco_id"), "poco_id", "left_anti")
    .count()
)

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        candidates.alias("source"),
        "target.poco_id = source.poco_id"
    )
    .whenNotMatchedInsert(values={
        "poco_id": "source.poco_id",
        "nome_poco_anp": "source.nome_poco_anp",
        "tipo_membro": "source.tipo_membro"
    })
    .execute()
)

published = spark.table(TARGET)

if (
    candidates.select(*columns)
    .exceptAll(published.select(*columns))
    .limit(1)
    .count()
):
    raise RuntimeError("Há membros candidatos ausentes ou divergentes.")

linked = (
    sources
    .withColumn(
        "_poco_id_esperado",
        F.when(
            F.col("nome_poco_anp").isNotNull(),
            F.sha2("nome_poco_anp", 256)
        ).otherwise(F.lit("SEM_ID_ANP"))
    )
    .join(
        published.select(
            F.col("poco_id").alias("_poco_id_encontrado")
        ),
        F.col("_poco_id_esperado") == F.col("_poco_id_encontrado"),
        "left"
    )
)

if linked.count() != sources.count():
    raise RuntimeError("O vínculo dimensional multiplicou registros.")

if linked.filter(
    F.col("_poco_id_encontrado").isNull()
).limit(1).count():
    raise RuntimeError("Há registros sem membro dimensional correspondente.")

print(f"Tabela: {TARGET}")
print(f"Novos membros inseridos: {new_members_count:,}")
print("Cobertura dos registros da Silver: PASS")

display(
    published.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

display(
    linked.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("nome_poco_anp").alias("identificadores_distintos"),
        F.sum(
            F.when(F.col("nome_poco_anp").isNull(), 1).otherwise(0)
        ).alias("registros_sem_id_anp")
    )
    .orderBy("_source_dataset")
)

Tabela: workspace.oil_gas_gold.dim_poco
Novos membros inseridos: 0
Cobertura dos registros da Silver: PASS


tipo_membro,count
ID_ANP_INFORMADO,27205
SEM_ID_ANP,1


_source_dataset,registros,identificadores_distintos,registros_sem_id_anp
intervencoes,17817,8419,1
producao_mar,43919,1083,0
producao_terra,271437,11446,0


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
TARGET = "workspace.oil_gas_gold.dim_campo"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin(
            "producao_mar", "producao_terra", "intervencoes"
        )
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

keys = ["_source_dataset", "_source_file_sha256"]

if (
    contracts.groupBy(*keys)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")


def read_context(table):
    return (
        spark.table(table)
        .filter(F.col("_transform_version") == "v2")
        .select(
            *keys,
            F.col("campo").alias("nome_campo_reportado"),
            F.col("bacia").alias("bacia_reportada")
        )
    )


sources = (
    read_context(
        "workspace.oil_gas_silver.producao_injecao_mensal_v2"
    )
    .unionByName(
        read_context("workspace.oil_gas_silver.intervencoes_v2")
    )
    .join(contracts.select(*keys), keys, "inner")
)

counts = sources.groupBy(*keys).count()

if (
    contracts.join(counts, keys, "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contagem das fontes diferente dos contratos.")

complete = (
    F.col("nome_campo_reportado").isNotNull()
    & (F.length(F.trim(F.col("nome_campo_reportado"))) > 0)
    & F.col("bacia_reportada").isNotNull()
    & (F.length(F.trim(F.col("bacia_reportada"))) > 0)
)

context_key = F.when(
    complete,
    F.sha2(
        F.to_json(
            F.struct(
                F.lit("CAMPO_REPORTADO_V1").alias("dominio"),
                F.col("nome_campo_reportado"),
                F.col("bacia_reportada")
            )
        ),
        256
    )
).otherwise(F.lit("SEM_CONTEXTO_CAMPO"))

sources = sources.withColumn("campo_id", context_key)

known = (
    sources.filter(complete)
    .select("campo_id", "nome_campo_reportado", "bacia_reportada")
    .distinct()
    .withColumn("tipo_membro", F.lit("CONTEXTO_INFORMADO"))
)

unknown = spark.createDataFrame(
    [("SEM_CONTEXTO_CAMPO", None, None, "CONTEXTO_INCOMPLETO")],
    """
    campo_id STRING,
    nome_campo_reportado STRING,
    bacia_reportada STRING,
    tipo_membro STRING
    """
)

candidates = known.unionByName(unknown)
columns = candidates.columns
existing = spark.table(TARGET)

for frame, label in [
    (existing, "dimensão existente"),
    (candidates, "membros candidatos")
]:
    if (
        frame.groupBy("campo_id")
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Chave repetida em {label}.")

    if frame.filter(F.col("campo_id").isNull()).limit(1).count():
        raise RuntimeError(f"Chave ausente em {label}.")

existing_overlap = (
    existing.join(candidates.select("campo_id"), "campo_id", "inner")
    .select(*columns)
)

expected_overlap = (
    candidates.join(existing.select("campo_id"), "campo_id", "inner")
    .select(*columns)
)

if (
    existing_overlap.exceptAll(expected_overlap).limit(1).count()
    or expected_overlap.exceptAll(existing_overlap).limit(1).count()
):
    raise RuntimeError("Contexto existente diferente da regra do piloto.")

new_count = (
    candidates.join(existing.select("campo_id"), "campo_id", "left_anti")
    .count()
)

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        candidates.alias("source"),
        "target.campo_id = source.campo_id"
    )
    .whenNotMatchedInsert(values={
        name: f"source.{name}" for name in columns
    })
    .execute()
)

published = spark.table(TARGET)

if (
    candidates.select(*columns)
    .exceptAll(published.select(*columns))
    .limit(1)
    .count()
):
    raise RuntimeError("Contexto candidato ausente ou divergente.")

linked = sources.join(
    published.select(
        F.col("campo_id").alias("_campo_id_encontrado")
    ),
    F.col("campo_id") == F.col("_campo_id_encontrado"),
    "left"
)

if (
    linked.count() != sources.count()
    or linked.filter(F.col("_campo_id_encontrado").isNull()).limit(1).count()
):
    raise RuntimeError("Falha de cobertura ou multiplicação no vínculo.")

print(f"Tabela: {TARGET}")
print(f"Novos contextos inseridos: {new_count}")
print("Cobertura dos registros da Silver: PASS")

display(
    published.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

display(
    linked.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("campo_id").alias("contextos_distintos"),
        F.sum(
            F.when(
                F.col("campo_id") == "SEM_CONTEXTO_CAMPO", 1
            ).otherwise(0)
        ).alias("registros_com_contexto_incompleto")
    )
    .orderBy("_source_dataset")
)

Tabela: workspace.oil_gas_gold.dim_campo
Novos contextos inseridos: 11
Cobertura dos registros da Silver: PASS


tipo_membro,count
CONTEXTO_INCOMPLETO,1
CONTEXTO_INFORMADO,550


_source_dataset,registros,contextos_distintos,registros_com_contexto_incompleto
intervencoes,17817,295,0
producao_mar,43919,80,0
producao_terra,271437,250,0


In [0]:
from functools import reduce
import operator

from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
TARGET = "workspace.oil_gas_gold.dim_instalacao"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

keys = ["_source_dataset", "_source_file_sha256"]

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if contracts.count() == 0:
    raise RuntimeError("Nenhum contrato de produção encontrado.")

if (
    contracts.groupBy(*keys)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

sources = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal_v2")
    .filter(F.col("_transform_version") == "v2")
    .join(contracts.select(*keys), keys, "inner")
    .select(
        *keys,
        F.col("instalacao").alias("nome_instalacao_reportado"),
        F.col("ambiente").alias("ambiente_reportado"),
        F.col("estado").alias("estado_reportado"),
        F.col("bacia").alias("bacia_reportada")
    )
)

counts = sources.groupBy(*keys).count()

if (
    contracts.join(counts, keys, "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contagem da Silver diferente dos contratos.")

attributes = [
    "nome_instalacao_reportado",
    "ambiente_reportado",
    "estado_reportado",
    "bacia_reportada"
]

complete = reduce(
    operator.and_,
    [
        F.col(name).isNotNull()
        & (F.length(F.trim(F.col(name))) > 0)
        for name in attributes
    ]
)

sources = sources.withColumn(
    "instalacao_id",
    F.when(
        complete,
        F.sha2(
            F.to_json(
                F.struct(
                    F.lit("INSTALACAO_REPORTADA_V1").alias("dominio"),
                    *[F.col(name) for name in attributes]
                )
            ),
            256
        )
    ).otherwise(F.lit("SEM_CONTEXTO_INSTALACAO"))
)

known = (
    sources.filter(complete)
    .select("instalacao_id", *attributes)
    .distinct()
    .withColumn("tipo_membro", F.lit("CONTEXTO_INFORMADO"))
)

unknown = spark.createDataFrame(
    [(
        "SEM_CONTEXTO_INSTALACAO",
        None, None, None, None,
        "CONTEXTO_INCOMPLETO"
    )],
    """
    instalacao_id STRING,
    nome_instalacao_reportado STRING,
    ambiente_reportado STRING,
    estado_reportado STRING,
    bacia_reportada STRING,
    tipo_membro STRING
    """
)

candidates = known.unionByName(unknown)
columns = candidates.columns
existing = spark.table(TARGET)

for frame, label in [
    (existing, "dimensão existente"),
    (candidates, "membros candidatos")
]:
    if (
        frame.groupBy("instalacao_id")
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Chave repetida em {label}.")

    if frame.filter(F.col("instalacao_id").isNull()).limit(1).count():
        raise RuntimeError(f"Chave ausente em {label}.")

existing_overlap = (
    existing.join(
        candidates.select("instalacao_id"),
        "instalacao_id",
        "inner"
    )
    .select(*columns)
)

expected_overlap = (
    candidates.join(
        existing.select("instalacao_id"),
        "instalacao_id",
        "inner"
    )
    .select(*columns)
)

if (
    existing_overlap.exceptAll(expected_overlap).limit(1).count()
    or expected_overlap.exceptAll(existing_overlap).limit(1).count()
):
    raise RuntimeError("Contexto existente diferente da regra do piloto.")

new_count = (
    candidates.join(
        existing.select("instalacao_id"),
        "instalacao_id",
        "left_anti"
    )
    .count()
)

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        candidates.alias("source"),
        "target.instalacao_id = source.instalacao_id"
    )
    .whenNotMatchedInsert(values={
        name: f"source.{name}" for name in columns
    })
    .execute()
)

published = spark.table(TARGET)

if (
    candidates.select(*columns)
    .exceptAll(published.select(*columns))
    .limit(1)
    .count()
):
    raise RuntimeError("Contexto candidato ausente ou divergente.")

linked = sources.join(
    published.select(
        F.col("instalacao_id").alias("_instalacao_id_encontrado")
    ),
    F.col("instalacao_id") == F.col("_instalacao_id_encontrado"),
    "left"
)

if (
    linked.count() != sources.count()
    or linked.filter(
        F.col("_instalacao_id_encontrado").isNull()
    ).limit(1).count()
):
    raise RuntimeError("Falha de cobertura ou multiplicação no vínculo.")

print(f"Tabela: {TARGET}")
print(f"Novos contextos inseridos: {new_count}")
print("Cobertura dos registros da Silver: PASS")

display(
    published.groupBy("tipo_membro")
    .count()
    .orderBy("tipo_membro")
)

display(
    linked.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("instalacao_id").alias("contextos_distintos"),
        F.sum(
            F.when(
                F.col("instalacao_id") == "SEM_CONTEXTO_INSTALACAO",
                1
            ).otherwise(0)
        ).alias("registros_com_contexto_incompleto")
    )
    .orderBy("_source_dataset")
)

Tabela: workspace.oil_gas_gold.dim_instalacao
Novos contextos inseridos: 93
Cobertura dos registros da Silver: PASS


tipo_membro,count
CONTEXTO_INCOMPLETO,1
CONTEXTO_INFORMADO,506


_source_dataset,registros,contextos_distintos,registros_com_contexto_incompleto
producao_mar,43919,89,16
producao_terra,271437,419,1116


In [0]:
from functools import reduce
import operator

from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

files = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & (F.col("dataset") == "producao_mar")
    )
    .select(
        "nome_arquivo",
        F.col("source_file_sha256").alias("_source_file_sha256")
    )
)

if files.count() == 0:
    raise RuntimeError("Nenhum arquivo de produção marítima selecionado.")

if (
    files.groupBy("_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Arquivos repetidos na seleção.")

data = (
    spark.table("workspace.oil_gas_silver.producao_injecao_mensal_v2")
    .filter(
        (F.col("_transform_version") == "v2")
        & (F.col("_source_dataset") == "producao_mar")
    )
    .join(files, "_source_file_sha256", "inner")
)

attributes = ["instalacao", "ambiente", "estado", "bacia"]

missing_conditions = {
    name: (
        F.col(name).isNull()
        | (F.length(F.trim(F.col(name))) == 0)
    )
    for name in attributes
}

exceptions = (
    data
    .filter(reduce(operator.or_, missing_conditions.values()))
    .withColumn(
        "componentes_ausentes",
        F.concat_ws(
            ", ",
            *[
                F.when(condition, F.lit(name))
                for name, condition in missing_conditions.items()
            ]
        )
    )
)

display(
    exceptions.groupBy("nome_arquivo", "componentes_ausentes")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("poco").alias("pocos_distintos"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia")
    )
    .orderBy("nome_arquivo", "componentes_ausentes")
)

display(
    exceptions.select(
        "nome_arquivo",
        "poco",
        "competencia",
        "campo",
        "instalacao",
        "ambiente",
        "estado",
        "bacia",
        "componentes_ausentes",
        "perfil_medidas",
        "producao_de_oleo_m3"
    )
    .orderBy("nome_arquivo", "poco", "competencia")
    .limit(30)
)

nome_arquivo,componentes_ausentes,registros,pocos_distintos,primeira_competencia,ultima_competencia
producao-mar-2026.csv,estado,16,3,2026-02-01,2026-08-01


nome_arquivo,poco,competencia,campo,instalacao,ambiente,estado,bacia,componentes_ausentes,perfil_medidas,producao_de_oleo_m3
producao-mar-2026.csv,7-NAU-2H-ESS,2026-02-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,101.630000000
producao-mar-2026.csv,7-NAU-2H-ESS,2026-03-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,171.800000000
producao-mar-2026.csv,7-NAU-2H-ESS,2026-04-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,88.540000000
producao-mar-2026.csv,7-NAU-2H-ESS,2026-06-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,34.940000000
producao-mar-2026.csv,7-NAU-2H-ESS,2026-08-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,147.550000000
producao-mar-2026.csv,7-NAU-3H-ESS,2026-02-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,132.520000000
producao-mar-2026.csv,7-NAU-3H-ESS,2026-03-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,267.910000000
producao-mar-2026.csv,7-NAU-3H-ESS,2026-04-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,262.060000000
producao-mar-2026.csv,7-NAU-3H-ESS,2026-05-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,242.280000000
producao-mar-2026.csv,7-NAU-3H-ESS,2026-06-01,ANC_ARGONAUTA_ME1,FPSO ESPIRITO SANTO,MAR,null,CAMPOS,estado,SOMENTE_PRODUCAO,218.000000000


In [0]:
from functools import reduce
import operator

from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
SILVER_TABLE = "workspace.oil_gas_silver.producao_injecao_mensal_v2"
TARGET = "workspace.oil_gas_gold.fato_producao_injecao_mensal_v2"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

keys = ["_source_dataset", "_source_file_sha256"]

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin("producao_mar", "producao_terra")
    )
    .select(
        F.col("dataset").alias("_source_dataset"),
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if contracts.count() == 0:
    raise RuntimeError("Nenhum contrato de produção encontrado.")

if (
    contracts.groupBy(*keys)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

source = (
    spark.table(SILVER_TABLE)
    .filter(F.col("_transform_version") == "v2")
    .join(contracts.select(*keys), keys, "inner")
)

counts = source.groupBy(*keys).count()

if (
    contracts.join(counts, keys, "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contagem Silver diferente dos contratos.")


def complete_columns(names):
    return reduce(
        operator.and_,
        [
            F.col(name).isNotNull()
            & (F.length(F.trim(F.col(name))) > 0)
            for name in names
        ]
    )


field_complete = complete_columns(["campo", "bacia"])

installation_complete = complete_columns([
    "instalacao", "ambiente", "estado", "bacia"
])

prepared = (
    source
    .withColumn(
        "data_id",
        F.date_format("competencia", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "poco_id",
        F.when(
            F.col("poco").isNotNull(),
            F.sha2("poco", 256)
        ).otherwise(F.lit("SEM_ID_ANP"))
    )
    .withColumn(
        "campo_id",
        F.when(
            field_complete,
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit("CAMPO_REPORTADO_V1").alias("dominio"),
                        F.col("campo").alias("nome_campo_reportado"),
                        F.col("bacia").alias("bacia_reportada")
                    )
                ),
                256
            )
        ).otherwise(F.lit("SEM_CONTEXTO_CAMPO"))
    )
    .withColumn(
        "instalacao_id",
        F.when(
            installation_complete,
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit("INSTALACAO_REPORTADA_V1").alias("dominio"),
                        F.col("instalacao").alias("nome_instalacao_reportado"),
                        F.col("ambiente").alias("ambiente_reportado"),
                        F.col("estado").alias("estado_reportado"),
                        F.col("bacia").alias("bacia_reportada")
                    )
                ),
                256
            )
        ).otherwise(F.lit("SEM_CONTEXTO_INSTALACAO"))
    )
    .withColumn(
        "status_contexto_instalacao",
        F.when(
            installation_complete, "CONTEXTO_INFORMADO"
        ).otherwise("CONTEXTO_INCOMPLETO")
    )
)

dimensions = [
    ("data_id", "workspace.oil_gas_gold.dim_data"),
    ("poco_id", "workspace.oil_gas_gold.dim_poco"),
    ("campo_id", "workspace.oil_gas_gold.dim_campo"),
    ("instalacao_id", "workspace.oil_gas_gold.dim_instalacao")
]

for key, table in dimensions:
    dimension = spark.table(table).select(key)

    if (
        dimension.groupBy(key)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Chave duplicada em {table}.")

    if prepared.filter(F.col(key).isNull()).limit(1).count():
        raise RuntimeError(f"Chave ausente nos fatos: {key}.")

    if (
        prepared.select(key).distinct()
        .join(dimension, key, "left_anti")
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Fatos sem correspondência em {table}.")

measures = sorted([
    name for name in source.columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")

facts = (
    prepared.select(
        "registro_id",
        "data_id",
        "poco_id",
        "campo_id",
        "instalacao_id",
        *measures,
        "perfil_medidas",
        "status_instalacao",
        "status_contexto_instalacao",

        F.col("ambiente").alias("ambiente_reportado"),
        F.col("estado").alias("estado_reportado"),
        F.col("bacia").alias("bacia_reportada"),
        F.col("campo").alias("campo_reportado"),
        F.col("instalacao").alias("instalacao_reportada"),

        *[
            F.col(name)
            for name in source.columns
            if name.startswith("_") and name != "_transform_version"
        ],
        F.col("_transform_version").alias("_silver_transform_version")
    )
    .withColumn("_gold_transform_version", F.lit("v2"))
    .withColumn(
        "fato_id",
        F.sha2(
            F.to_json(
                F.struct(
                    F.lit("FATO_PRODUCAO_INJECAO_V2").alias("dominio"),
                    "_source_dataset",
                    "_source_file_sha256",
                    "registro_id",
                    "_gold_transform_version"
                )
            ),
            256
        )
    )
    .withColumn("_gold_loaded_at", F.current_timestamp())
)

if (
    facts.groupBy("fato_id")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Identificador de fato repetido.")

if not spark.catalog.tableExists(TARGET):
    (
        facts.limit(0)
        .write.format("delta")
        .mode("append")
        .saveAsTable(TARGET)
    )

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        facts.alias("source"),
        "target.fato_id = source.fato_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

published = (
    spark.table(TARGET)
    .filter(F.col("_gold_transform_version") == "v2")
    .join(contracts.select(*keys), keys, "inner")
)

columns = [
    name for name in facts.columns
    if name != "_gold_loaded_at"
]

expected = facts.select(*columns)
actual = published.select(*columns)

if (
    expected.exceptAll(actual).limit(1).count()
    or actual.exceptAll(expected).limit(1).count()
):
    raise RuntimeError("Fatos publicados diferentes da transformação.")

print(f"Tabela: {TARGET}")
print("Integridade dimensional e conteúdo publicado: PASS")

display(
    published.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("fato_id").alias("fatos_distintos"),
        F.countDistinct("poco_id").alias("pocos_distintos"),
        F.countDistinct("campo_id").alias("campos_distintos"),
        F.countDistinct("instalacao_id").alias("membros_instalacao_distintos")
    )
    .orderBy("_source_dataset")
)

display(
    published.groupBy(
        "_source_dataset",
        "status_instalacao",
        "status_contexto_instalacao"
    )
    .count()
    .orderBy(
        "_source_dataset",
        "status_instalacao",
        "status_contexto_instalacao"
    )
)

Tabela: workspace.oil_gas_gold.fato_producao_injecao_mensal_v2
Integridade dimensional e conteúdo publicado: PASS


_source_dataset,registros,fatos_distintos,pocos_distintos,campos_distintos,membros_instalacao_distintos
producao_mar,43919,43919,1083,80,89
producao_terra,271437,271437,11446,250,419


_source_dataset,status_instalacao,status_contexto_instalacao,count
producao_mar,INFORMADA,CONTEXTO_INCOMPLETO,16
producao_mar,INFORMADA,CONTEXTO_INFORMADO,43903
producao_terra,AUSENTE,CONTEXTO_INCOMPLETO,1116
producao_terra,INFORMADA,CONTEXTO_INFORMADO,270321


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
SILVER_TABLE = "workspace.oil_gas_silver.intervencoes_v2"
TARGET = "workspace.oil_gas_gold.fato_intervencoes_v2"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & (F.col("dataset") == "intervencoes")
    )
    .select(
        F.col("source_file_sha256").alias("_source_file_sha256"),
        "registros_esperados"
    )
)

if contracts.count() == 0:
    raise RuntimeError("Nenhum contrato de intervenções encontrado.")

if (
    contracts.groupBy("_source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Contratos repetidos para o mesmo conteúdo.")

source = (
    spark.table(SILVER_TABLE)
    .filter(F.col("_transform_version") == "v2")
    .join(
        contracts.select("_source_file_sha256"),
        "_source_file_sha256",
        "inner"
    )
)

counts = source.groupBy("_source_file_sha256").count()

if (
    contracts.join(counts, "_source_file_sha256", "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Contagem Silver diferente dos contratos.")

field_complete = (
    F.col("campo").isNotNull()
    & (F.length(F.trim(F.col("campo"))) > 0)
    & F.col("bacia").isNotNull()
    & (F.length(F.trim(F.col("bacia"))) > 0)
)

prepared = (
    source
    .withColumn(
        "data_inicio_id",
        F.date_format("data_inicio", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "data_termino_id",
        F.date_format("data_termino", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "poco_id",
        F.when(
            F.col("nome_poco_anp").isNotNull(),
            F.sha2("nome_poco_anp", 256)
        ).otherwise(F.lit("SEM_ID_ANP"))
    )
    .withColumn(
        "campo_id",
        F.when(
            field_complete,
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit("CAMPO_REPORTADO_V1").alias("dominio"),
                        F.col("campo").alias("nome_campo_reportado"),
                        F.col("bacia").alias("bacia_reportada")
                    )
                ),
                256
            )
        ).otherwise(F.lit("SEM_CONTEXTO_CAMPO"))
    )
)

dimensions = [
    ("data_inicio_id", "workspace.oil_gas_gold.dim_data", "data_id"),
    ("data_termino_id", "workspace.oil_gas_gold.dim_data", "data_id"),
    ("poco_id", "workspace.oil_gas_gold.dim_poco", "poco_id"),
    ("campo_id", "workspace.oil_gas_gold.dim_campo", "campo_id")
]

for fact_key, table, dimension_key in dimensions:
    dimension = spark.table(table).select(
        F.col(dimension_key).alias(fact_key)
    )

    if (
        dimension.groupBy(fact_key)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Chave duplicada em {table}.")

    if prepared.filter(F.col(fact_key).isNull()).limit(1).count():
        raise RuntimeError(f"Chave ausente nos fatos: {fact_key}.")

    if (
        prepared.select(fact_key).distinct()
        .join(dimension, fact_key, "left_anti")
        .limit(1)
        .count()
    ):
        raise RuntimeError(f"Fatos sem correspondência em {table}.")

facts = (
    prepared.select(
        "data_inicio_id",
        "data_termino_id",
        "poco_id",
        "campo_id",
        "nome_poco_operador",
        "ambiente",
        "sonda",
        "objetivo",
        "operador_epoca",
        F.col("operador_atual").alias(
            "operador_atual_reportado_na_coleta"
        ),
        "status_identificacao",
        "dias_em_intervencao",
        "dias_calendario_inclusivos",
        *[
            F.col(name)
            for name in source.columns
            if name.startswith("_") and name != "_transform_version"
        ],
        F.col("_transform_version").alias("_silver_transform_version")
    )
    .withColumn("quantidade_intervencoes", F.lit(1))
    .withColumn("_gold_transform_version", F.lit("v2"))
    .withColumn(
        "fato_id",
        F.sha2(
            F.to_json(
                F.struct(
                    F.lit("FATO_INTERVENCAO_V2").alias("dominio"),
                    "_source_file_sha256",
                    "_source_record_sha256",
                    "_gold_transform_version"
                )
            ),
            256
        )
    )
    .withColumn("_gold_loaded_at", F.current_timestamp())
)

if (
    facts.groupBy("fato_id")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Identificador de fato repetido.")

if not spark.catalog.tableExists(TARGET):
    (
        facts.limit(0)
        .write.format("delta")
        .mode("append")
        .saveAsTable(TARGET)
    )

(
    DeltaTable.forName(spark, TARGET)
    .alias("target")
    .merge(
        facts.alias("source"),
        "target.fato_id = source.fato_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

published = (
    spark.table(TARGET)
    .filter(F.col("_gold_transform_version") == "v2")
    .join(
        contracts.select("_source_file_sha256"),
        "_source_file_sha256",
        "inner"
    )
)

columns = [
    name for name in facts.columns
    if name != "_gold_loaded_at"
]

expected = facts.select(*columns)
actual = published.select(*columns)

if (
    expected.exceptAll(actual).limit(1).count()
    or actual.exceptAll(expected).limit(1).count()
):
    raise RuntimeError("Fatos publicados diferentes da transformação.")

print(f"Tabela: {TARGET}")
print("Integridade dimensional e conteúdo publicado: PASS")

display(
    published.groupBy("status_identificacao")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("fato_id").alias("fatos_distintos"),
        F.countDistinct("poco_id").alias("membros_dim_poco")
    )
    .orderBy("status_identificacao")
)

display(
    published.groupBy("ambiente", "objetivo")
    .agg(F.sum("quantidade_intervencoes").alias("intervencoes"))
    .orderBy("ambiente", F.desc("intervencoes"))
)

Tabela: workspace.oil_gas_gold.fato_intervencoes_v2
Integridade dimensional e conteúdo publicado: PASS


status_identificacao,registros,fatos_distintos,membros_dim_poco
ID_ANP_INFORMADO,17816,17816,8419
SEM_ID_ANP,1,1,1


ambiente,objetivo,intervencoes
MAR,ABANDONO,515
MAR,PERFURAÇÃO,495
MAR,RESTAURAÇÃO,431
MAR,COMPLETAÇÃO,315
MAR,AVALIAÇÃO,31
MAR,TLD,2
TERRA,RESTAURAÇÃO,10520
TERRA,COMPLETAÇÃO,4071
TERRA,PERFURAÇÃO,665
TERRA,ABANDONO,595


In [0]:
from pyspark.sql import functions as F

CONTRACT_TABLE = "workspace.oil_gas_ops.source_file_read_contracts"
SCOPE_TABLE = "workspace.oil_gas_ops.analytical_scope_sources"

SCOPE_ID = "historico_anp_v2"
PILOT_SCOPE_ID = "piloto_anp_v1"

contracts_table = spark.table(CONTRACT_TABLE)

latest = (
    contracts_table.groupBy("contract_run_id")
    .agg(F.max("registrado_em").alias("registrado_em"))
    .orderBy(F.desc("registrado_em"), F.desc("contract_run_id"))
    .first()
)

if latest is None:
    raise RuntimeError("Nenhum contrato encontrado.")

contracts = (
    contracts_table
    .filter(
        (F.col("contract_run_id") == latest["contract_run_id"])
        & F.col("dataset").isin(
            "producao_mar", "producao_terra", "intervencoes"
        )
    )
    .select(
        "dataset",
        "source_file_sha256",
        "registros_esperados"
    )
)

keys = ["dataset", "source_file_sha256"]

if contracts.count() == 0:
    raise RuntimeError("Nenhum arquivo contratado.")

if (
    contracts.groupBy(*keys)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Conteúdos repetidos nos contratos.")

fact_tables = [
    "workspace.oil_gas_gold.fato_producao_injecao_mensal_v2",
    "workspace.oil_gas_gold.fato_intervencoes_v2"
]

frames = []

for table in fact_tables:
    frames.append(
        spark.table(table)
        .filter(F.col("_gold_transform_version") == "v2")
        .select(
            F.col("_source_dataset").alias("dataset"),
            F.col("_source_file_sha256").alias("source_file_sha256"),
            F.col("_gold_transform_version").alias(
                "gold_transform_version"
            ),
            F.col("_ingestion_run_id").alias("ingestion_run_id")
        )
    )

facts = (
    frames[0].unionByName(frames[1])
    .join(contracts.select(*keys), keys, "inner")
)

counts = facts.groupBy(*keys).count()

if (
    contracts.join(counts, keys, "left")
    .filter(
        F.col("count").isNull()
        | (F.col("count") != F.col("registros_esperados"))
    )
    .limit(1)
    .count()
):
    raise RuntimeError("Fatos Gold diferentes das contagens contratadas.")


versions = facts.select(
    *keys,
    "gold_transform_version",
    "ingestion_run_id"
).distinct()

if (
    versions.groupBy(*keys)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Mais de uma versão ou procedência para o mesmo arquivo.")

scope_columns = [
    "escopo_id",
    "dataset",
    "source_file_sha256",
    "gold_transform_version",
    "geo_transform_version",
    "ingestion_run_id"
]

expanded_sources = (
    versions
    .withColumn("escopo_id", F.lit(SCOPE_ID))
    .withColumn("geo_transform_version", F.lit(None).cast("string"))
    .select(*scope_columns)
)

scope_table = spark.table(SCOPE_TABLE)

catalog_source = (
    scope_table
    .filter(
        (F.col("escopo_id") == PILOT_SCOPE_ID)
        & (F.col("dataset") == "situacao_pocos")
    )
    .select(*scope_columns)
    .withColumn("escopo_id", F.lit(SCOPE_ID))
)

if catalog_source.count() != 1:
    raise RuntimeError("Esperada uma única versão cadastral no piloto.")

candidates = expanded_sources.unionByName(catalog_source)

if (
    candidates.groupBy("dataset", "source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Arquivo repetido na definição do escopo.")

existing = (
    scope_table
    .filter(F.col("escopo_id") == SCOPE_ID)
    .select(*scope_columns)
)

if existing.limit(1).count():
    if (
        existing.exceptAll(candidates).limit(1).count()
        or candidates.exceptAll(existing).limit(1).count()
    ):
        raise RuntimeError(
            "Este escopo já existe com outra composição. "
            "Uma nova composição exige outro escopo_id."
        )

    print("Escopo existente com composição idêntica: REUSED")

else:
    (
        candidates
        .withColumn("registrado_em", F.current_timestamp())
        .write.format("delta")
        .mode("append")
        .saveAsTable(SCOPE_TABLE)
    )

    print("Escopo registrado: CREATED")

published = (
    spark.table(SCOPE_TABLE)
    .filter(F.col("escopo_id") == SCOPE_ID)
)

print(f"Escopo: {SCOPE_ID}")

display(
    published.groupBy(
        "dataset",
        "gold_transform_version",
        "geo_transform_version"
    )
    .agg(
        F.count("*").alias("arquivos"),
        F.countDistinct("source_file_sha256").alias("conteudos_distintos")
    )
    .orderBy("dataset")
)

Escopo registrado: CREATED
Escopo: historico_anp_v2


dataset,gold_transform_version,geo_transform_version,arquivos,conteudos_distintos
intervencoes,v2,null,2,2
producao_mar,v2,null,3,3
producao_terra,v2,null,10,10
situacao_pocos,v1,v1,1,1


In [0]:
import uuid

from pyspark.sql import functions as F

SCOPE_ID = "historico_anp_v2"
SCOPE_TABLE = "workspace.oil_gas_ops.analytical_scope_sources"
CHECK_TABLE = "workspace.oil_gas_ops.published_scope_checks"

CHECK_RUN_ID = str(uuid.uuid4())

scope = (
    spark.table(SCOPE_TABLE)
    .filter(F.col("escopo_id") == SCOPE_ID)
)

if scope.count() != 16:
    raise RuntimeError("O escopo precisa conter os 16 arquivos registrados.")

if (
    scope.groupBy("dataset", "source_file_sha256")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Há arquivos repetidos no escopo.")

definitions = [
    {
        "view": "workspace.oil_gas_gold.vw_producao_historico_v2",
        "table": "workspace.oil_gas_gold.fato_producao_injecao_mensal_v2",
        "dataset_condition": """
            s.dataset IN ('producao_mar', 'producao_terra')
            AND f._source_dataset = s.dataset
        """,
        "geo_condition": "",
        "expected": 315356
    },
    {
        "view": "workspace.oil_gas_gold.vw_intervencoes_historico_v2",
        "table": "workspace.oil_gas_gold.fato_intervencoes_v2",
        "dataset_condition": """
            s.dataset = 'intervencoes'
            AND f._source_dataset = s.dataset
        """,
        "geo_condition": "",
        "expected": 17817
    },
    {
        "view": "workspace.oil_gas_gold.vw_cadastro_historico_v2",
        "table": "workspace.oil_gas_gold.fato_snapshot_cadastro",
        "dataset_condition": "s.dataset = 'situacao_pocos'",
        "geo_condition": """
            AND f._geo_transform_version = s.geo_transform_version
        """,
        "expected": 27316
    }
]

checks = []

for item in definitions:
    spark.sql(f"""
        CREATE OR REPLACE VIEW {item["view"]} AS
        SELECT
            s.escopo_id,
            f.*
        FROM {item["table"]} f
        INNER JOIN {SCOPE_TABLE} s
            ON f._source_file_sha256 = s.source_file_sha256
            AND f._gold_transform_version = s.gold_transform_version
            {item["geo_condition"]}
        WHERE s.escopo_id = '{SCOPE_ID}'
          AND ({item["dataset_condition"]})
    """)

    published = spark.table(item["view"])

    metrics = published.agg(
        F.count("*").alias("registros"),
        F.countDistinct("fato_id").alias("fatos_distintos")
    ).first()

    passed = (
        metrics["registros"] == item["expected"]
        and metrics["fatos_distintos"] == item["expected"]
    )

    checks.append((
        CHECK_RUN_ID,
        SCOPE_ID,
        item["view"],
        item["expected"],
        metrics["registros"],
        metrics["fatos_distintos"],
        "PASS" if passed else "FAIL"
    ))

schema = """
    check_run_id STRING,
    escopo_id STRING,
    view STRING,
    registros_esperados BIGINT,
    registros_publicados BIGINT,
    fatos_distintos BIGINT,
    status STRING
"""

(
    spark.createDataFrame(checks, schema=schema)
    .withColumn("verificado_em", F.current_timestamp())
    .write.format("delta")
    .mode("append")
    .saveAsTable(CHECK_TABLE)
)

display(
    spark.table(CHECK_TABLE)
    .filter(F.col("check_run_id") == CHECK_RUN_ID)
    .select(
        "view",
        "registros_esperados",
        "registros_publicados",
        "fatos_distintos",
        "status"
    )
    .orderBy("view")
)

if any(item[-1] != "PASS" for item in checks):
    raise RuntimeError("Uma ou mais views falharam na validação.")

production_counts = {
    row["_source_dataset"]: row["count"]
    for row in (
        spark.table("workspace.oil_gas_gold.vw_producao_historico_v2")
        .groupBy("_source_dataset")
        .count()
        .collect()
    )
}

if production_counts != {
    "producao_mar": 43919,
    "producao_terra": 271437
}:
    raise RuntimeError("Distribuição dos fatos de produção divergente.")

print(f"Views publicadas e conferidas para o escopo {SCOPE_ID}: PASS")

view,registros_esperados,registros_publicados,fatos_distintos,status
workspace.oil_gas_gold.vw_cadastro_historico_v2,27316,27316,27316,PASS
workspace.oil_gas_gold.vw_intervencoes_historico_v2,17817,17817,17817,PASS
workspace.oil_gas_gold.vw_producao_historico_v2,315356,315356,315356,PASS


Views publicadas e conferidas para o escopo historico_anp_v2: PASS


In [0]:
from pyspark.sql import functions as F

SOURCE = "workspace.oil_gas_gold.vw_producao_historico_v2"
MONTHLY = "workspace.oil_gas_gold.vw_indicadores_producao_mensal_historico_v2"
COVERAGE = "workspace.oil_gas_gold.vw_cobertura_mensal_historico_v2"

measures = sorted([
    name for name in spark.table(SOURCE).columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")

measure_sql = ",\n".join(
    f"""
    SUM(f.`{name}`) AS `{name}`,
    COUNT(f.`{name}`) AS `preenchidos_{name}`
    """
    for name in measures
)


spark.sql(f"""
    CREATE OR REPLACE VIEW {MONTHLY} AS
    SELECT
        f.escopo_id,
        f._source_dataset,
        d.primeiro_dia_mes AS competencia,
        d.mes_id,
        f.ambiente_reportado AS ambiente,

        COUNT(*) AS registros_fonte,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP' THEN f.poco_id
        END) AS pocos_com_id_distintos,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP'
             AND (
                f.producao_de_oleo_m3 > 0
                OR f.producao_de_condensado_m3 > 0
                OR f.producao_de_gas_associado_mm3 > 0
                OR f.producao_de_gas_nao_associado_mm3 > 0
             )
            THEN f.poco_id
        END) AS pocos_com_producao_hidrocarbonetos_positiva,

        {measure_sql},

        CAST(
            SUM(f.producao_de_oleo_m3)
            / DAYOFMONTH(LAST_DAY(d.primeiro_dia_mes))
            AS DECIMAL(28,9)
        ) AS oleo_m3_dia_calendario

    FROM {SOURCE} f
    INNER JOIN workspace.oil_gas_gold.dim_data d
        ON f.data_id = d.data_id

    GROUP BY
        f.escopo_id,
        f._source_dataset,
        d.primeiro_dia_mes,
        d.mes_id,
        f.ambiente_reportado
""")


spark.sql(f"""
    CREATE OR REPLACE VIEW {COVERAGE} AS
    WITH limites_globais AS (
        SELECT
            escopo_id,
            MIN(competencia) AS inicio,
            MAX(competencia) AS fim
        FROM {MONTHLY}
        GROUP BY escopo_id
    ),
    meses AS (
        SELECT
            escopo_id,
            EXPLODE(
                SEQUENCE(inicio, fim, INTERVAL 1 MONTH)
            ) AS competencia
        FROM limites_globais
    ),
    limites_dataset AS (
        SELECT
            escopo_id,
            _source_dataset,
            ambiente,
            MIN(competencia) AS primeiro_mes_observado,
            MAX(competencia) AS ultimo_mes_observado
        FROM {MONTHLY}
        GROUP BY escopo_id, _source_dataset, ambiente
    )
    SELECT
        l.escopo_id,
        l._source_dataset,
        l.ambiente,
        m.competencia,
        l.primeiro_mes_observado,
        l.ultimo_mes_observado,

        CASE
            WHEN i.registros_fonte IS NOT NULL
                THEN 'COM_REGISTROS'
            WHEN m.competencia < l.primeiro_mes_observado
                THEN 'ANTES_DO_PRIMEIRO_MES_OBSERVADO'
            WHEN m.competencia > l.ultimo_mes_observado
                THEN 'APOS_O_ULTIMO_MES_OBSERVADO'
            ELSE 'LACUNA_ENTRE_MESES_OBSERVADOS'
        END AS status_cobertura,

        COALESCE(i.registros_fonte, 0) AS registros_fonte,
        i.preenchidos_producao_de_oleo_m3 AS registros_com_oleo_informado,
        i.producao_de_oleo_m3,
        i.oleo_m3_dia_calendario

    FROM limites_dataset l
    INNER JOIN meses m
        ON l.escopo_id = m.escopo_id
    LEFT JOIN {MONTHLY} i
        ON i.escopo_id = l.escopo_id
        AND i._source_dataset = l._source_dataset
        AND i.ambiente = l.ambiente
        AND i.competencia = m.competencia
""")

source_long = (
    spark.table(SOURCE)
    .select(
        "escopo_id",
        "_source_dataset",
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(name).alias("medida"),
                    F.col(name).alias("valor")
                )
                for name in measures
            ])
        ).alias("item")
    )
    .select("escopo_id", "_source_dataset", "item.*")
    .groupBy("escopo_id", "_source_dataset", "medida")
    .agg(
        F.sum("valor").alias("total_fatos"),
        F.count("valor").alias("preenchidos_fatos")
    )
)

monthly_long = (
    spark.table(MONTHLY)
    .select(
        "escopo_id",
        "_source_dataset",
        F.explode(
            F.array(*[
                F.struct(
                    F.lit(name).alias("medida"),
                    F.col(name).alias("total"),
                    F.col(f"preenchidos_{name}").alias("preenchidos")
                )
                for name in measures
            ])
        ).alias("item")
    )
    .select("escopo_id", "_source_dataset", "item.*")
    .groupBy("escopo_id", "_source_dataset", "medida")
    .agg(
        F.sum("total").alias("total_mensal"),
        F.sum("preenchidos").alias("preenchidos_mensal")
    )
)

checks = (
    source_long.join(
        monthly_long,
        ["escopo_id", "_source_dataset", "medida"],
        "full"
    )
    .withColumn(
        "status",
        F.when(
            F.col("total_fatos").eqNullSafe(F.col("total_mensal"))
            & F.col("preenchidos_fatos").eqNullSafe(
                F.col("preenchidos_mensal")
            ),
            "PASS"
        ).otherwise("FAIL")
    )
)

display(
    checks.groupBy("_source_dataset", "status")
    .count()
    .orderBy("_source_dataset", "status")
)

if checks.filter(F.col("status") != "PASS").limit(1).count():
    display(checks.filter(F.col("status") != "PASS"))
    raise RuntimeError("Agregação mensal com divergência.")

display(
    spark.table(COVERAGE)
    .groupBy("_source_dataset", "status_cobertura")
    .count()
    .orderBy("_source_dataset", "status_cobertura")
)

display(
    spark.table(COVERAGE)
    .filter(F.year("competencia") == 2026)
    .select(
        "_source_dataset",
        "competencia",
        "status_cobertura",
        "registros_fonte",
        "producao_de_oleo_m3",
        "oleo_m3_dia_calendario"
    )
    .orderBy("_source_dataset", "competencia")
)

_source_dataset,status,count
producao_mar,PASS,13
producao_terra,PASS,13


_source_dataset,status_cobertura,count
producao_mar,COM_REGISTROS,32
producao_terra,APOS_O_ULTIMO_MES_OBSERVADO,2
producao_terra,COM_REGISTROS,29
producao_terra,LACUNA_ENTRE_MESES_OBSERVADOS,1


_source_dataset,competencia,status_cobertura,registros_fonte,producao_de_oleo_m3,oleo_m3_dia_calendario
producao_mar,2026-01-01,COM_REGISTROS,1377,19063631.170000000,614955.844193548
producao_mar,2026-02-01,COM_REGISTROS,1416,17700561.390000000,632162.906785714
producao_mar,2026-03-01,COM_REGISTROS,1466,20508242.310000000,661556.203548387
producao_mar,2026-04-01,COM_REGISTROS,1464,20295101.650000000,676503.388333333
producao_mar,2026-05-01,COM_REGISTROS,1530,20781780.660000000,670380.021290323
producao_mar,2026-06-01,COM_REGISTROS,1570,20926299.420000000,697543.314000000
producao_mar,2026-07-01,COM_REGISTROS,1577,21737154.620000000,701198.536129032
producao_mar,2026-08-01,COM_REGISTROS,1600,22309773.210000000,719670.103548387
producao_terra,2026-01-01,COM_REGISTROS,7297,377316.160000000,12171.489032258
producao_terra,2026-02-01,COM_REGISTROS,7244,339172.860000000,12113.316428571


In [0]:
from pyspark.sql import functions as F

SOURCE = "workspace.oil_gas_gold.vw_producao_historico_v2"
TARGET = "workspace.oil_gas_gold.vw_producao_poco_mes_historico_v2"

measures = sorted([
    name for name in spark.table(SOURCE).columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")

measure_sql = ",\n".join(
    f"""
    SUM(f.`{name}`) AS `{name}`,
    COUNT(f.`{name}`) AS `preenchidos_{name}`
    """
    for name in measures
)

spark.sql(f"""
    CREATE OR REPLACE VIEW {TARGET} AS
    WITH agregado AS (
        SELECT
            f.escopo_id,
            f.poco_id,
            d.primeiro_dia_mes AS competencia,
            DAYOFMONTH(
                LAST_DAY(d.primeiro_dia_mes)
            ) AS dias_calendario_mes,

            COUNT(*) AS registros_fonte,

            SUM(CASE
                WHEN f.perfil_medidas IN (
                    'SOMENTE_PRODUCAO', 'PRODUCAO_E_INJECAO'
                )
                THEN 1 ELSE 0
            END) AS registros_com_medidas_producao,

            COUNT(DISTINCT f._source_dataset) AS datasets_distintos,
            SORT_ARRAY(
                COLLECT_SET(f._source_dataset)
            ) AS datasets_observados,

            SORT_ARRAY(
                COLLECT_SET(f.ambiente_reportado)
            ) AS ambientes_observados,

            COUNT(DISTINCT f.campo_id) AS contextos_campo_distintos,

            {measure_sql}

        FROM {SOURCE} f
        INNER JOIN workspace.oil_gas_gold.dim_data d
            ON f.data_id = d.data_id

        WHERE f.poco_id <> 'SEM_ID_ANP'

        GROUP BY
            f.escopo_id,
            f.poco_id,
            d.primeiro_dia_mes
    )
    SELECT
        *,
        CASE
            WHEN preenchidos_producao_de_oleo_m3 = 0
                THEN 'SEM_OLEO_INFORMADO'
            WHEN preenchidos_producao_de_oleo_m3
                 <> registros_com_medidas_producao
                THEN 'OLEO_PARCIAL_NOS_REGISTROS_DE_PRODUCAO'
            ELSE 'OLEO_INFORMADO_NOS_REGISTROS_DE_PRODUCAO'
        END AS status_informacao_oleo,

        CAST(
            producao_de_oleo_m3 / dias_calendario_mes
            AS DECIMAL(28,9)
        ) AS oleo_m3_dia_calendario

    FROM agregado
""")

monthly = spark.table(TARGET)

if (
    monthly.groupBy("escopo_id", "poco_id", "competencia")
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Mais de uma linha por poço e competência.")

source_metrics = (
    spark.table(SOURCE)
    .filter(F.col("poco_id") != "SEM_ID_ANP")
    .agg(
        F.count("*").alias("registros"),
        F.sum("producao_de_oleo_m3").alias("oleo"),
        F.count("producao_de_oleo_m3").alias("oleo_preenchidos")
    )
    .first()
)

monthly_metrics = monthly.agg(
    F.sum("registros_fonte").alias("registros"),
    F.sum("producao_de_oleo_m3").alias("oleo"),
    F.sum("preenchidos_producao_de_oleo_m3").alias("oleo_preenchidos")
).first()

if source_metrics.asDict() != monthly_metrics.asDict():
    raise RuntimeError("A agregação por poço alterou as métricas conferidas.")

print(f"View: {TARGET}")
print("Granularidade, contagem de origem e óleo reconciliados: PASS")

display(
    monthly.groupBy("status_informacao_oleo")
    .agg(
        F.count("*").alias("registros_poco_mes"),
        F.countDistinct("poco_id").alias("pocos_distintos")
    )
    .orderBy("status_informacao_oleo")
)

display(
    monthly
    .filter(F.col("datasets_distintos") > 1)
    .select(
        "poco_id",
        "competencia",
        "datasets_distintos",
        "datasets_observados",
        "ambientes_observados",
        "registros_fonte"
    )
    .orderBy("poco_id", "competencia")
    .limit(20)
)

View: workspace.oil_gas_gold.vw_producao_poco_mes_historico_v2
Granularidade, contagem de origem e óleo reconciliados: PASS


status_informacao_oleo,registros_poco_mes,pocos_distintos
OLEO_INFORMADO_NOS_REGISTROS_DE_PRODUCAO,225676,9606
SEM_OLEO_INFORMADO,66145,3073


poco_id,competencia,datasets_distintos,datasets_observados,ambientes_observados,registros_fonte


In [0]:
from pyspark.sql import functions as F

EVENTS = "workspace.oil_gas_gold.vw_intervencoes_historico_v2"
PRODUCTION = "workspace.oil_gas_gold.vw_producao_poco_mes_historico_v2"

WINDOWS = "workspace.oil_gas_gold.vw_janelas_intervencoes_historico_v2"
COVERAGE = "workspace.oil_gas_gold.vw_cobertura_intervencoes_historico_v2"

spark.sql(f"""
    CREATE OR REPLACE VIEW {WINDOWS} AS
    WITH eventos AS (
        SELECT
            i.escopo_id,
            i.fato_id,
            i.poco_id,
            i.ambiente,
            i.objetivo,
            di.data AS data_inicio,
            dt.data AS data_termino
        FROM {EVENTS} i
        INNER JOIN workspace.oil_gas_gold.dim_data di
            ON i.data_inicio_id = di.data_id
        INNER JOIN workspace.oil_gas_gold.dim_data dt
            ON i.data_termino_id = dt.data_id
    ),
    meses AS (
        SELECT
            e.*,
            CASE
                WHEN deslocamento < 0 THEN 'ANTES'
                ELSE 'DEPOIS'
            END AS fase,
            ADD_MONTHS(
                TRUNC(
                    CASE
                        WHEN deslocamento < 0 THEN data_inicio
                        ELSE data_termino
                    END,
                    'MONTH'
                ),
                deslocamento
            ) AS competencia
        FROM eventos e
        LATERAL VIEW EXPLODE(
            ARRAY(-3, -2, -1, 1, 2, 3)
        ) offsets AS deslocamento
    )
    SELECT
        m.*,
        p.registros_fonte,
        p.registros_com_medidas_producao,
        p.preenchidos_producao_de_oleo_m3 AS registros_com_oleo,
        p.producao_de_oleo_m3 AS oleo_reportado_m3,
        p.datasets_distintos,
        p.status_informacao_oleo,

        COALESCE(
            p.status_informacao_oleo =
                'OLEO_INFORMADO_NOS_REGISTROS_DE_PRODUCAO'
            AND p.datasets_distintos = 1,
            FALSE
        ) AS mes_elegivel_para_comparacao

    FROM meses m
    LEFT JOIN {PRODUCTION} p
        ON m.escopo_id = p.escopo_id
        AND m.poco_id = p.poco_id
        AND m.competencia = p.competencia
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {COVERAGE} AS
    WITH cobertura AS (
        SELECT
            escopo_id,
            fato_id,
            poco_id,
            ambiente,
            objetivo,
            data_inicio,
            data_termino,

            SUM(CASE
                WHEN fase = 'ANTES' AND registros_fonte > 0
                THEN 1 ELSE 0
            END) AS meses_com_registro_antes,

            SUM(CASE
                WHEN fase = 'DEPOIS' AND registros_fonte > 0
                THEN 1 ELSE 0
            END) AS meses_com_registro_depois,

            SUM(CASE
                WHEN fase = 'ANTES' AND registros_com_oleo > 0
                THEN 1 ELSE 0
            END) AS meses_com_oleo_informado_antes,

            SUM(CASE
                WHEN fase = 'DEPOIS' AND registros_com_oleo > 0
                THEN 1 ELSE 0
            END) AS meses_com_oleo_informado_depois,

            SUM(CASE
                WHEN fase = 'ANTES' AND mes_elegivel_para_comparacao
                THEN 1 ELSE 0
            END) AS meses_elegiveis_antes,

            SUM(CASE
                WHEN fase = 'DEPOIS' AND mes_elegivel_para_comparacao
                THEN 1 ELSE 0
            END) AS meses_elegiveis_depois,

            SUM(CASE
                WHEN fase = 'ANTES'
                 AND registros_com_oleo > 0
                 AND oleo_reportado_m3 = 0
                THEN 1 ELSE 0
            END) AS meses_com_oleo_zero_reportado_antes,

            SUM(CASE
                WHEN fase = 'DEPOIS'
                 AND registros_com_oleo > 0
                 AND oleo_reportado_m3 = 0
                THEN 1 ELSE 0
            END) AS meses_com_oleo_zero_reportado_depois,

            SUM(CASE
                WHEN datasets_distintos > 1 THEN 1 ELSE 0
            END) AS meses_com_multiplos_datasets

        FROM {WINDOWS}
        GROUP BY
            escopo_id, fato_id, poco_id, ambiente,
            objetivo, data_inicio, data_termino
    )
    SELECT
        *,
        CASE
            WHEN poco_id = 'SEM_ID_ANP'
                THEN 'SEM_IDENTIFICADOR'
            WHEN meses_elegiveis_antes = 3
             AND meses_elegiveis_depois = 3
                THEN 'OLEO_INFORMADO_NAS_DUAS_JANELAS'
            WHEN meses_com_oleo_informado_antes = 0
             AND meses_com_oleo_informado_depois = 0
                THEN 'SEM_OLEO_INFORMADO_NAS_JANELAS'
            WHEN meses_com_oleo_informado_antes > 0
             AND meses_com_oleo_informado_depois = 0
                THEN 'OLEO_INFORMADO_SOMENTE_ANTES'
            WHEN meses_com_oleo_informado_antes = 0
             AND meses_com_oleo_informado_depois > 0
                THEN 'OLEO_INFORMADO_SOMENTE_DEPOIS'
            ELSE 'DUAS_JANELAS_PARCIAIS_OU_COM_RESTRICOES'
        END AS status_cobertura
    FROM cobertura
""")

events = spark.table(EVENTS)
windows = spark.table(WINDOWS)
coverage = spark.table(COVERAGE)

event_count = events.count()

if windows.count() != event_count * 6:
    raise RuntimeError("Quantidade de meses nas janelas divergente.")

invalid_windows = (
    windows.groupBy("escopo_id", "fato_id")
    .agg(
        F.count("*").alias("linhas"),
        F.countDistinct(
            F.struct("fase", "competencia")
        ).alias("meses_distintos"),
        F.sum(
            F.when(F.col("fase") == "ANTES", 1).otherwise(0)
        ).alias("meses_antes")
    )
    .filter(
        (F.col("linhas") != 6)
        | (F.col("meses_distintos") != 6)
        | (F.col("meses_antes") != 3)
    )
)

if invalid_windows.limit(1).count():
    raise RuntimeError("Evento sem exatamente três meses em cada janela.")

if (
    coverage.count() != event_count
    or coverage.select("escopo_id", "fato_id").distinct().count()
       != event_count
):
    raise RuntimeError("A cobertura não preservou uma linha por evento.")

print(f"Intervenções avaliadas: {event_count:,}")
print("Granularidade das janelas e cobertura: PASS")

display(
    coverage.groupBy("ambiente", "status_cobertura")
    .count()
    .orderBy("ambiente", F.desc("count"))
)

display(
    coverage
    .filter(
        F.col("status_cobertura")
        == "OLEO_INFORMADO_NAS_DUAS_JANELAS"
    )
    .groupBy("ambiente", "objetivo")
    .count()
    .orderBy("ambiente", F.desc("count"))
)

Intervenções avaliadas: 17,817
Granularidade das janelas e cobertura: PASS


ambiente,status_cobertura,count
MAR,SEM_OLEO_INFORMADO_NAS_JANELAS,1499
MAR,OLEO_INFORMADO_SOMENTE_DEPOIS,114
MAR,OLEO_INFORMADO_NAS_DUAS_JANELAS,112
MAR,DUAS_JANELAS_PARCIAIS_OU_COM_RESTRICOES,41
MAR,OLEO_INFORMADO_SOMENTE_ANTES,23
TERRA,SEM_OLEO_INFORMADO_NAS_JANELAS,6214
TERRA,OLEO_INFORMADO_NAS_DUAS_JANELAS,3478
TERRA,OLEO_INFORMADO_SOMENTE_DEPOIS,3133
TERRA,DUAS_JANELAS_PARCIAIS_OU_COM_RESTRICOES,2366
TERRA,OLEO_INFORMADO_SOMENTE_ANTES,836


ambiente,objetivo,count
MAR,RESTAURAÇÃO,87
MAR,COMPLETAÇÃO,13
MAR,ABANDONO,6
MAR,TLD,2
MAR,PERFURAÇÃO,2
MAR,AVALIAÇÃO,2
TERRA,RESTAURAÇÃO,2574
TERRA,COMPLETAÇÃO,866
TERRA,AVALIAÇÃO,32
TERRA,ABANDONO,4


In [0]:
from pyspark.sql import functions as F

COVERAGE = "workspace.oil_gas_gold.vw_cobertura_intervencoes_historico_v2"
EVENTS = "workspace.oil_gas_gold.vw_intervencoes_historico_v2"
TARGET = "workspace.oil_gas_gold.vw_contexto_intervencoes_historico_v2"

spark.sql(f"""
    CREATE OR REPLACE VIEW {TARGET} AS
    WITH elegiveis AS (
        SELECT
            c.*,

            ADD_MONTHS(
                TRUNC(data_inicio, 'MONTH'), -3
            ) AS janela_antes_inicio,

            LAST_DAY(
                ADD_MONTHS(TRUNC(data_inicio, 'MONTH'), -1)
            ) AS janela_antes_fim,

            ADD_MONTHS(
                TRUNC(data_termino, 'MONTH'), 1
            ) AS janela_depois_inicio,

            LAST_DAY(
                ADD_MONTHS(TRUNC(data_termino, 'MONTH'), 3)
            ) AS janela_depois_fim

        FROM {COVERAGE} c
        WHERE status_cobertura = 'OLEO_INFORMADO_NAS_DUAS_JANELAS'
    ),
    todos_eventos AS (
        SELECT
            i.escopo_id,
            i.fato_id,
            i.poco_id,
            di.data AS data_inicio,
            dt.data AS data_termino
        FROM {EVENTS} i
        INNER JOIN workspace.oil_gas_gold.dim_data di
            ON i.data_inicio_id = di.data_id
        INNER JOIN workspace.oil_gas_gold.dim_data dt
            ON i.data_termino_id = dt.data_id
        WHERE i.poco_id <> 'SEM_ID_ANP'
    ),
    contexto AS (
        SELECT
            e.escopo_id,
            e.fato_id,

            COUNT(DISTINCT o.fato_id) AS outros_eventos_observados,

            COUNT(DISTINCT CASE
                WHEN o.data_inicio < e.data_inicio
                 AND o.data_termino >= e.janela_antes_inicio
                THEN o.fato_id
            END) AS outros_eventos_antes,

            COUNT(DISTINCT CASE
                WHEN o.data_inicio <= e.data_termino
                 AND o.data_termino >= e.data_inicio
                THEN o.fato_id
            END) AS outros_eventos_durante,

            COUNT(DISTINCT CASE
                WHEN o.data_termino > e.data_termino
                 AND o.data_inicio <= e.janela_depois_fim
                THEN o.fato_id
            END) AS outros_eventos_depois,

            SORT_ARRAY(
                COLLECT_SET(o.fato_id)
            ) AS outros_eventos_ids

        FROM elegiveis e
        LEFT JOIN todos_eventos o
            ON o.escopo_id = e.escopo_id
            AND o.poco_id = e.poco_id
            AND o.fato_id <> e.fato_id
            AND o.data_inicio <= e.janela_depois_fim
            AND o.data_termino >= e.janela_antes_inicio

        GROUP BY e.escopo_id, e.fato_id
    )
    SELECT
        e.*,
        c.outros_eventos_observados,
        c.outros_eventos_antes,
        c.outros_eventos_durante,
        c.outros_eventos_depois,
        c.outros_eventos_ids,

        CASE
            WHEN c.outros_eventos_observados = 0
                THEN 'SEM_OUTRA_INTERVENCAO_OBSERVADA'
            ELSE 'COM_OUTRA_INTERVENCAO_OBSERVADA'
        END AS status_contexto,

        CASE
            WHEN e.objetivo <> 'RESTAURAÇÃO'
                THEN 'OUTRO_OBJETIVO'
            WHEN c.outros_eventos_observados > 0
                THEN 'RESTAURACAO_COM_OUTROS_EVENTOS'
            ELSE 'RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO'
        END AS grupo_analise

    FROM elegiveis e
    INNER JOIN contexto c
        ON e.escopo_id = c.escopo_id
        AND e.fato_id = c.fato_id
""")

expected_count = (
    spark.table(COVERAGE)
    .filter(
        F.col("status_cobertura")
        == "OLEO_INFORMADO_NAS_DUAS_JANELAS"
    )
    .count()
)

published = spark.table(TARGET)

if (
    published.count() != expected_count
    or published.select("escopo_id", "fato_id").distinct().count()
       != expected_count
):
    raise RuntimeError("O contexto não preservou uma linha por evento.")

if published.filter(
    F.size("outros_eventos_ids") != F.col("outros_eventos_observados")
).limit(1).count():
    raise RuntimeError("Lista de eventos diferente da contagem registrada.")

if published.filter(
    F.array_contains(F.col("outros_eventos_ids"), F.col("fato_id"))
).limit(1).count():
    raise RuntimeError("Uma intervenção foi contada como outro evento de si mesma.")

print(f"Intervenções contextualizadas: {expected_count:,}")
print("Granularidade e identificação dos outros eventos: PASS")

display(
    published.groupBy("ambiente", "objetivo", "grupo_analise")
    .count()
    .orderBy("ambiente", "objetivo", "grupo_analise")
)

display(
    published
    .filter(
        F.col("grupo_analise") == "RESTAURACAO_COM_OUTROS_EVENTOS"
    )
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco")
        .select("poco_id", "nome_poco_anp"),
        "poco_id",
        "inner"
    )
    .select(
        "nome_poco_anp",
        "fato_id",
        "data_inicio",
        "data_termino",
        "janela_antes_inicio",
        "janela_depois_fim",
        "outros_eventos_observados",
        "outros_eventos_antes",
        "outros_eventos_durante",
        "outros_eventos_depois"
    )
    .orderBy(F.desc("outros_eventos_observados"), "nome_poco_anp")
    .limit(10)
)

Intervenções contextualizadas: 3,590
Granularidade e identificação dos outros eventos: PASS


ambiente,objetivo,grupo_analise,count
MAR,ABANDONO,OUTRO_OBJETIVO,6
MAR,AVALIAÇÃO,OUTRO_OBJETIVO,2
MAR,COMPLETAÇÃO,OUTRO_OBJETIVO,13
MAR,PERFURAÇÃO,OUTRO_OBJETIVO,2
MAR,RESTAURAÇÃO,RESTAURACAO_COM_OUTROS_EVENTOS,27
MAR,RESTAURAÇÃO,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,60
MAR,TLD,OUTRO_OBJETIVO,2
TERRA,ABANDONO,OUTRO_OBJETIVO,4
TERRA,AVALIAÇÃO,OUTRO_OBJETIVO,32
TERRA,COMPLETAÇÃO,OUTRO_OBJETIVO,866


nome_poco_anp,fato_id,data_inicio,data_termino,janela_antes_inicio,janela_depois_fim,outros_eventos_observados,outros_eventos_antes,outros_eventos_durante,outros_eventos_depois
7-BR-16-RN,c6fdc8d0a235e0a9224f07ab634681a182d79b5bbc3c6e41b16ddbf4b6dda820,2024-05-30,2024-06-01,2024-02-01,2024-09-30,8,4,0,4
7-LOR-73-RN,99ac48f1c17b07e0cc4bb86062832c4f92cf88a52cafe3691666e3579a78b867,2025-01-30,2025-01-30,2024-10-01,2025-04-30,8,1,0,7
7-BR-78D-RN,53f1740f39d81a6f9eed933f3738f77d1dfdc30ba1432f736887c5400c99075d,2024-08-20,2024-08-20,2024-05-01,2024-11-30,7,5,0,2
7-BR-78D-RN,dd72d27d07da82eceb08474288e2caa784050322524bb0965447cd6a9c20f819,2024-05-30,2024-06-02,2024-02-01,2024-09-30,7,3,0,4
7-LOR-73-RN,e77a84dd9d423db244b44d952c569bc99d3d8c542dcae651cecee29b2a51e6ac,2025-04-07,2025-04-11,2025-01-01,2025-07-31,7,6,0,1
7-LOR-73-RN,870e3dbc20f927687566ea760256ef06b75c44948e2f5a34d1297e2cc4425635,2025-02-14,2025-02-14,2024-11-01,2025-05-31,7,1,0,6
7-LOR-73-RN,444f0ebc6e984d2481d98b2f0f361ef9e32c2541c9eebf1db27ea4e9f3b665c0,2025-02-28,2025-03-01,2024-11-01,2025-06-30,7,2,0,5
7-LOR-73-RN,d5e58c8db3be4d6d946c4fc5beb2a8b7cfac8acab03d8296bd4b7312f6577ce0,2025-04-05,2025-04-05,2025-01-01,2025-07-31,7,5,0,2
7-LOR-73-RN,4b1f5a369fb242cd83f7471af8624ff9ca9489b8369c896c5897c348c762ed26,2025-04-23,2025-04-26,2025-01-01,2025-07-31,7,7,0,0
7-LOR-73-RN,96215703e12be36d44d7a2611a987debedbd45d978862616e7ced70f5127fa12,2025-03-10,2025-03-14,2024-12-01,2025-06-30,7,3,0,4


In [0]:
from pyspark.sql import functions as F

CONTEXT = "workspace.oil_gas_gold.vw_contexto_intervencoes_historico_v2"
WINDOWS = "workspace.oil_gas_gold.vw_janelas_intervencoes_historico_v2"
TARGET = "workspace.oil_gas_gold.vw_comparacao_oleo_intervencoes_historico_v2"

context = spark.table(CONTEXT)

selected_windows = (
    spark.table(WINDOWS)
    .join(
        context.select("escopo_id", "fato_id"),
        ["escopo_id", "fato_id"],
        "inner"
    )
)

if selected_windows.count() != context.count() * 6:
    raise RuntimeError("Quantidade de meses divergente nas janelas selecionadas.")

if selected_windows.filter(
    ~F.col("mes_elegivel_para_comparacao")
    | F.col("oleo_reportado_m3").isNull()
    | (F.col("oleo_reportado_m3") < 0)
).limit(1).count():
    raise RuntimeError("Há meses sem condições para a comparação.")

spark.sql(f"""
    CREATE OR REPLACE VIEW {TARGET} AS
    WITH totais AS (
        SELECT
            w.escopo_id,
            w.fato_id,

            SUM(CASE
                WHEN w.fase = 'ANTES'
                THEN w.oleo_reportado_m3 ELSE 0
            END) AS oleo_antes_m3,

            SUM(CASE
                WHEN w.fase = 'DEPOIS'
                THEN w.oleo_reportado_m3 ELSE 0
            END) AS oleo_depois_m3,

            SUM(CASE
                WHEN w.fase = 'ANTES'
                THEN DAYOFMONTH(LAST_DAY(w.competencia)) ELSE 0
            END) AS dias_antes,

            SUM(CASE
                WHEN w.fase = 'DEPOIS'
                THEN DAYOFMONTH(LAST_DAY(w.competencia)) ELSE 0
            END) AS dias_depois,

            SUM(CASE
                WHEN w.fase = 'ANTES' THEN 1 ELSE 0
            END) AS meses_antes,

            SUM(CASE
                WHEN w.fase = 'DEPOIS' THEN 1 ELSE 0
            END) AS meses_depois

        FROM {WINDOWS} w
        INNER JOIN {CONTEXT} c
            ON w.escopo_id = c.escopo_id
            AND w.fato_id = c.fato_id

        GROUP BY w.escopo_id, w.fato_id
    ),
    taxas AS (
        SELECT
            c.*,
            t.oleo_antes_m3,
            t.oleo_depois_m3,
            t.dias_antes,
            t.dias_depois,
            t.meses_antes,
            t.meses_depois,

            CAST(
                t.oleo_antes_m3 / t.dias_antes
                AS DECIMAL(28,9)
            ) AS taxa_antes_m3_dia_calendario,

            CAST(
                t.oleo_depois_m3 / t.dias_depois
                AS DECIMAL(28,9)
            ) AS taxa_depois_m3_dia_calendario

        FROM {CONTEXT} c
        INNER JOIN totais t
            ON c.escopo_id = t.escopo_id
            AND c.fato_id = t.fato_id
    )
    SELECT
        *,

        CAST(
            taxa_depois_m3_dia_calendario
            - taxa_antes_m3_dia_calendario
            AS DECIMAL(28,9)
        ) AS diferenca_taxa_m3_dia_calendario,

        CASE
            WHEN taxa_antes_m3_dia_calendario > 0
            THEN CAST(
                (
                    taxa_depois_m3_dia_calendario
                    / taxa_antes_m3_dia_calendario - 1
                ) * 100
                AS DECIMAL(28,6)
            )
        END AS variacao_percentual,

        CASE
            WHEN oleo_antes_m3 = 0 AND oleo_depois_m3 = 0
                THEN 'ZERO_NAS_DUAS_JANELAS'
            WHEN oleo_antes_m3 = 0 AND oleo_depois_m3 > 0
                THEN 'DE_ZERO_PARA_POSITIVO'
            WHEN oleo_antes_m3 > 0 AND oleo_depois_m3 = 0
                THEN 'DE_POSITIVO_PARA_ZERO'
            WHEN taxa_depois_m3_dia_calendario
                 > taxa_antes_m3_dia_calendario
                THEN 'TAXA_POSTERIOR_MAIOR'
            WHEN taxa_depois_m3_dia_calendario
                 < taxa_antes_m3_dia_calendario
                THEN 'TAXA_POSTERIOR_MENOR'
            ELSE 'TAXAS_IGUAIS_NA_PRECISAO_ADOTADA'
        END AS comportamento_observado

    FROM taxas
""")

published = spark.table(TARGET)

if (
    published.count() != context.count()
    or published.select("escopo_id", "fato_id").distinct().count()
       != context.count()
):
    raise RuntimeError("A comparação não preservou uma linha por evento.")

invalid = (
    (F.col("meses_antes") != 3)
    | (F.col("meses_depois") != 3)
    | (
        F.col("dias_antes")
        != F.datediff("janela_antes_fim", "janela_antes_inicio") + 1
    )
    | (
        F.col("dias_depois")
        != F.datediff("janela_depois_fim", "janela_depois_inicio") + 1
    )
    | F.col("taxa_antes_m3_dia_calendario").isNull()
    | F.col("taxa_depois_m3_dia_calendario").isNull()
    | (
        (F.col("oleo_antes_m3") == 0)
        & F.col("variacao_percentual").isNotNull()
    )
)

if published.filter(invalid).limit(1).count():
    raise RuntimeError("Inconsistência nos meses, dias, taxas ou percentuais.")

print(f"Comparações publicadas: {published.count():,}")
print("Granularidade, janelas e regras de cálculo: PASS")

display(
    published.groupBy(
        "ambiente", "grupo_analise", "comportamento_observado"
    )
    .count()
    .orderBy("ambiente", "grupo_analise", "comportamento_observado")
)

display(
    published
    .filter(
        F.col("grupo_analise")
        == "RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO"
    )
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco")
        .select("poco_id", "nome_poco_anp"),
        "poco_id",
        "inner"
    )
    .select(
        "nome_poco_anp",
        "fato_id",
        "ambiente",
        "data_inicio",
        "data_termino",
        "oleo_antes_m3",
        "oleo_depois_m3",
        "dias_antes",
        "dias_depois",
        "taxa_antes_m3_dia_calendario",
        "taxa_depois_m3_dia_calendario",
        "diferenca_taxa_m3_dia_calendario",
        "variacao_percentual",
        "comportamento_observado"
    )
    .orderBy(F.desc(F.abs(F.col("diferenca_taxa_m3_dia_calendario"))))
    .limit(10)
)

Comparações publicadas: 3,590
Granularidade, janelas e regras de cálculo: PASS


ambiente,grupo_analise,comportamento_observado,count
MAR,OUTRO_OBJETIVO,DE_ZERO_PARA_POSITIVO,9
MAR,OUTRO_OBJETIVO,TAXA_POSTERIOR_MAIOR,10
MAR,OUTRO_OBJETIVO,TAXA_POSTERIOR_MENOR,4
MAR,OUTRO_OBJETIVO,ZERO_NAS_DUAS_JANELAS,2
MAR,RESTAURACAO_COM_OUTROS_EVENTOS,DE_POSITIVO_PARA_ZERO,1
MAR,RESTAURACAO_COM_OUTROS_EVENTOS,DE_ZERO_PARA_POSITIVO,8
MAR,RESTAURACAO_COM_OUTROS_EVENTOS,TAXA_POSTERIOR_MAIOR,6
MAR,RESTAURACAO_COM_OUTROS_EVENTOS,TAXA_POSTERIOR_MENOR,10
MAR,RESTAURACAO_COM_OUTROS_EVENTOS,ZERO_NAS_DUAS_JANELAS,2
MAR,RESTAURACAO_SEM_OUTRO_EVENTO_OBSERVADO,DE_POSITIVO_PARA_ZERO,1


nome_poco_anp,fato_id,ambiente,data_inicio,data_termino,oleo_antes_m3,oleo_depois_m3,dias_antes,dias_depois,taxa_antes_m3_dia_calendario,taxa_depois_m3_dia_calendario,diferenca_taxa_m3_dia_calendario,variacao_percentual,comportamento_observado
9-BUZ-7-RJS,882bd96a7c302d7f861ff096df781f73b33815fc92c85e093425305cc134befb,MAR,2024-07-04,2024-07-26,138666.382000000,254323.134000000,91,92,1523.806395604,2764.381891304,1240.575495700,81.412934,TAXA_POSTERIOR_MAIOR
7-PRG-74H-RJS,ac2a0231aef33c343c411688255f64ad8919c5d00aa98ac7dd876a768e5f1d28,MAR,2024-07-29,2024-08-16,0E-9,43622.135000000,91,91,0E-9,479.364120879,479.364120879,null,DE_ZERO_PARA_POSITIVO
7-PRG-81H-RJS,0dfe1948a5de6a7657ef2a4b36e8e00bc34e7ba61ac8bfe734bd5fed475bba81,MAR,2025-05-11,2025-05-20,0E-9,34123.160000000,89,92,0E-9,370.903913043,370.903913043,null,DE_ZERO_PARA_POSITIVO
7-VM-46D-RJS,b22ebdaf63eba26ab696de45979de569894337b3048f9cfaa066193809675f19,MAR,2024-10-14,2024-11-18,0E-9,31761.986000000,92,90,0E-9,352.910955556,352.910955556,null,DE_ZERO_PARA_POSITIVO
7-PRG-72HP-RJS,620cd3234de090ae294b031e7fce238de6bcf3060f6b4dc3ad0549e360052bbc,MAR,2024-11-25,2024-12-05,0E-9,29852.755000000,92,90,0E-9,331.697277778,331.697277778,null,DE_ZERO_PARA_POSITIVO
7-PRG-62H-RJS,2b603c6229a68433e338d294942f70d552ac93f08e0c71788a5a10aad40a9f81,MAR,2024-11-18,2024-11-25,40732.635000000,66245.398000000,92,90,442.746032609,736.059977778,293.313945169,66.248803,TAXA_POSTERIOR_MAIOR
7-PRG-63H-RJS,c177c42d400db2ea0a1a1d07a0388ce88ca3abe30b5c2991d8e87fe48fd7d3e3,MAR,2024-12-06,2024-12-19,0E-9,21797.055000000,91,90,0E-9,242.189500000,242.189500000,null,DE_ZERO_PARA_POSITIVO
7-TBMT-4HP-RJS,8cded65b726fd2fe0f66642d1a02855b3eb321d1e8826979febb018cde1270e8,MAR,2025-06-10,2025-06-23,0E-9,20734.665000000,92,92,0E-9,225.376793478,225.376793478,null,DE_ZERO_PARA_POSITIVO
7-PRG-76HB-RJS,214802753b09897d50ace180e19034830852ab542c1cfa34a99e5aea276b728b,MAR,2025-08-30,2025-09-16,150029.178000000,170395.608000000,92,92,1630.751934783,1852.126173913,221.374239130,13.574979,TAXA_POSTERIOR_MAIOR
7-PRG-29H-RJS,ff6d978a9a715b1ca00a392ebcfed7039584a95137bf8e8027144a4b2c73bf8f,MAR,2025-01-10,2025-01-23,0E-9,18068.423000000,92,89,0E-9,203.015988764,203.015988764,null,DE_ZERO_PARA_POSITIVO


In [0]:
from pyspark.sql import functions as F

SOURCE = "workspace.oil_gas_gold.vw_producao_historico_v2"
TARGET = "workspace.oil_gas_gold.vw_producao_campo_mes_historico_v2"

measures = sorted([
    name for name in spark.table(SOURCE).columns
    if name.startswith(("producao_de_", "injecao_de_"))
])

if len(measures) != 13:
    raise RuntimeError("Esperadas exatamente 13 medidas.")

measure_sql = ",\n".join(
    f"""
    SUM(f.`{name}`) AS `{name}`,
    COUNT(f.`{name}`) AS `preenchidos_{name}`
    """
    for name in measures
)

spark.sql(f"""
    CREATE OR REPLACE VIEW {TARGET} AS
    SELECT
        f.escopo_id,
        f._source_dataset,
        f.ambiente_reportado AS ambiente,
        f.campo_id,
        c.nome_campo_reportado,
        c.bacia_reportada,
        d.primeiro_dia_mes AS competencia,
        d.mes_id,

        COUNT(*) AS registros_fonte,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP' THEN f.poco_id
        END) AS pocos_com_id_distintos,

        COUNT(DISTINCT CASE
            WHEN f.poco_id <> 'SEM_ID_ANP'
             AND (
                f.producao_de_oleo_m3 > 0
                OR f.producao_de_condensado_m3 > 0
                OR f.producao_de_gas_associado_mm3 > 0
                OR f.producao_de_gas_nao_associado_mm3 > 0
             )
            THEN f.poco_id
        END) AS pocos_com_producao_hidrocarbonetos_positiva,

        {measure_sql},

        CAST(
            SUM(f.producao_de_oleo_m3)
            / DAYOFMONTH(LAST_DAY(d.primeiro_dia_mes))
            AS DECIMAL(28,9)
        ) AS oleo_m3_dia_calendario

    FROM {SOURCE} f
    INNER JOIN workspace.oil_gas_gold.dim_data d
        ON f.data_id = d.data_id
    INNER JOIN workspace.oil_gas_gold.dim_campo c
        ON f.campo_id = c.campo_id

    GROUP BY
        f.escopo_id,
        f._source_dataset,
        f.ambiente_reportado,
        f.campo_id,
        c.nome_campo_reportado,
        c.bacia_reportada,
        d.primeiro_dia_mes,
        d.mes_id
""")

published = spark.table(TARGET)

grain = [
    "escopo_id",
    "_source_dataset",
    "ambiente",
    "campo_id",
    "competencia"
]

if (
    published.groupBy(*grain)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("Mais de uma linha por campo, mês e contexto de origem.")

source_metrics = (
    spark.table(SOURCE)
    .groupBy("escopo_id", "_source_dataset")
    .agg(
        F.count("*").alias("registros"),
        *[
            F.sum(name).alias(f"total_{name}")
            for name in measures
        ],
        *[
            F.count(name).alias(f"preenchidos_{name}")
            for name in measures
        ]
    )
)

field_metrics = (
    published
    .groupBy("escopo_id", "_source_dataset")
    .agg(
        F.sum("registros_fonte").alias("registros"),
        *[
            F.sum(name).alias(f"total_{name}")
            for name in measures
        ],
        *[
            F.sum(f"preenchidos_{name}").alias(f"preenchidos_{name}")
            for name in measures
        ]
    )
)

columns = source_metrics.columns

if (
    source_metrics.select(*columns)
    .exceptAll(field_metrics.select(*columns))
    .limit(1)
    .count()
    or field_metrics.select(*columns)
    .exceptAll(source_metrics.select(*columns))
    .limit(1)
    .count()
):
    raise RuntimeError("A agregação por campo alterou registros ou medidas.")

print(f"View: {TARGET}")
print("Granularidade, registros e 13 medidas reconciliados: PASS")

display(
    published.groupBy("_source_dataset")
    .agg(
        F.count("*").alias("registros_campo_mes"),
        F.countDistinct("campo_id").alias("contextos_campo_distintos"),
        F.min("competencia").alias("primeira_competencia"),
        F.max("competencia").alias("ultima_competencia")
    )
    .orderBy("_source_dataset")
)

display(
    published
    .filter(F.col("competencia") == F.lit("2025-12-01").cast("date"))
    .select(
        "ambiente",
        "nome_campo_reportado",
        "bacia_reportada",
        "competencia",
        "pocos_com_id_distintos",
        "producao_de_oleo_m3",
        "oleo_m3_dia_calendario"
    )
    .orderBy(F.desc("producao_de_oleo_m3"))
    .limit(10)
)

View: workspace.oil_gas_gold.vw_producao_campo_mes_historico_v2
Granularidade, registros e 13 medidas reconciliados: PASS


_source_dataset,registros_campo_mes,contextos_campo_distintos,primeira_competencia,ultima_competencia
producao_mar,2217,80,2024-01-01,2026-08-01
producao_terra,6425,250,2024-01-01,2026-06-01


ambiente,nome_campo_reportado,bacia_reportada,competencia,pocos_com_id_distintos,producao_de_oleo_m3,oleo_m3_dia_calendario
MAR,TUPI,SANTOS,2025-12-01,115,4135002.542000000,133387.178774194
MAR,MERO,SANTOS,2025-12-01,39,3020779.746000000,97444.507935484
MAR,BÚZIOS_ECO,SANTOS,2025-12-01,54,2954808.369000000,95316.399000000
MAR,BÚZIOS,SANTOS,2025-12-01,54,1044689.196000000,33699.651483871
MAR,JUBARTE,CAMPOS,2025-12-01,56,783016.051000000,25258.582290323
MAR,SAPINHOÁ,SANTOS,2025-12-01,24,556046.445000000,17936.982096774
MAR,BERBIGÃO,SANTOS,2025-12-01,14,555515.649000000,17919.859645161
MAR,PEREGRINO,CAMPOS,2025-12-01,60,480983.437000000,15515.594741935
MAR,SÉPIA_ECO,SANTOS,2025-12-01,9,463251.250000000,14943.588709677
MAR,ITAPU,SANTOS,2025-12-01,9,403309.134000000,13009.972064516


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE = "workspace.oil_gas_gold.vw_producao_campo_mes_historico_v2"
COVERAGE = "workspace.oil_gas_gold.vw_cobertura_mensal_historico_v2"
TARGET = "workspace.oil_gas_gold.vw_variacao_oleo_campo_mes_historico_v2"

spark.sql(f"""
    CREATE OR REPLACE VIEW {TARGET} AS
    WITH pares AS (
        SELECT
            a.*,
            ADD_MONTHS(a.competencia, -1) AS competencia_anterior,

            b.campo_id AS campo_id_anterior,
            b.producao_de_oleo_m3 AS oleo_anterior_m3,
            b.preenchidos_producao_de_oleo_m3
                AS preenchidos_oleo_anterior,
            b.oleo_m3_dia_calendario
                AS taxa_anterior_m3_dia_calendario,

            c.status_cobertura AS cobertura_dataset_mes_anterior

        FROM {SOURCE} a

        LEFT JOIN {SOURCE} b
            ON a.escopo_id = b.escopo_id
            AND a._source_dataset = b._source_dataset
            AND a.ambiente = b.ambiente
            AND a.campo_id = b.campo_id
            AND b.competencia = ADD_MONTHS(a.competencia, -1)

        LEFT JOIN {COVERAGE} c
            ON a.escopo_id = c.escopo_id
            AND a._source_dataset = c._source_dataset
            AND a.ambiente = c.ambiente
            AND c.competencia = ADD_MONTHS(a.competencia, -1)
    ),
    classificados AS (
        SELECT
            *,
            CASE
                WHEN cobertura_dataset_mes_anterior IS NULL
                  OR cobertura_dataset_mes_anterior <> 'COM_REGISTROS'
                    THEN 'SEM_MES_ANTERIOR_COM_REGISTROS_NO_DATASET'
                WHEN campo_id_anterior IS NULL
                    THEN 'CAMPO_SEM_REGISTRO_NO_MES_ANTERIOR'
                WHEN preenchidos_producao_de_oleo_m3 = 0
                    THEN 'SEM_OLEO_INFORMADO_NO_MES_ATUAL'
                WHEN preenchidos_oleo_anterior = 0
                    THEN 'SEM_OLEO_INFORMADO_NO_MES_ANTERIOR'
                ELSE 'COMPARAVEL_NOS_REGISTROS_DISPONIVEIS'
            END AS status_comparacao
        FROM pares
    )
    SELECT
        *,

        CASE
            WHEN status_comparacao =
                'COMPARAVEL_NOS_REGISTROS_DISPONIVEIS'
            THEN CAST(
                oleo_m3_dia_calendario
                - taxa_anterior_m3_dia_calendario
                AS DECIMAL(28,9)
            )
        END AS diferenca_taxa_m3_dia_calendario,

        CASE
            WHEN status_comparacao =
                'COMPARAVEL_NOS_REGISTROS_DISPONIVEIS'
             AND taxa_anterior_m3_dia_calendario > 0
            THEN CAST(
                (
                    oleo_m3_dia_calendario
                    / taxa_anterior_m3_dia_calendario - 1
                ) * 100
                AS DECIMAL(28,6)
            )
        END AS variacao_percentual_taxa

    FROM classificados
""")

source = spark.table(SOURCE)
published = spark.table(TARGET)

grain = [
    "escopo_id", "_source_dataset", "ambiente",
    "campo_id", "competencia"
]

if published.count() != source.count():
    raise RuntimeError("A comparação alterou a quantidade de registros.")

if (
    published.groupBy(*grain)
    .count()
    .filter(F.col("count") != 1)
    .limit(1)
    .count()
):
    raise RuntimeError("A comparação multiplicou registros na granularidade.")

comparable = (
    F.col("status_comparacao")
    == "COMPARAVEL_NOS_REGISTROS_DISPONIVEIS"
)

if published.filter(
    comparable
    & (
        F.col("diferenca_taxa_m3_dia_calendario").isNull()
        | F.col("taxa_anterior_m3_dia_calendario").isNull()
        | F.col("oleo_m3_dia_calendario").isNull()
    )
).limit(1).count():
    raise RuntimeError("Comparação elegível com taxa ou diferença ausente.")

if published.filter(
    ~comparable
    & (
        F.col("diferenca_taxa_m3_dia_calendario").isNotNull()
        | F.col("variacao_percentual_taxa").isNotNull()
    )
).limit(1).count():
    raise RuntimeError("Foi calculada variação para uma comparação inelegível.")

print(f"View: {TARGET}")
print("Granularidade e regras de comparação: PASS")

display(
    published.groupBy("_source_dataset", "status_comparacao")
    .count()
    .orderBy("_source_dataset", "status_comparacao")
)

# Até cinco maiores quedas por ambiente em dezembro de 2025.
# Ordenação pela diferença da taxa, em m³ por dia calendário.
ranking = Window.partitionBy("ambiente").orderBy(
    F.col("diferenca_taxa_m3_dia_calendario").asc(),
    F.col("campo_id").asc()
)

display(
    published
    .filter(
        comparable
        & (F.col("competencia") == F.lit("2025-12-01").cast("date"))
        & (F.col("diferenca_taxa_m3_dia_calendario") < 0)
    )
    .withColumn("posicao", F.row_number().over(ranking))
    .filter(F.col("posicao") <= 5)
    .select(
        "ambiente",
        "posicao",
        "nome_campo_reportado",
        "bacia_reportada",
        "competencia_anterior",
        "competencia",
        "taxa_anterior_m3_dia_calendario",
        "oleo_m3_dia_calendario",
        "diferenca_taxa_m3_dia_calendario",
        "variacao_percentual_taxa"
    )
    .orderBy("ambiente", "posicao")
)

View: workspace.oil_gas_gold.vw_variacao_oleo_campo_mes_historico_v2
Granularidade e regras de comparação: PASS


_source_dataset,status_comparacao,count
producao_mar,CAMPO_SEM_REGISTRO_NO_MES_ANTERIOR,22
producao_mar,COMPARAVEL_NOS_REGISTROS_DISPONIVEIS,2126
producao_mar,SEM_MES_ANTERIOR_COM_REGISTROS_NO_DATASET,69
producao_terra,CAMPO_SEM_REGISTRO_NO_MES_ANTERIOR,63
producao_terra,COMPARAVEL_NOS_REGISTROS_DISPONIVEIS,5932
producao_terra,SEM_MES_ANTERIOR_COM_REGISTROS_NO_DATASET,427
producao_terra,SEM_OLEO_INFORMADO_NO_MES_ANTERIOR,1
producao_terra,SEM_OLEO_INFORMADO_NO_MES_ATUAL,2


ambiente,posicao,nome_campo_reportado,bacia_reportada,competencia_anterior,competencia,taxa_anterior_m3_dia_calendario,oleo_m3_dia_calendario,diferenca_taxa_m3_dia_calendario,variacao_percentual_taxa
MAR,1,JUBARTE,CAMPOS,2025-11-01,2025-12-01,29705.811966667,25258.582290323,-4447.229676344,-14.970908
MAR,2,ARGONAUTA,CAMPOS,2025-11-01,2025-12-01,2373.527433333,108.964451613,-2264.562981720,-95.409177
MAR,3,MERO,SANTOS,2025-11-01,2025-12-01,98673.251700000,97444.507935484,-1228.743764516,-1.245265
MAR,4,OSTRA,CAMPOS,2025-11-01,2025-12-01,476.175800000,0E-9,-476.175800000,-100.000000
MAR,5,PEREGRINO,CAMPOS,2025-11-01,2025-12-01,15844.038533333,15515.594741935,-328.443791398,-2.072980
TERRA,1,SALINA CRISTAL,POTIGUAR,2025-11-01,2025-12-01,389.037700000,264.992483871,-124.045216129,-31.885140
TERRA,2,CARMÓPOLIS,SERGIPE,2025-11-01,2025-12-01,1308.566066667,1204.175354839,-104.390711828,-7.977489
TERRA,3,ESTREITO,POTIGUAR,2025-11-01,2025-12-01,759.122866667,680.302161290,-78.820705377,-10.383129
TERRA,4,CANCÃ,ESPÍRITO SANTO,2025-11-01,2025-12-01,155.667066667,96.614032258,-59.053034409,-37.935471
TERRA,5,CANTO DO AMARO,POTIGUAR,2025-11-01,2025-12-01,920.596466667,868.988548387,-51.607918280,-5.605922


In [0]:
from pyspark.sql import functions as F

VARIATION = "workspace.oil_gas_gold.vw_variacao_oleo_campo_mes_historico_v2"
PRODUCTION = "workspace.oil_gas_gold.vw_producao_historico_v2"

selection = (
    spark.table(VARIATION)
    .filter(
        (F.col("nome_campo_reportado") == "JUBARTE")
        & (F.col("bacia_reportada") == "CAMPOS")
        & (F.col("ambiente") == "MAR")
        & (F.col("competencia") == F.lit("2025-12-01").cast("date"))
        & (
            F.col("status_comparacao")
            == "COMPARAVEL_NOS_REGISTROS_DISPONIVEIS"
        )
    )
    .collect()
)

if len(selection) != 1:
    raise RuntimeError("Esperada uma única comparação para JUBARTE.")

selected = selection[0]

source = (
    spark.table(PRODUCTION)
    .filter(
        (F.col("escopo_id") == selected["escopo_id"])
        & (F.col("_source_dataset") == selected["_source_dataset"])
        & (F.col("campo_id") == selected["campo_id"])
        & (F.col("ambiente_reportado") == selected["ambiente"])
    )
    .join(
        spark.table("workspace.oil_gas_gold.dim_data")
        .select("data_id", F.col("data").alias("competencia")),
        "data_id",
        "inner"
    )
    .filter(
        F.col("competencia").isin(
            selected["competencia_anterior"],
            selected["competencia"]
        )
    )
)


def aggregate_month(month, suffix):
    return (
        source.filter(F.col("competencia") == F.lit(month))
        .groupBy("poco_id")
        .agg(
            F.count("*").alias(f"registros_{suffix}"),
            F.count("producao_de_oleo_m3").alias(f"oleo_preenchidos_{suffix}"),
            F.sum("producao_de_oleo_m3").alias(f"oleo_{suffix}_m3"),
            F.sum(
                F.when(
                    F.col("perfil_medidas").isin(
                        "SOMENTE_PRODUCAO", "PRODUCAO_E_INJECAO"
                    ),
                    1
                ).otherwise(0)
            ).alias(f"registros_producao_{suffix}")
        )
    )


before = aggregate_month(selected["competencia_anterior"], "anterior")
after = aggregate_month(selected["competencia"], "atual")

days_before = selected["competencia_anterior"].replace(day=1)
days_after = selected["competencia"].replace(day=1)

import calendar

days_before = calendar.monthrange(days_before.year, days_before.month)[1]
days_after = calendar.monthrange(days_after.year, days_after.month)[1]

detail = before.join(after, "poco_id", "full")

both_oil = (
    (F.col("oleo_preenchidos_anterior") > 0)
    & (F.col("oleo_preenchidos_atual") > 0)
)

partial_oil = (
    (F.col("oleo_preenchidos_anterior") != F.col("registros_producao_anterior"))
    | (F.col("oleo_preenchidos_atual") != F.col("registros_producao_atual"))
)

detail = (
    detail
    .withColumn(
        "status_comparacao_poco",
        F.when(
            F.col("registros_anterior").isNull(),
            "REGISTRO_SOMENTE_NO_MES_ATUAL"
        )
        .when(
            F.col("registros_atual").isNull(),
            "REGISTRO_SOMENTE_NO_MES_ANTERIOR"
        )
        .when(
            (F.col("oleo_preenchidos_anterior") == 0)
            & (F.col("oleo_preenchidos_atual") == 0),
            "SEM_OLEO_INFORMADO_NOS_DOIS_MESES"
        )
        .when(
            F.col("oleo_preenchidos_anterior") == 0,
            "OLEO_INFORMADO_SOMENTE_NO_MES_ATUAL"
        )
        .when(
            F.col("oleo_preenchidos_atual") == 0,
            "OLEO_INFORMADO_SOMENTE_NO_MES_ANTERIOR"
        )
        .when(partial_oil, "OLEO_PARCIAL_EM_ALGUM_DOS_MESES")
        .otherwise("OLEO_INFORMADO_NOS_DOIS_MESES")
    )
    .withColumn(
        "taxa_anterior_m3_dia_calendario",
        (F.col("oleo_anterior_m3") / F.lit(days_before))
        .cast("decimal(28,9)")
    )
    .withColumn(
        "taxa_atual_m3_dia_calendario",
        (F.col("oleo_atual_m3") / F.lit(days_after))
        .cast("decimal(28,9)")
    )
    .withColumn(
        "diferenca_taxa_comparavel",
        F.when(
            both_oil & ~partial_oil,
            F.col("taxa_atual_m3_dia_calendario")
            - F.col("taxa_anterior_m3_dia_calendario")
        )
    )
)

totals = detail.agg(
    F.sum("oleo_anterior_m3").alias("anterior"),
    F.sum("oleo_atual_m3").alias("atual")
).first()

if (
    totals["anterior"] != selected["oleo_anterior_m3"]
    or totals["atual"] != selected["producao_de_oleo_m3"]
):
    raise RuntimeError("Volumes por poço diferentes dos totais do campo.")

print("Campo: JUBARTE | Bacia: CAMPOS")
print(
    f"Comparação: {selected['competencia_anterior']} "
    f"→ {selected['competencia']}"
)
print("Volumes por poço reconciliados com o campo: PASS")

display(
    detail.groupBy("status_comparacao_poco")
    .agg(
        F.count("*").alias("pocos"),
        F.sum("oleo_anterior_m3").alias("oleo_anterior_m3"),
        F.sum("oleo_atual_m3").alias("oleo_atual_m3"),
        F.sum("diferenca_taxa_comparavel").alias(
            "soma_diferencas_comparaveis_m3_dia"
        )
    )
    .orderBy("status_comparacao_poco")
)

display(
    detail
    .join(
        spark.table("workspace.oil_gas_gold.dim_poco")
        .select("poco_id", "nome_poco_anp"),
        "poco_id",
        "inner"
    )
    .select(
        "nome_poco_anp",
        "status_comparacao_poco",
        "oleo_anterior_m3",
        "oleo_atual_m3",
        "taxa_anterior_m3_dia_calendario",
        "taxa_atual_m3_dia_calendario",
        "diferenca_taxa_comparavel"
    )
    .orderBy(F.col("diferenca_taxa_comparavel").asc_nulls_last())
    .limit(15)
)

Campo: JUBARTE | Bacia: CAMPOS
Comparação: 2025-11-01 → 2025-12-01
Volumes por poço reconciliados com o campo: PASS


status_comparacao_poco,pocos,oleo_anterior_m3,oleo_atual_m3,soma_diferencas_comparaveis_m3_dia
OLEO_INFORMADO_NOS_DOIS_MESES,38,891174.359000000,783016.051000000,-4447.229676343
REGISTRO_SOMENTE_NO_MES_ANTERIOR,1,null,null,null
REGISTRO_SOMENTE_NO_MES_ATUAL,1,null,null,null
SEM_OLEO_INFORMADO_NOS_DOIS_MESES,17,null,null,null


nome_poco_anp,status_comparacao_poco,oleo_anterior_m3,oleo_atual_m3,taxa_anterior_m3_dia_calendario,taxa_atual_m3_dia_calendario,diferenca_taxa_comparavel
7-JUB-74D-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,105192.804000000,32513.529000000,3506.426800000,1048.823516129,-2457.603283871
7-JUB-64D-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,91166.100000000,31968.973000000,3038.870000000,1031.257193548,-2007.612806452
7-JUB-73D-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,67600.269000000,20558.816000000,2253.342300000,663.187612903,-1590.154687097
7-JUB-62DA-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,24738.433000000,7414.176000000,824.614433333,239.166967742,-585.447465591
7-BFR-7-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,16728.340000000,11664.807000000,557.611333333,376.284096774,-181.327236559
7-JUB-27HP-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,13224.800000000,13474.828000000,440.826666667,434.671870968,-6.154795699
7-JUB-31H-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,11847.441000000,12086.593000000,394.914700000,389.890096774,-5.024603226
7-JUB-2HPA-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,7742.763000000,7867.534000000,258.092100000,253.791419355,-4.300680645
7-JUB-24H-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,11745.359000000,12041.668000000,391.511966667,388.440903226,-3.071063441
7-JUB-51H-ESS,OLEO_INFORMADO_NOS_DOIS_MESES,7659.918000000,7900.864000000,255.330600000,254.866580645,-0.464019355


In [0]:
from pyspark.sql import functions as F

SCOPE_ID = "historico_anp_v2"

SELECTED_WELLS = [
    "7-JUB-74D-ESS",
    "7-JUB-64D-ESS",
    "7-JUB-73D-ESS"
]

wells = (
    spark.table("workspace.oil_gas_gold.dim_poco")
    .filter(F.col("nome_poco_anp").isin(SELECTED_WELLS))
    .select("poco_id", "nome_poco_anp")
)

if (
    wells.count() != 3
    or wells.select("poco_id").distinct().count() != 3
):
    raise RuntimeError("Esperados três poços distintos.")

dates = spark.table("workspace.oil_gas_gold.dim_data")

months = (
    dates
    .filter(
        (F.col("ano") == 2025)
        & F.col("eh_primeiro_dia_mes")
    )
    .select(
        F.col("data").alias("competencia"),
        "ultimo_dia_mes"
    )
)

grid = wells.crossJoin(months)

production = (
    spark.table(
        "workspace.oil_gas_gold.vw_producao_historico_v2"
    )
    .filter(
        (F.col("escopo_id") == SCOPE_ID)
        & (F.col("_source_dataset") == "producao_mar")
        & (F.col("campo_reportado") == "JUBARTE")
        & (F.col("bacia_reportada") == "CAMPOS")
    )
    .join(wells.select("poco_id"), "poco_id", "inner")
    .join(
        dates.select(
            "data_id",
            F.col("data").alias("competencia")
        ),
        "data_id",
        "inner"
    )
    .filter(F.year("competencia") == 2025)
    .groupBy("poco_id", "competencia")
    .agg(
        F.count("*").alias("registros_producao_injecao"),
        F.count("producao_de_oleo_m3").alias("registros_com_oleo"),
        F.sum("producao_de_oleo_m3").alias("oleo_reportado_m3")
    )
    .withColumn(
        "oleo_m3_dia_calendario",
        (
            F.col("oleo_reportado_m3")
            / F.dayofmonth(F.last_day("competencia"))
        ).cast("decimal(28,9)")
    )
)

events = (
    spark.table(
        "workspace.oil_gas_gold.vw_intervencoes_historico_v2"
    )
    .filter(F.col("escopo_id") == SCOPE_ID)
    .join(wells, "poco_id", "inner")
    .join(
        dates.select(
            F.col("data_id").alias("data_inicio_id"),
            F.col("data").alias("data_inicio")
        ),
        "data_inicio_id",
        "inner"
    )
    .join(
        dates.select(
            F.col("data_id").alias("data_termino_id"),
            F.col("data").alias("data_termino")
        ),
        "data_termino_id",
        "inner"
    )
    .filter(
        (F.col("data_inicio") <= F.lit("2025-12-31").cast("date"))
        & (F.col("data_termino") >= F.lit("2025-01-01").cast("date"))
    )
)

event_months = (
    grid.alias("g")
    .join(
        events.alias("e"),
        (F.col("g.poco_id") == F.col("e.poco_id"))
        & (F.col("e.data_inicio") <= F.col("g.ultimo_dia_mes"))
        & (F.col("e.data_termino") >= F.col("g.competencia")),
        "left"
    )
    .select(
        F.col("g.poco_id").alias("poco_id"),
        F.col("g.competencia").alias("competencia"),
        F.col("e.fato_id").alias("fato_id"),
        F.col("e.objetivo").alias("objetivo")
    )
    .groupBy("poco_id", "competencia")
    .agg(
        F.countDistinct("fato_id").alias("intervencoes_no_mes"),
        F.sort_array(
            F.collect_set("objetivo")
        ).alias("objetivos_observados")
    )
)

timeline = (
    grid
    .join(production, ["poco_id", "competencia"], "left")
    .join(event_months, ["poco_id", "competencia"], "left")
    .withColumn(
        "status_oleo",
        F.when(
            F.col("registros_producao_injecao").isNull(),
            "SEM_REGISTRO_DE_PRODUCAO_INJECAO"
        )
        .when(
            F.col("registros_com_oleo") == 0,
            "SEM_OLEO_INFORMADO"
        )
        .otherwise("OLEO_INFORMADO")
    )
)

if (
    timeline.count() != 36
    or timeline.select("poco_id", "competencia").distinct().count() != 36
):
    raise RuntimeError("A linha do tempo não preservou 12 meses por poço.")

print("Linha do tempo: três poços × 12 meses de 2025")
print("Granularidade após associação dos eventos: PASS")

display(
    timeline
    .filter(F.col("competencia") >= F.lit("2025-09-01").cast("date"))
    .select(
        "nome_poco_anp",
        "competencia",
        "status_oleo",
        "oleo_reportado_m3",
        "oleo_m3_dia_calendario",
        "intervencoes_no_mes",
        "objetivos_observados"
    )
    .orderBy("nome_poco_anp", "competencia")
)

display(
    events.select(
        "nome_poco_anp",
        "fato_id",
        "objetivo",
        "sonda",
        "data_inicio",
        "data_termino",
        "dias_em_intervencao"
    )
    .orderBy("nome_poco_anp", "data_inicio")
)

Linha do tempo: três poços × 12 meses de 2025
Granularidade após associação dos eventos: PASS


nome_poco_anp,competencia,status_oleo,oleo_reportado_m3,oleo_m3_dia_calendario,intervencoes_no_mes,objetivos_observados
7-JUB-64D-ESS,2025-09-01,OLEO_INFORMADO,97606.934000000,3253.564466667,0,List()
7-JUB-64D-ESS,2025-10-01,OLEO_INFORMADO,100449.901000000,3240.319387097,0,List()
7-JUB-64D-ESS,2025-11-01,OLEO_INFORMADO,91166.100000000,3038.870000000,0,List()
7-JUB-64D-ESS,2025-12-01,OLEO_INFORMADO,31968.973000000,1031.257193548,0,List()
7-JUB-73D-ESS,2025-09-01,OLEO_INFORMADO,84099.175000000,2803.305833333,0,List()
7-JUB-73D-ESS,2025-10-01,OLEO_INFORMADO,69751.150000000,2250.037096774,0,List()
7-JUB-73D-ESS,2025-11-01,OLEO_INFORMADO,67600.269000000,2253.342300000,0,List()
7-JUB-73D-ESS,2025-12-01,OLEO_INFORMADO,20558.816000000,663.187612903,0,List()
7-JUB-74D-ESS,2025-09-01,OLEO_INFORMADO,93478.469000000,3115.948966667,0,List()
7-JUB-74D-ESS,2025-10-01,OLEO_INFORMADO,108337.453000000,3494.756548387,0,List()


nome_poco_anp,fato_id,objetivo,sonda,data_inicio,data_termino,dias_em_intervencao
7-JUB-64D-ESS,71705ee7b6039ff78a5c2d51e1725cb2d72ee746301dc436574349dfc6857b94,COMPLETAÇÃO,DEEPWATER CORCOVADO,2025-03-19,2025-04-05,18
7-JUB-73D-ESS,4957f90bfc853e50f7a3dd2610ae74bbcdd246d83e7274dd7f69262a2618156e,COMPLETAÇÃO,DEEPWATER CORCOVADO,2024-12-16,2025-01-11,27
7-JUB-74D-ESS,61c5a39be0ca61d73725856376397ab62d190c9d22972959c72e7e8c08a5b5eb,PERFURAÇÃO,DEEPWATER CORCOVADO,2025-01-12,2025-02-04,24
7-JUB-74D-ESS,df9d384710c96e0b08b3ee02e37ab8ee8f96232124d7778231c38fdcc466ec27,COMPLETAÇÃO,DEEPWATER CORCOVADO,2025-02-05,2025-03-18,42
7-JUB-74D-ESS,c82948478870e0d94563ca91d82951c3c561a9968189619a7480ca132d005a27,COMPLETAÇÃO,DEEPWATER CORCOVADO,2025-04-05,2025-05-24,50


In [0]:
from pyspark.sql import functions as F

SOURCE = "workspace.oil_gas_gold.vw_producao_historico_v2"

SELECTED_WELLS = [
    "7-JUB-64D-ESS",
    "7-JUB-73D-ESS",
    "7-JUB-74D-ESS"
]

wells = (
    spark.table("workspace.oil_gas_gold.dim_poco")
    .filter(F.col("nome_poco_anp").isin(SELECTED_WELLS))
    .select("poco_id", "nome_poco_anp")
)

if (
    wells.count() != 3
    or wells.select("poco_id").distinct().count() != 3
):
    raise RuntimeError("Esperados três poços distintos.")

source = (
    spark.table(SOURCE)
    .filter(
        (F.col("escopo_id") == "historico_anp_v2")
        & (F.col("_source_dataset") == "producao_mar")
    )
    .join(wells, "poco_id", "inner")
    .join(
        spark.table("workspace.oil_gas_gold.dim_data")
        .select("data_id", F.col("data").alias("competencia")),
        "data_id",
        "inner"
    )
    .filter(
        F.col("competencia").isin(
            F.lit("2025-11-01").cast("date"),
            F.lit("2025-12-01").cast("date")
        )
    )
)

is_jubarte = (
    (F.col("campo_reportado") == "JUBARTE")
    & (F.col("bacia_reportada") == "CAMPOS")
)

summary = (
    source.groupBy("poco_id", "nome_poco_anp", "competencia")
    .agg(
        F.count("*").alias("registros_fonte"),
        F.countDistinct("campo_id").alias("contextos_campo"),
        F.sort_array(
            F.collect_set("campo_reportado")
        ).alias("campos_reportados"),
        F.sort_array(
            F.collect_set("instalacao_reportada")
        ).alias("instalacoes_reportadas"),
        F.sum("producao_de_oleo_m3").alias("oleo_todos_contextos_m3"),
        F.sum(
            F.when(is_jubarte, F.col("producao_de_oleo_m3"))
        ).alias("oleo_contexto_jubarte_m3")
    )
    .withColumn(
        "oleo_outros_contextos_m3",
        F.col("oleo_todos_contextos_m3")
        - F.col("oleo_contexto_jubarte_m3")
    )
    .withColumn(
        "taxa_todos_contextos_m3_dia_calendario",
        (
            F.col("oleo_todos_contextos_m3")
            / F.dayofmonth(F.last_day("competencia"))
        ).cast("decimal(28,9)")
    )
)

if (
    summary.count() != 6
    or summary.select("poco_id", "competencia").distinct().count() != 6
):
    raise RuntimeError("Esperados dois meses para cada um dos três poços.")

before = (
    summary
    .filter(F.col("competencia") == F.lit("2025-11-01").cast("date"))
    .select(
        "poco_id",
        F.col("taxa_todos_contextos_m3_dia_calendario")
        .alias("taxa_total_reportada_novembro")
    )
)

after = (
    summary
    .filter(F.col("competencia") == F.lit("2025-12-01").cast("date"))
    .select(
        "poco_id",
        "nome_poco_anp",
        F.col("taxa_todos_contextos_m3_dia_calendario")
        .alias("taxa_total_reportada_dezembro")
    )
)

comparison = (
    before.join(after, "poco_id", "inner")
    .withColumn(
        "diferenca_taxa_total_reportada",
        F.col("taxa_total_reportada_dezembro")
        - F.col("taxa_total_reportada_novembro")
    )
)

display(
    summary.select(
        "nome_poco_anp",
        "competencia",
        "contextos_campo",
        "campos_reportados",
        "instalacoes_reportadas",
        "oleo_contexto_jubarte_m3",
        "oleo_outros_contextos_m3",
        "oleo_todos_contextos_m3",
        "taxa_todos_contextos_m3_dia_calendario"
    )
    .orderBy("nome_poco_anp", "competencia")
)

display(
    comparison.select(
        "nome_poco_anp",
        "taxa_total_reportada_novembro",
        "taxa_total_reportada_dezembro",
        "diferenca_taxa_total_reportada"
    )
    .orderBy("diferenca_taxa_total_reportada")
)

nome_poco_anp,competencia,contextos_campo,campos_reportados,instalacoes_reportadas,oleo_contexto_jubarte_m3,oleo_outros_contextos_m3,oleo_todos_contextos_m3,taxa_todos_contextos_m3_dia_calendario
7-JUB-64D-ESS,2025-11-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),91166.100000000,2577.96200000,93744.062000000,3124.802066667
7-JUB-64D-ESS,2025-12-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),31968.973000000,904.00700000,32872.980000000,1060.418709677
7-JUB-73D-ESS,2025-11-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),67600.269000000,1911.57600000,69511.845000000,2317.061500000
7-JUB-73D-ESS,2025-12-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),20558.816000000,581.35300000,21140.169000000,681.940935484
7-JUB-74D-ESS,2025-11-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),105192.804000000,2974.60400000,108167.408000000,3605.580266667
7-JUB-74D-ESS,2025-12-01,4,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",List(FPSO MARIA QUITÉRIA),32513.529000000,919.40500000,33432.934000000,1078.481741935


nome_poco_anp,taxa_total_reportada_novembro,taxa_total_reportada_dezembro,diferenca_taxa_total_reportada
7-JUB-74D-ESS,3605.580266667,1078.481741935,-2527.098524732
7-JUB-64D-ESS,3124.802066667,1060.418709677,-2064.383356990
7-JUB-73D-ESS,2317.061500000,681.940935484,-1635.120564516


In [0]:
from pyspark.sql import functions as F

SOURCE = "workspace.oil_gas_gold.vw_producao_historico_v2"

SELECTED_WELLS = [
    "7-JUB-64D-ESS",
    "7-JUB-73D-ESS",
    "7-JUB-74D-ESS"
]

wells = (
    spark.table("workspace.oil_gas_gold.dim_poco")
    .select("poco_id", "nome_poco_anp")
)

dates = (
    spark.table("workspace.oil_gas_gold.dim_data")
    .select("data_id", F.col("data").alias("competencia"))
)

source = (
    spark.table(SOURCE)
    .filter(
        (F.col("escopo_id") == "historico_anp_v2")
        & (F.col("_source_dataset") == "producao_mar")
        & (F.col("instalacao_reportada") == "FPSO MARIA QUITÉRIA")
        & (F.col("ambiente_reportado") == "MAR")
    )
    .join(dates, "data_id", "inner")
    .filter(
        F.col("competencia").isin(
            F.lit("2025-11-01").cast("date"),
            F.lit("2025-12-01").cast("date")
        )
    )
    .join(wells, "poco_id", "inner")
    .withColumn(
        "grupo_pocos",
        F.when(
            F.col("nome_poco_anp").isin(SELECTED_WELLS),
            "TRES_POCOS_SELECIONADOS"
        ).otherwise("DEMAIS_POCOS")
    )
)

monthly = (
    source.groupBy("competencia")
    .agg(
        F.count("*").alias("registros_fonte"),
        F.countDistinct("poco_id").alias("pocos_distintos"),
        F.countDistinct(
            F.when(
                F.col("nome_poco_anp").isin(SELECTED_WELLS),
                F.col("poco_id")
            )
        ).alias("pocos_selecionados_presentes"),
        F.sum("producao_de_oleo_m3").alias("oleo_reportado_m3"),
        F.sort_array(
            F.collect_set("campo_reportado")
        ).alias("campos_reportados")
    )
    .withColumn(
        "oleo_m3_dia_calendario",
        (
            F.col("oleo_reportado_m3")
            / F.dayofmonth(F.last_day("competencia"))
        ).cast("decimal(28,9)")
    )
)

if (
    monthly.count() != 2
    or monthly.filter(
        F.col("pocos_selecionados_presentes") != 3
    ).limit(1).count()
):
    raise RuntimeError("Os três poços precisam estar presentes nos dois meses.")

groups = (
    source.groupBy("grupo_pocos", "competencia")
    .agg(
        F.countDistinct("poco_id").alias("pocos"),
        F.sum("producao_de_oleo_m3").alias("oleo_m3")
    )
    .withColumn(
        "taxa_m3_dia_calendario",
        (
            F.col("oleo_m3")
            / F.dayofmonth(F.last_day("competencia"))
        ).cast("decimal(28,9)")
    )
)

before = (
    groups
    .filter(F.col("competencia") == F.lit("2025-11-01").cast("date"))
    .select(
        "grupo_pocos",
        F.col("pocos").alias("pocos_novembro"),
        F.col("taxa_m3_dia_calendario").alias("taxa_novembro")
    )
)

after = (
    groups
    .filter(F.col("competencia") == F.lit("2025-12-01").cast("date"))
    .select(
        "grupo_pocos",
        F.col("pocos").alias("pocos_dezembro"),
        F.col("taxa_m3_dia_calendario").alias("taxa_dezembro")
    )
)

comparison = (
    before.join(after, "grupo_pocos", "full")
    .withColumn(
        "diferenca_taxa_m3_dia_calendario",
        F.col("taxa_dezembro") - F.col("taxa_novembro")
    )
    .withColumn(
        "variacao_percentual_taxa",
        F.when(
            F.col("taxa_novembro") > 0,
            (
                (F.col("taxa_dezembro") / F.col("taxa_novembro") - 1)
                * 100
            ).cast("decimal(28,6)")
        )
    )
)

display(monthly.orderBy("competencia"))

display(
    comparison.orderBy("diferenca_taxa_m3_dia_calendario")
)

competencia,registros_fonte,pocos_distintos,pocos_selecionados_presentes,oleo_reportado_m3,campos_reportados,oleo_m3_dia_calendario
2025-11-01,20,5,3,296861.292000000,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",9895.376400000
2025-12-01,16,4,3,95069.915000000,"List(ANC_JUBARTE_NORDESTE, ANC_JUBARTE_SUDOESTE, ARGONAUTA, JUBARTE)",3066.771451613


grupo_pocos,pocos_novembro,taxa_novembro,pocos_dezembro,taxa_dezembro,diferenca_taxa_m3_dia_calendario,variacao_percentual_taxa
TRES_POCOS_SELECIONADOS,3,9047.443833333,3,2820.841387097,-6226.602446236,-68.821676
DEMAIS_POCOS,2,847.932566667,1,245.930064516,-602.002502151,-70.996507
